In [1]:
# ============================================================================
# CELL 1: State Management & Utilities
# ============================================================================
# SYNC TO AI AGENT v17.2:
# 1. Dual BASE_PATH detection — tries both known paths
# 2. All file path constants matching AI agent v17.2
# 3. NEW: TAUGHT_START_MENU_TRANSITIONS_FILE path
# 4. NEW: TYPE_CLUSTERS_FILE, AI_EVENT_TIMELINE_FILE, TYPE_DATA_FILE paths
#    (teaching code does NOT produce these — listed for awareness/reference)
# 5. NEW: Start Menu Markov weight constants
# 6. Full defaults: battle, party, menu, bag, item knowledge
# 7. Full parsers: battle, party, menu, bag, game_state
# 8. read_game_state() returns 10 values matching AI agent
# 9. Chain-specific learning state builders (overworld, battle, party, bag)
# 10. Per-chain dimension constants
# 11. All Markov weight constants (overworld, battle, bag, start menu)
# ============================================================================

from pathlib import Path
import json
import numpy as np
import time
from collections import deque

# === DUAL BASE_PATH DETECTION ===
_CANDIDATE_PATHS = [
    Path("C:/Users/HP/Documents/cogai/"),
    Path("C:/Users/natmaw/Documents/Boston Stuff/CS 5100 Foundations of AI/PokeAI/"),
]

BASE_PATH = None
for _p in _CANDIDATE_PATHS:
    if _p.exists():
        BASE_PATH = _p
        break

if BASE_PATH is None:
    BASE_PATH = _CANDIDATE_PATHS[0]
    print(f"⚠️ WARNING: No valid base path found. Defaulting to {BASE_PATH}")
else:
    print(f"📂 BASE_PATH: {BASE_PATH}")

# === FILE PATHS ===
ACTION_FILE = BASE_PATH / "action.json"
STATE_FILE = BASE_PATH / "game_state.json"
INPUT_FILE = BASE_PATH / "input_cache.txt"

# Teaching output files
TAUGHT_TRANSITIONS_FILE = BASE_PATH / "taught_transitions.json"
TAUGHT_BATTLE_TRANSITIONS_FILE = BASE_PATH / "taught_battle_transitions.json"
TAUGHT_BAG_TRANSITIONS_FILE = BASE_PATH / "taught_bag_transitions.json"
TAUGHT_START_MENU_TRANSITIONS_FILE = BASE_PATH / "taught_start_menu_transitions.json"  # NEW v17.2
TAUGHT_NAV_TARGETS_FILE = BASE_PATH / "taught_nav_targets.json"
MODEL_CHECKPOINT_FILE = BASE_PATH / "taught_model_checkpoint.json"
EXPLORATION_MEMORY_FILE = BASE_PATH / "taught_exploration_memory.json"
EVENT_TIMELINE_FILE = BASE_PATH / "event_timeline.json"

# Aliases
MODEL_FILE = MODEL_CHECKPOINT_FILE
TAUGHT_EXPLORATION_FILE = EXPLORATION_MEMORY_FILE

# AI agent files (read-only references — teaching code does NOT produce these)
AI_MODEL_CHECKPOINT_FILE = BASE_PATH / "model_checkpoint.json"
AI_EXPLORATION_MEMORY_FILE = BASE_PATH / "exploration_memory.json"
ROSTER_FILE = BASE_PATH / "roster.json"
MOVE_KNOWLEDGE_FILE = BASE_PATH / "move_knowledge.json"
ITEM_KNOWLEDGE_FILE = BASE_PATH / "item_knowledge.json"
TYPE_CLUSTERS_FILE = BASE_PATH / "type_clusters.json"              # NEW v17.2 — AI produces
AI_EVENT_TIMELINE_FILE = BASE_PATH / "ai_event_timeline.json"      # NEW v17.2 — AI produces
TYPE_DATA_FILE = BASE_PATH / "type_data.json"                      # NEW v17.2 — Lua verification produces

# === OVERWORLD MARKOV SIMILARITY WEIGHTS ===
MARKOV_IMMEDIATE_WEIGHT = 0.5
MARKOV_SEQUENTIAL_WEIGHT = 0.3
MARKOV_PARTIAL_WEIGHT = 0.2
MARKOV_FAMILIARITY_THRESHOLD = 0.6

MARKOV_SEQ_FULL_WEIGHT = 1.0
MARKOV_SEQ_MEDIUM_WEIGHT = 0.6
MARKOV_SEQ_SHORT_WEIGHT = 0.3

MARKOV_POS_EXACT_BONUS = 0.35
MARKOV_POS_NEAR_BONUS = 0.25
MARKOV_POS_FAR_BONUS = 0.1
MARKOV_POS_MAX_DIST = 5

# === BATTLE MARKOV WEIGHTS ===
BATTLE_MARKOV_ACTION_SEQ_WEIGHT = 0.70
BATTLE_MARKOV_PALETTE_WEIGHT = 0.20
BATTLE_MARKOV_MENU_STATE_WEIGHT = 0.10
BATTLE_MARKOV_THRESHOLD_LOW = 0.35
BATTLE_MARKOV_THRESHOLD_HIGH = 0.45

# === BAG MARKOV WEIGHTS ===
BAG_MARKOV_MENU_STATE_WEIGHT = 0.40
BAG_MARKOV_ACTION_SEQ_WEIGHT = 0.35
BAG_MARKOV_PARTY_CONTEXT_WEIGHT = 0.25
BAG_MARKOV_THRESHOLD = 0.35

# === START MENU MARKOV WEIGHTS (NEW v17.2) ===
START_MENU_MARKOV_MENU_STATE_WEIGHT = 0.45
START_MENU_MARKOV_ACTION_SEQ_WEIGHT = 0.35
START_MENU_MARKOV_CONTEXT_WEIGHT = 0.20
START_MENU_MARKOV_THRESHOLD = 0.35

# === STATE DIMENSIONS ===
EXPECTED_STATE_DIM = 6
PALETTE_DIM = 768
TILE_DIM = 600

# Per-chain learning state dimensions (must match AI agent exactly)
OVERWORLD_STATE_DIM = 8 + TILE_DIM + PALETTE_DIM   # 1376
OVERWORLD_BATTLE_STATE_DIM = 8 + PALETTE_DIM        # 776
BATTLE_CHAIN_DIM = 41
PARTY_CHAIN_DIM = 50
BAG_CHAIN_DIM = 18

# Legacy
LEARNING_STATE_DIM = OVERWORLD_STATE_DIM  # 1376

# Action code mapping (from Lua short codes)
ACTION_MAP = {
    'U': 'UP', 'D': 'DOWN', 'L': 'LEFT', 'R': 'RIGHT',
    'A': 'A', 'B': 'B', 'S': 'Start', 'E': 'Select'
}

# === START MENU CURSOR POSITIONS (gs==1) ===
# Matches AI agent's START_MENU_OPTIONS
START_MENU_OPTIONS = {
    'pokedex': 0, 'pokemon': 1, 'bag': 2, 'trainer_card': 3,
    'save': 4, 'option': 5, 'exit': 6
}
START_MENU_MC_NAMES = {
    0: 'pokedex', 1: 'pokemon', 2: 'bag', 3: 'trainer_card',
    4: 'save', 5: 'option', 6: 'exit'
}

# === DEFAULT BATTLE DATA ===
DEFAULT_BATTLE_DATA = {
    'battle_cursor': -1, 'move_cursor': -1, 'party_cursor': -1,
    'player_species': -1, 'enemy_species': -1,
    'player_hp': -1, 'player_max_hp': -1, 'enemy_hp': -1, 'enemy_max_hp': -1,
    'player_level': -1, 'enemy_level': -1,
    'player_status': 0, 'enemy_status': 0, 'battle_type': 0,
    'move0': -1, 'move1': -1, 'move2': -1, 'move3': -1,
    'pp0': -1, 'pp1': -1, 'pp2': -1, 'pp3': -1,
    'player_stat_stages': [-1, -1, -1, -1, -1, -1, -1],
    'enemy_move0': -1, 'enemy_move1': -1, 'enemy_move2': -1, 'enemy_move3': -1,
    'enemy_pp0': -1, 'enemy_pp1': -1, 'enemy_pp2': -1, 'enemy_pp3': -1,
    'enemy_stat_stages': [-1, -1, -1, -1, -1, -1, -1],
}

# === DEFAULT PARTY DATA ===
DEFAULT_PARTY_SLOT = {
    'level': 0, 'hp': 0, 'max_hp': 0,
    'atk': 0, 'def': 0, 'spd': 0, 'spatk': 0, 'spdef': 0,
    'status': 0
}
DEFAULT_PARTY_DATA = {'count': 0, 'slots': []}

# === DEFAULT MENU DATA ===
DEFAULT_MENU_DATA = {'mc': -1, 'mm': -1, 'pc': -1, 'sc': -1}

# === DEFAULT BAG DATA ===
DEFAULT_BAG_DATA = {'pocket': -1, 'cursor': -1, 'active': 0, 'items': []}

# === DEFAULT ITEM KNOWLEDGE ENTRY ===
DEFAULT_ITEM_KNOWLEDGE_ENTRY = {
    'uses': 0, 'category': 'unknown', 'confidence': 0.0,
    'avg_hp_restored': 0.0, 'total_hp_restored': 0,
    'status_cured': [], 'catch_attempts': 0,
    'catch_successes': 0, 'last_used_timestep': 0,
}


# ============================================================================
# NORMALIZATION & DERIVED FEATURES
# ============================================================================

def normalize_game_state(raw_state):
    if len(raw_state) < 6:
        raw_state = list(raw_state) + [0] * (6 - len(raw_state))
    normalized = np.array(raw_state, dtype=float)
    normalized[0] = raw_state[0] / 255.0
    normalized[1] = raw_state[1] / 255.0
    normalized[2] = np.clip(raw_state[2], 0, 255)
    normalized[3] = 1.0 if raw_state[3] > 0 else 0.0
    normalized[4] = 1.0 if raw_state[4] > 0 else 0.0
    normalized[5] = int(raw_state[5]) % 4
    return normalized


def compute_derived_features(current, prev):
    if prev is None:
        return np.zeros(8)
    vel_x = current[0] - prev[0]
    vel_y = current[1] - prev[1]
    map_changed = 1.0 if abs(current[2] - prev[2]) > 0.5 else 0.0
    battle_started = 1.0 if current[3] > prev[3] else 0.0
    battle_ended = 1.0 if current[3] < prev[3] else 0.0
    menu_opened = 1.0 if current[4] > prev[4] else 0.0
    menu_closed = 1.0 if current[4] < prev[4] else 0.0
    direction_changed = 1.0 if current[5] != prev[5] else 0.0
    return np.array([vel_x, vel_y, map_changed, battle_started, battle_ended,
                     menu_opened, menu_closed, direction_changed])


# ============================================================================
# CHAIN-SPECIFIC LEARNING STATE BUILDERS
# (Must match AI agent's versions exactly so taught model weights are compatible)
# ============================================================================

def build_learning_state_overworld(derived, palette, tiles, in_battle):
    """
    Overworld chain learning state.
    In battle: derived(8) + palette(768) = 776 dims
    In overworld: derived(8) + tiles(600) + palette(768) = 1376 dims
    """
    if len(derived) != 8:
        derived = np.zeros(8)
    if len(tiles) != TILE_DIM:
        tiles = np.zeros(TILE_DIM)
    if len(palette) != PALETTE_DIM:
        palette = np.zeros(PALETTE_DIM)

    if in_battle > 0.5:
        state = np.concatenate([derived, palette])
    else:
        state = np.concatenate([derived, tiles, palette])
    noise = np.random.randn(len(state)) * 0.0001
    return state + noise


# Backward compatibility alias
build_learning_state = build_learning_state_overworld


def build_learning_state_battle(battle_data, party_data=None, turn_count=0):
    """
    Battle chain learning state — compact features from battle memory.
    All values normalized to roughly 0-1 range for perceptron compatibility.

    Layout (41 dims):
    [0]     player_species / 500
    [1]     player_hp_ratio
    [2]     player_level / 100
    [3]     player_status_flag
    [4:8]   player_move_ids / 500
    [8:12]  player_pp_ratios
    [12:19] player_stat_stages / 12
    [19]    enemy_species / 500
    [20]    enemy_hp_ratio
    [21]    enemy_level / 100
    [22]    enemy_status_flag
    [23:27] enemy_move_ids / 500
    [27:31] enemy_pp_ratios
    [31:38] enemy_stat_stages / 12
    [38]    is_trainer
    [39]    turn_count / 20
    [40]    damage_trend (placeholder)
    """
    bd = battle_data
    state = np.zeros(BATTLE_CHAIN_DIM)

    # Player
    state[0] = max(0, bd.get('player_species', -1)) / 500.0
    php, pmhp = bd.get('player_hp', -1), bd.get('player_max_hp', -1)
    state[1] = php / pmhp if pmhp > 0 and php >= 0 else 0.0
    state[2] = max(0, bd.get('player_level', -1)) / 100.0
    state[3] = 1.0 if bd.get('player_status', 0) != 0 else 0.0

    for i, mk in enumerate(['move0', 'move1', 'move2', 'move3']):
        state[4 + i] = max(0, bd.get(mk, -1)) / 500.0

    for i, pk in enumerate(['pp0', 'pp1', 'pp2', 'pp3']):
        pp = bd.get(pk, -1)
        state[8 + i] = pp / 40.0 if pp >= 0 else 0.0

    pss = bd.get('player_stat_stages', [-1]*7)
    for i in range(7):
        state[12 + i] = pss[i] / 12.0 if pss[i] >= 0 else 0.5

    # Enemy
    state[19] = max(0, bd.get('enemy_species', -1)) / 500.0
    ehp, emhp = bd.get('enemy_hp', -1), bd.get('enemy_max_hp', -1)
    state[20] = ehp / emhp if emhp > 0 and ehp >= 0 else 0.0
    state[21] = max(0, bd.get('enemy_level', -1)) / 100.0
    state[22] = 1.0 if bd.get('enemy_status', 0) != 0 else 0.0

    for i, mk in enumerate(['enemy_move0', 'enemy_move1', 'enemy_move2', 'enemy_move3']):
        state[23 + i] = max(0, bd.get(mk, -1)) / 500.0

    for i, pk in enumerate(['enemy_pp0', 'enemy_pp1', 'enemy_pp2', 'enemy_pp3']):
        pp = bd.get(pk, -1)
        state[27 + i] = pp / 40.0 if pp >= 0 else 0.0

    ess = bd.get('enemy_stat_stages', [-1]*7)
    for i in range(7):
        state[31 + i] = ess[i] / 12.0 if ess[i] >= 0 else 0.5

    # Context
    state[38] = 1.0 if (bd.get('battle_type', 0) & 8) != 0 else 0.0
    state[39] = min(turn_count, 20) / 20.0
    state[40] = 0.0  # placeholder

    state += np.random.randn(BATTLE_CHAIN_DIM) * 0.0001
    return state


def build_learning_state_party(party_data, active_slot=-1):
    """
    Party chain learning state — per-slot stats + context.

    Layout (50 dims):
    Per slot (6 × 8 = 48):
      [i*8+0] hp_ratio, [i*8+1] level/100, [i*8+2] status_flag
      [i*8+3] atk/500, [i*8+4] def/500, [i*8+5] spd/500
      [i*8+6] spatk/500, [i*8+7] spdef/500
    Context (2):
      [48] party_count/6, [49] active_slot/6
    """
    state = np.zeros(PARTY_CHAIN_DIM)
    slots = party_data.get('slots', [])

    for i in range(min(6, len(slots))):
        slot = slots[i]
        base = i * 8
        mhp = slot.get('max_hp', 0)
        state[base + 0] = slot.get('hp', 0) / mhp if mhp > 0 else 0.0
        state[base + 1] = slot.get('level', 0) / 100.0
        state[base + 2] = 1.0 if slot.get('status', 0) != 0 else 0.0
        state[base + 3] = slot.get('atk', 0) / 500.0
        state[base + 4] = slot.get('def', 0) / 500.0
        state[base + 5] = slot.get('spd', 0) / 500.0
        state[base + 6] = slot.get('spatk', 0) / 500.0
        state[base + 7] = slot.get('spdef', 0) / 500.0

    state[48] = party_data.get('count', 0) / 6.0
    state[49] = active_slot / 6.0 if active_slot >= 0 else 0.0

    state += np.random.randn(PARTY_CHAIN_DIM) * 0.0001
    return state


def build_learning_state_bag(bag_data, party_data, menu_data, in_battle=False, item_knowledge=None):
    """
    Bag chain learning state — bag navigation context + party needs.

    Layout (18 dims):
    [0] pocket/4, [1] cursor/20, [2] item_at_cursor_id/500
    [3] n_items/20, [4:10] party_hp_ratios (6 slots)
    [10:16] party_status_flags (6 slots)
    [16] in_battle, [17] mc/6
    """
    state = np.zeros(BAG_CHAIN_DIM)

    pocket = bag_data.get('pocket', -1)
    cursor = bag_data.get('cursor', -1)
    items = bag_data.get('items', [])

    state[0] = max(0, pocket) / 4.0
    state[1] = max(0, cursor) / 20.0

    if 0 <= cursor < len(items):
        state[2] = items[cursor].get('id', 0) / 500.0
    else:
        state[2] = 0.0

    state[3] = len(items) / 20.0

    slots = party_data.get('slots', [])
    for i in range(min(6, len(slots))):
        mhp = slots[i].get('max_hp', 0)
        state[4 + i] = slots[i].get('hp', 0) / mhp if mhp > 0 else 0.0

    for i in range(min(6, len(slots))):
        state[10 + i] = 1.0 if slots[i].get('status', 0) != 0 else 0.0

    state[16] = 1.0 if in_battle else 0.0
    state[17] = max(0, menu_data.get('mc', -1)) / 6.0

    state += np.random.randn(BAG_CHAIN_DIM) * 0.0001
    return state


# ============================================================================
# ARRAY HELPERS
# ============================================================================

def _pad_or_trim(arr, target_dim):
    if arr.shape[0] < target_dim:
        return np.pad(arr, (0, target_dim - arr.shape[0]))
    elif arr.shape[0] > target_dim:
        return arr[:target_dim]
    return arr


# ============================================================================
# PARSERS
# ============================================================================

def parse_battle_data(data):
    b = data.get('b')
    if b is None:
        return DEFAULT_BATTLE_DATA.copy()

    pss = b.get('pss')
    if not isinstance(pss, list) or len(pss) != 7:
        pss = [-1, -1, -1, -1, -1, -1, -1]
    else:
        pss = list(pss)

    ess = b.get('ess')
    if not isinstance(ess, list) or len(ess) != 7:
        ess = [-1, -1, -1, -1, -1, -1, -1]
    else:
        ess = list(ess)

    return {
        'battle_cursor': b.get('bc', -1), 'move_cursor': b.get('mc', -1),
        'party_cursor': b.get('pc', -1),
        'player_species': b.get('ps', -1), 'enemy_species': b.get('es', -1),
        'player_hp': b.get('ph', -1), 'player_max_hp': b.get('pm', -1),
        'enemy_hp': b.get('eh', -1), 'enemy_max_hp': b.get('em', -1),
        'player_level': b.get('pl', -1), 'enemy_level': b.get('el', -1),
        'player_status': b.get('pst', 0), 'enemy_status': b.get('est', 0),
        'battle_type': b.get('bt', 0),
        'move0': b.get('m0', -1), 'move1': b.get('m1', -1),
        'move2': b.get('m2', -1), 'move3': b.get('m3', -1),
        'pp0': b.get('pp0', -1), 'pp1': b.get('pp1', -1),
        'pp2': b.get('pp2', -1), 'pp3': b.get('pp3', -1),
        'player_stat_stages': pss,
        'enemy_move0': b.get('em0', -1), 'enemy_move1': b.get('em1', -1),
        'enemy_move2': b.get('em2', -1), 'enemy_move3': b.get('em3', -1),
        'enemy_pp0': b.get('epp0', -1), 'enemy_pp1': b.get('epp1', -1),
        'enemy_pp2': b.get('epp2', -1), 'enemy_pp3': b.get('epp3', -1),
        'enemy_stat_stages': ess,
    }


def parse_party_data(data):
    pa = data.get('pa')
    if pa is None:
        return DEFAULT_PARTY_DATA.copy()
    count = pa.get('c', 0)
    raw_slots = pa.get('s', [])
    slots = []
    for s in raw_slots:
        if not isinstance(s, dict): continue
        slots.append({
            'level': s.get('l', 0), 'hp': s.get('h', 0), 'max_hp': s.get('m', 0),
            'atk': s.get('a', 0), 'def': s.get('d', 0), 'spd': s.get('sp', 0),
            'spatk': s.get('sa', 0), 'spdef': s.get('sd', 0), 'status': s.get('st', 0),
        })
    return {'count': count, 'slots': slots}


def parse_menu_data(data):
    mu = data.get('mu')
    if mu is None:
        return DEFAULT_MENU_DATA.copy()
    return {'mc': mu.get('mc', -1), 'mm': mu.get('mm', -1),
            'pc': mu.get('pc', -1), 'sc': mu.get('sc', -1)}


def parse_bag_data(data):
    bg = data.get('bg')
    if bg is None:
        return DEFAULT_BAG_DATA.copy()
    items = []
    for item in bg.get('it', []):
        if isinstance(item, dict) and 'id' in item:
            items.append({'id': item.get('id', 0), 'q': item.get('q', 0)})
    return {'pocket': bg.get('pk', -1), 'cursor': bg.get('bc', -1),
            'active': bg.get('a', 0), 'items': items}


def parse_game_state_data(data):
    raw = data.get("state") or data.get("s") or []
    palette_raw = data.get("palette") or data.get("p") or []
    tiles_raw = data.get("tiles") or data.get("t") or []
    dead = bool(data.get("dead", False))
    battle_data = parse_battle_data(data)
    party_data = parse_party_data(data)
    game_state_raw = data.get("gs", 0)
    menu_data = parse_menu_data(data)
    bag_data = parse_bag_data(data)
    text_flag = int(data.get("tf", 0))  # NEW: 0=no text, 1=text box active
    return raw, palette_raw, tiles_raw, dead, battle_data, party_data, game_state_raw, menu_data, bag_data, text_flag



def read_game_state(max_retries=3):
    """Now returns 11 values (was 10). New 11th value: text_flag (int, 0 or 1)."""
    if not STATE_FILE.exists():
        return (np.zeros(EXPECTED_STATE_DIM), np.zeros(PALETTE_DIM), np.zeros(TILE_DIM),
                False, (0, 0), DEFAULT_BATTLE_DATA.copy(), DEFAULT_PARTY_DATA.copy(),
                0, DEFAULT_MENU_DATA.copy(), DEFAULT_BAG_DATA.copy(), 0)

    for attempt in range(max_retries):
        try:
            with open(STATE_FILE, "r") as f:
                data = json.loads(f.read())

            (raw, palette_raw, tiles_raw, dead, battle_data, party_data,
             game_state_raw, menu_data, bag_data, text_flag) = parse_game_state_data(data)

            raw_x = int(raw[0]) if len(raw) > 0 else 0
            raw_y = int(raw[1]) if len(raw) > 1 else 0
            raw_position = (raw_x, raw_y)

            context_state = normalize_game_state(np.array(raw, dtype=float))
            palette_state = np.array(palette_raw, dtype=float) if palette_raw else np.zeros(PALETTE_DIM)
            tile_state = np.array(tiles_raw, dtype=float) if tiles_raw else np.zeros(TILE_DIM)

            context_state = _pad_or_trim(context_state, EXPECTED_STATE_DIM)
            palette_state = _pad_or_trim(palette_state, PALETTE_DIM)
            tile_state = _pad_or_trim(tile_state, TILE_DIM)

            return (context_state, palette_state, tile_state, dead, raw_position,
                    battle_data, party_data, game_state_raw, menu_data, bag_data, text_flag)

        except (json.JSONDecodeError, ValueError):
            if attempt < max_retries - 1:
                time.sleep(0.001)
                continue
            return (np.zeros(EXPECTED_STATE_DIM), np.zeros(PALETTE_DIM), np.zeros(TILE_DIM),
                    False, (0, 0), DEFAULT_BATTLE_DATA.copy(), DEFAULT_PARTY_DATA.copy(),
                    0, DEFAULT_MENU_DATA.copy(), DEFAULT_BAG_DATA.copy(), 0)
        except Exception:
            return (np.zeros(EXPECTED_STATE_DIM), np.zeros(PALETTE_DIM), np.zeros(TILE_DIM),
                    False, (0, 0), DEFAULT_BATTLE_DATA.copy(), DEFAULT_PARTY_DATA.copy(),
                    0, DEFAULT_MENU_DATA.copy(), DEFAULT_BAG_DATA.copy(), 0)
        

def write_action(action_name):
    if action_name:
        action_name = action_name.upper()
    try:
        with open(ACTION_FILE, "w") as f:
            json.dump({"action": action_name}, f)
            f.flush()
    except Exception as e:
        print(f"[ERROR] Failed to write action: {e}")

📂 BASE_PATH: C:\Users\HP\Documents\cogai


In [2]:
# ============================================================================
# CELL 2: Perceptron Classes + Pool/Pipeline Infrastructure
# ============================================================================
# SYNC TO AI AGENT v17.4 (Multi-Pool Pipeline):
# 1. ActivationLibrary — UNCHANGED from v17.2
# 2. Perceptron class — UPDATED:
#    - NEW: pool_id, layer_index, trigger_context fields
#    - NEW: get_pool_state() / set_pool_state() for serialization
#    - All other methods UNCHANGED (predict, update, fit_activation, etc.)
# 3. ControlSwapPerceptron — UNCHANGED
# 4. NEW: Pool class — fixed-width output layer containing perceptrons
#    - Computes output vector from internal perceptrons (weighted avg)
#    - Spawn threshold (novelty-based, adaptive)
#    - Memory paging (serialize pruned perceptrons to residual file)
#    - Authority tracking (how much the pool should be trusted)
# 5. NEW: Pipeline class — ordered sequence of Pools forming a processing chain
#    - Each pool takes input from previous pool's output (or raw state for L0)
#    - Backward credit assignment with decay factor per layer
#    - Fallback to neutral vectors when pools are empty
# ============================================================================


class ActivationLibrary:
    """
    Shared library of candidate activation functions.
    Each candidate: {name, func, category, suited_for, param}
    Starts with standard functions, can grow via discovery.
    """

    def __init__(self):
        self.candidates = []
        self._build_standard_library()

    def _build_standard_library(self):
        self.candidates = [
            {
                'name': 'linear',
                'func': lambda x: x,
                'category': 'standard',
                'suited_for': ['action_continuous', 'entity'],
                'param': None,
            },
            {
                'name': 'tanh',
                'func': lambda x: np.tanh(x),
                'category': 'standard',
                'suited_for': ['entity', 'action_continuous'],
                'param': None,
            },
            {
                'name': 'relu',
                'func': lambda x: max(0.0, x),
                'category': 'standard',
                'suited_for': ['action_continuous'],
                'param': None,
            },
            {
                'name': 'sigmoid',
                'func': lambda x: 1.0 / (1.0 + np.exp(-np.clip(x, -20.0, 20.0))),
                'category': 'standard',
                'suited_for': ['action_binary', 'entity'],
                'param': None,
            },
            {
                'name': 'bounded_linear',
                'func': lambda x: np.clip(x, -1.0, 1.0),
                'category': 'standard',
                'suited_for': ['action_continuous', 'entity'],
                'param': None,
            },
            {
                'name': 'soft_threshold_0.1',
                'func': lambda x: x if abs(x) > 0.1 else 0.0,
                'category': 'standard',
                'suited_for': ['entity', 'action_continuous'],
                'param': 0.1,
            },
            {
                'name': 'soft_threshold_0.3',
                'func': lambda x: x if abs(x) > 0.3 else 0.0,
                'category': 'standard',
                'suited_for': ['entity'],
                'param': 0.3,
            },
            {
                'name': 'leaky_relu',
                'func': lambda x: x if x > 0 else 0.01 * x,
                'category': 'standard',
                'suited_for': ['action_continuous', 'entity'],
                'param': None,
            },
            {
                'name': 'abs',
                'func': lambda x: abs(x),
                'category': 'standard',
                'suited_for': ['entity'],
                'param': None,
            },
            {
                'name': 'squared',
                'func': lambda x: x * abs(x),
                'category': 'standard',
                'suited_for': ['entity', 'action_continuous'],
                'param': None,
            },
        ]

    def get_candidates(self, suited_for=None):
        if suited_for is None:
            return self.candidates
        return [c for c in self.candidates if suited_for in c.get('suited_for', [])]

    def get_by_name(self, name):
        for c in self.candidates:
            if c['name'] == name:
                return c
        return None

    def add_discovered(self, name, func, suited_for, param=None):
        if self.get_by_name(name) is not None:
            return
        self.candidates.append({
            'name': name, 'func': func, 'category': 'discovered',
            'suited_for': suited_for, 'param': param,
        })

    def get_names(self):
        return [c['name'] for c in self.candidates]


# Global shared library
ACTIVATION_LIBRARY = ActivationLibrary()


class Perceptron:
    def __init__(self, kind, action=None, group=None, entity_type=None, chain="shared"):
        self.kind = kind
        self.action = action
        self.group = group
        self.entity_type = entity_type
        self.chain = chain

        self.utility = 1.0
        self.weights = None

        self.eligibility_fast = 0.0
        self.eligibility_slow = 0.0

        self.familiarity = 0.0
        self.activation_history = deque(maxlen=10)
        self.cluster_activations = deque(maxlen=50)

        self.learning_rate = 0.01
        self.prediction_errors = deque(maxlen=50)

        # === POOL MEMBERSHIP (NEW — matches AI agent) ===
        # Which pool this perceptron belongs to (None = legacy flat system)
        self.pool_id = None
        # Which layer index within the pipeline (0-based, None = legacy)
        self.layer_index = None
        # Context that triggered this perceptron's creation
        # Used as key for residual file (e.g. species ID, map region, type pair)
        self.trigger_context = None

        # === EMPIRICAL ACTIVATION DISCOVERY ===
        self.activation_observations = deque(maxlen=200)
        self.active_activation = 'linear'
        self.ACTIVATION_FIT_INTERVAL = 100
        self.ACTIVATION_MIN_OBSERVATIONS = 30
        self._activation_update_counter = 0
        self.activation_fit_score = 0.0
        self.activation_fit_history = deque(maxlen=10)
        self.activation_change_count = 0

    def _get_activation_func(self):
        candidate = ACTIVATION_LIBRARY.get_by_name(self.active_activation)
        if candidate is not None:
            return candidate['func']
        return lambda x: x

    def _apply_activation(self, x):
        func = self._get_activation_func()
        try:
            return func(x)
        except (ValueError, OverflowError):
            return x

    def _get_suited_for_hint(self):
        if self.kind == "entity":
            return "entity"
        if self.kind == "action":
            if self.group == "interact":
                return "action_binary"
            if self.group == "move":
                return "action_continuous"
        return "action_continuous"

    def _evaluate_activation_fit(self, candidate_func):
        if len(self.activation_observations) < self.ACTIVATION_MIN_OBSERVATIONS:
            return 0.0

        observations = list(self.activation_observations)
        raw_inputs = [obs[0] for obs in observations]
        errors = [obs[1] for obs in observations]

        try:
            candidate_outputs = [candidate_func(x) for x in raw_inputs]
        except (ValueError, OverflowError):
            return -1.0

        abs_outputs = np.array([abs(o) for o in candidate_outputs])
        abs_errors = np.array([abs(e) for e in errors])

        if np.std(abs_outputs) < 1e-10 or np.std(abs_errors) < 1e-10:
            return 0.0

        correlation = np.corrcoef(abs_outputs, abs_errors)[0, 1]
        if np.isnan(correlation):
            return 0.0

        output_variance = np.std(abs_outputs)
        variance_bonus = min(0.2, output_variance * 0.5)

        max_output = max(abs_outputs)
        if max_output > 100:
            return correlation * 0.5

        return correlation + variance_bonus

    def fit_activation(self):
        if len(self.activation_observations) < self.ACTIVATION_MIN_OBSERVATIONS:
            return

        hint = self._get_suited_for_hint()
        suited_candidates = ACTIVATION_LIBRARY.get_candidates(suited_for=hint)
        all_candidates = ACTIVATION_LIBRARY.candidates

        scored = []
        suited_names = {c['name'] for c in suited_candidates}

        for candidate in all_candidates:
            score = self._evaluate_activation_fit(candidate['func'])
            if candidate['name'] in suited_names:
                score += 0.05
            scored.append((candidate['name'], score))

        scored.sort(key=lambda x: x[1], reverse=True)
        if not scored:
            return

        best_name, best_score = scored[0]
        current_score = self._evaluate_activation_fit(self._get_activation_func())

        self.activation_fit_score = current_score
        self.activation_fit_history.append(current_score)

        SWITCH_THRESHOLD = 0.1
        if best_name != self.active_activation and best_score > current_score + SWITCH_THRESHOLD:
            old_name = self.active_activation
            self.active_activation = best_name
            self.activation_change_count += 1

            if self.activation_change_count <= 5 or self.activation_change_count % 10 == 0:
                id_str = self.action or self.entity_type or "?"
                print(f"  🧬 ACTIVATION [{self.chain}:{id_str}]: {old_name} → {best_name} "
                      f"(score {current_score:.3f} → {best_score:.3f}, "
                      f"change #{self.activation_change_count})")

    def ensure_weights(self, dim):
        if self.weights is None:
            self.weights = np.random.randn(dim) * 0.001
        elif len(self.weights) != dim:
            old_weights = self.weights
            self.weights = np.random.randn(dim) * 0.001
            min_len = min(len(old_weights), dim)
            self.weights[:min_len] = old_weights[:min_len]

    def predict(self, state):
        self.ensure_weights(len(state))

        if len(self.weights) != len(state):
            min_dim = min(len(self.weights), len(state))
            raw_dot = np.dot(self.weights[:min_dim], state[:min_dim])
        else:
            raw_dot = np.dot(self.weights, state)

        activated = self._apply_activation(raw_dot)

        if self.kind == "entity":
            novelty_factor = 1.0 / (1.0 + np.sqrt(self.familiarity * 0.5))
            decayed = activated * novelty_factor
            self.activation_history.append(abs(raw_dot))
            self.cluster_activations.append(abs(raw_dot))
            return decayed
        else:
            return activated

    def adapt_learning_rate(self):
        if len(self.prediction_errors) >= 50:
            avg_error = np.mean(self.prediction_errors)
            if avg_error < 0.1:
                self.learning_rate = max(0.001, self.learning_rate * 0.99)
            elif avg_error > 0.5:
                self.learning_rate = min(0.05, self.learning_rate * 1.01)

    def update(self, state, error, gamma_fast=0.5, gamma_slow=0.95, stagnation=0.0):
        self.ensure_weights(len(state))

        if len(self.weights) != len(state):
            min_dim = min(len(self.weights), len(state))
            state = state[:min_dim]
            self.weights = self.weights[:min_dim]

        self.eligibility_fast = gamma_fast * self.eligibility_fast + 1.0
        self.eligibility_slow = gamma_slow * self.eligibility_slow + 1.0

        self.adapt_learning_rate()

        fast_update = 0.7 * self.learning_rate * error * state * self.eligibility_fast
        slow_update = 0.3 * self.learning_rate * error * state * self.eligibility_slow
        self.weights += fast_update + slow_update

        if self.kind == "action":
            if error > 0.01:
                if stagnation > 0.5:
                    self.utility *= 0.97
                elif error > 0.2:
                    self.utility = min(self.utility * 1.02, 2.0)
                else:
                    self.utility *= 0.995

            if self.group == "move":
                self.utility = np.clip(self.utility, 0.1, 2.0)
            else:
                self.utility = np.clip(self.utility, 0.01, 2.0)

        if self.kind == "entity" and len(self.activation_history) > 0:
            recent_avg = np.mean(self.activation_history)
            if recent_avg > 0.1:
                self.familiarity += 0.03

        if self.kind == "entity":
            prediction = self.predict(state)
            self.prediction_errors.append(abs(prediction - error))

        # Record observation for activation fitting
        raw_dot = np.dot(self.weights, state) if len(self.weights) == len(state) else 0.0
        self.activation_observations.append((raw_dot, error))

        # Periodic activation fitting
        self._activation_update_counter += 1
        if self._activation_update_counter >= self.ACTIVATION_FIT_INTERVAL:
            self._activation_update_counter = 0
            self.fit_activation()

    def get_activation_state(self):
        return {
            'active_activation': self.active_activation,
            'fit_score': float(self.activation_fit_score),
            'change_count': self.activation_change_count,
            'observations_count': len(self.activation_observations),
        }

    def set_activation_state(self, state_dict):
        if state_dict is None:
            return
        self.active_activation = state_dict.get('active_activation', 'linear')
        self.activation_fit_score = state_dict.get('fit_score', 0.0)
        self.activation_change_count = state_dict.get('change_count', 0)

    def get_pool_state(self):
        """Get pool membership state for save/load."""
        return {
            'pool_id': self.pool_id,
            'layer_index': self.layer_index,
            'trigger_context': self.trigger_context,
        }

    def set_pool_state(self, state_dict):
        """Restore pool membership state from save/load."""
        if state_dict is None:
            return
        self.pool_id = state_dict.get('pool_id')
        self.layer_index = state_dict.get('layer_index')
        self.trigger_context = state_dict.get('trigger_context')


class ControlSwapPerceptron(Perceptron):
    def __init__(self):
        super().__init__(kind="control_swap", chain="shared")
        self.swap_history = deque(maxlen=100)
        self.confidence = 0.0

    def should_swap(self, state, movement_stagnation):
        if self.weights is None:
            return False, 0.0
        self.ensure_weights(len(state))
        swap_score = self.predict(state)
        stagnation_factor = np.tanh(movement_stagnation / 5.0)
        combined_score = swap_score * 0.7 + stagnation_factor * 0.3
        return combined_score > 0.5, abs(combined_score)

    def record_swap_outcome(self, state, swapped, novelty_gained):
        self.swap_history.append((swapped, novelty_gained))
        if len(self.swap_history) >= 20:
            recent = list(self.swap_history)[-20:]
            successful = sum(1 for swap, nov in recent if swap and nov > 0.2)
            self.confidence = successful / 20.0


# ============================================================================
# POOL CLASS — Fixed-Width Output Layer of Perceptrons
# ============================================================================

class Pool:
    """
    A pool is a layer within a pipeline. It contains zero or more perceptrons
    that share a semantic role. The pool produces a fixed-width output vector
    regardless of how many perceptrons it contains.

    When empty, it produces a neutral (zero) vector — triggering fallback
    to Markov/hardcoded systems downstream.

    Key properties:
    - output_width: fixed dimensionality of output (default 8)
    - perceptrons: list of perceptrons in this pool
    - spawn_threshold: novelty threshold for creating new perceptrons
    - authority: how much this pool's output should be trusted (0.0-1.0)
    - residual: serialized pruned perceptrons keyed by trigger_context
    """

    DEFAULT_OUTPUT_WIDTH = 8
    DEFAULT_MAX_PERCEPTRONS = 20
    AUTHORITY_FIT_THRESHOLD = 5.0
    RESIDUAL_MAX_ENTRIES = 50

    def __init__(self, pool_id, name, output_width=None, max_perceptrons=None):
        self.pool_id = pool_id
        self.name = name
        self.output_width = output_width or self.DEFAULT_OUTPUT_WIDTH
        self.max_perceptrons = max_perceptrons or self.DEFAULT_MAX_PERCEPTRONS

        self.perceptron_ids = []

        self.spawn_threshold = 0.0005
        self.spawn_count = 0
        self.last_spawn_timestep = 0

        self.authority = 0.0
        self.error_history = deque(maxlen=200)
        self.residual = {}

        self._cached_output = np.zeros(self.output_width)
        self._cached_output_valid = False

    def compute_output(self, input_vector, brain_perceptrons):
        pool_perceptrons = self._get_perceptrons(brain_perceptrons)

        if not pool_perceptrons:
            self._cached_output = np.zeros(self.output_width)
            self._cached_output_valid = True
            self.authority = 0.0
            return self._cached_output.copy()

        activations = []
        weights_sum = 0.0

        for p in pool_perceptrons:
            pred = p.predict(input_vector)
            weight = max(0.01, p.utility)
            activations.append((pred, weight, p))
            weights_sum += weight

        if weights_sum < 1e-10:
            self._cached_output = np.zeros(self.output_width)
            self._cached_output_valid = True
            return self._cached_output.copy()

        output = np.zeros(self.output_width)

        for i, (pred, weight, p) in enumerate(activations):
            dim_idx = i % self.output_width
            output[dim_idx] += pred * (weight / weights_sum)

        output_norm = np.linalg.norm(output)
        if output_norm > 0:
            output = output / max(1.0, output_norm)

        self._cached_output = output
        self._cached_output_valid = True

        avg_fit = np.mean([p.activation_fit_score for p in pool_perceptrons])
        n = len(pool_perceptrons)
        self.authority = min(1.0, n * max(0.01, avg_fit) / self.AUTHORITY_FIT_THRESHOLD)

        return output.copy()

    def get_cached_output(self):
        return self._cached_output.copy()

    def invalidate_cache(self):
        self._cached_output_valid = False

    def _get_perceptrons(self, brain_perceptrons):
        pool_id = self.pool_id
        return [p for p in brain_perceptrons if p.pool_id == pool_id]

    def get_perceptron_count(self, brain_perceptrons):
        return len(self._get_perceptrons(brain_perceptrons))

    def needs_spawn(self, error):
        return error > self.spawn_threshold

    def update_spawn_threshold(self, percentile=75):
        if len(self.error_history) >= 50:
            self.spawn_threshold = max(0.001, np.percentile(list(self.error_history), percentile))

    def record_error(self, error):
        self.error_history.append(error)

    def should_cluster(self, brain_perceptrons):
        return self.get_perceptron_count(brain_perceptrons) >= self.max_perceptrons

    # =================================================================
    # RESIDUAL FILE (Memory Paging)
    # =================================================================

    def page_to_residual(self, perceptron):
        key = perceptron.trigger_context
        if key is None:
            key = f"_unnamed_{self.pool_id}_{perceptron.entity_type or perceptron.action or 'unknown'}"

        entry = {
            'kind': perceptron.kind,
            'action': perceptron.action,
            'group': perceptron.group,
            'entity_type': perceptron.entity_type,
            'chain': perceptron.chain,
            'utility': float(perceptron.utility),
            'familiarity': float(perceptron.familiarity),
            'learning_rate': float(perceptron.learning_rate),
            'active_activation': perceptron.active_activation,
            'activation_fit_score': float(perceptron.activation_fit_score),
            'trigger_context': perceptron.trigger_context,
            'weights_shape': len(perceptron.weights) if perceptron.weights is not None else 0,
            'weights_nonzero': [[i, float(v)] for i, v in enumerate(perceptron.weights)
                                if abs(v) > 1e-10] if perceptron.weights is not None else [],
            'paged_at': 0,
        }

        self.residual[str(key)] = entry

        if len(self.residual) > self.RESIDUAL_MAX_ENTRIES:
            oldest_key = min(self.residual.keys(),
                            key=lambda k: self.residual[k].get('paged_at', 0))
            del self.residual[oldest_key]

    def restore_from_residual(self, trigger_context):
        key = str(trigger_context)
        if key not in self.residual:
            return None

        entry = self.residual.pop(key)

        p = Perceptron(
            kind=entry['kind'],
            action=entry.get('action'),
            group=entry.get('group'),
            entity_type=entry.get('entity_type'),
            chain=entry.get('chain', 'shared'),
        )
        p.utility = entry.get('utility', 1.0)
        p.familiarity = entry.get('familiarity', 0.0)
        p.learning_rate = entry.get('learning_rate', 0.01)
        p.active_activation = entry.get('active_activation', 'linear')
        p.activation_fit_score = entry.get('activation_fit_score', 0.0)
        p.trigger_context = entry.get('trigger_context')
        p.pool_id = self.pool_id
        p.layer_index = None  # caller sets this

        ws = entry.get('weights_shape', 0)
        if ws > 0 and entry.get('weights_nonzero'):
            p.weights = np.zeros(ws)
            for idx, val in entry['weights_nonzero']:
                if idx < ws:
                    p.weights[idx] = val

        return p

    def has_residual(self, trigger_context):
        return str(trigger_context) in self.residual

    # =================================================================
    # SERIALIZATION
    # =================================================================

    def get_save_state(self):
        return {
            'pool_id': self.pool_id,
            'name': self.name,
            'output_width': self.output_width,
            'max_perceptrons': self.max_perceptrons,
            'spawn_threshold': float(self.spawn_threshold),
            'spawn_count': self.spawn_count,
            'authority': float(self.authority),
            'residual': self.residual,
        }

    def load_save_state(self, state_dict):
        if state_dict is None:
            return
        self.output_width = state_dict.get('output_width', self.DEFAULT_OUTPUT_WIDTH)
        self.max_perceptrons = state_dict.get('max_perceptrons', self.DEFAULT_MAX_PERCEPTRONS)
        self.spawn_threshold = state_dict.get('spawn_threshold', 0.0005)
        self.spawn_count = state_dict.get('spawn_count', 0)
        self.authority = state_dict.get('authority', 0.0)
        self.residual = state_dict.get('residual', {})


# ============================================================================
# PIPELINE CLASS — Ordered Sequence of Pools
# ============================================================================

class Pipeline:
    """
    A pipeline is an ordered sequence of pools (layers) that process
    game state through increasing levels of abstraction.

    Each pipeline corresponds to a game context:
    - battle_pipeline (6 layers)
    - overworld_pipeline (7 layers)
    - bag_pipeline (3 layers)
    - party_pipeline (2 layers)

    Key behaviors:
    - Layer 0 pools take raw game state as input
    - Layer N pools (N>0) take the fixed-width output of Layer N-1
    - When a pool is empty, it passes a neutral vector
    - Credit assignment flows backward with decay factor per layer
    - If ALL pools are empty, the pipeline signals fallback to Markov/hardcoded
    """

    DEFAULT_CREDIT_DECAY = 0.7

    def __init__(self, pipeline_id, name, pool_definitions, credit_decay=None):
        self.pipeline_id = pipeline_id
        self.name = name
        self.credit_decay = credit_decay or self.DEFAULT_CREDIT_DECAY

        self.pools = []
        for i, pdef in enumerate(pool_definitions):
            pool_id = f"{pipeline_id}_L{i}_{pdef['name']}"
            pool = Pool(
                pool_id=pool_id,
                name=pdef['name'],
                output_width=pdef.get('output_width', Pool.DEFAULT_OUTPUT_WIDTH),
                max_perceptrons=pdef.get('max_perceptrons', Pool.DEFAULT_MAX_PERCEPTRONS),
            )
            self.pools.append(pool)

        self.num_layers = len(self.pools)

        self._layer_inputs = []
        self._layer_outputs = []

        self.active = False

    def forward(self, raw_input, brain_perceptrons):
        self._layer_inputs = []
        self._layer_outputs = []

        current_input = raw_input
        any_active = False

        for i, pool in enumerate(self.pools):
            self._layer_inputs.append(current_input.copy())

            output = pool.compute_output(current_input, brain_perceptrons)
            self._layer_outputs.append(output.copy())

            if pool.authority > 0.0:
                any_active = True

            current_input = output

        self.active = any_active

        return self._layer_outputs[-1] if self._layer_outputs else np.zeros(
            self.pools[-1].output_width if self.pools else Pool.DEFAULT_OUTPUT_WIDTH
        ), any_active

    def backward(self, error_signal, brain_perceptrons):
        if not self._layer_inputs:
            return

        for i in reversed(range(self.num_layers)):
            pool = self.pools[i]

            layers_from_output = self.num_layers - i - 1
            layer_error = error_signal * (self.credit_decay ** layers_from_output)

            pool.record_error(abs(layer_error))
            pool.update_spawn_threshold()

            layer_input = self._layer_inputs[i]
            pool_perceptrons = [p for p in brain_perceptrons if p.pool_id == pool.pool_id]

            for p in pool_perceptrons:
                p.update(layer_input, layer_error)

    def get_pool_at_layer(self, layer_index):
        if 0 <= layer_index < len(self.pools):
            return self.pools[layer_index]
        return None

    def get_input_width_for_layer(self, layer_index):
        if layer_index == 0:
            return None
        return self.pools[layer_index - 1].output_width

    def is_fallback_needed(self):
        return not self.active

    def get_total_authority(self):
        if not self.pools:
            return 0.0
        return np.mean([p.authority for p in self.pools])

    def get_layer_authorities(self):
        return {pool.name: pool.authority for pool in self.pools}

    # =================================================================
    # SPAWNING INTO POOLS
    # =================================================================

    def spawn_into_pool(self, layer_index, perceptron, brain):
        pool = self.pools[layer_index]

        # Check residual first
        if perceptron.trigger_context and pool.has_residual(perceptron.trigger_context):
            restored = pool.restore_from_residual(perceptron.trigger_context)
            if restored is not None:
                restored.pool_id = pool.pool_id
                restored.layer_index = layer_index
                brain.add(restored)
                print(f"  🔄 RESTORED from residual: {pool.name} [{restored.trigger_context}]")
                return restored

        # Fresh spawn
        perceptron.pool_id = pool.pool_id
        perceptron.layer_index = layer_index
        brain.add(perceptron)
        pool.spawn_count += 1

        return perceptron

    def prune_from_pool(self, layer_index, perceptron, brain_perceptrons, timestep=0):
        pool = self.pools[layer_index]

        entry = pool.page_to_residual(perceptron)
        if entry and hasattr(entry, 'get'):
            entry['paged_at'] = timestep

        perceptron.pool_id = None
        perceptron.layer_index = None

        return True

    # =================================================================
    # SERIALIZATION
    # =================================================================

    def get_save_state(self):
        return {
            'pipeline_id': self.pipeline_id,
            'name': self.name,
            'credit_decay': self.credit_decay,
            'pools': [pool.get_save_state() for pool in self.pools],
        }

    def load_save_state(self, state_dict):
        if state_dict is None:
            return
        self.credit_decay = state_dict.get('credit_decay', self.DEFAULT_CREDIT_DECAY)
        pool_states = state_dict.get('pools', [])
        for i, ps in enumerate(pool_states):
            if i < len(self.pools):
                self.pools[i].load_save_state(ps)

    def get_status(self, brain_perceptrons):
        status = {
            'pipeline': self.pipeline_id,
            'active': self.active,
            'total_authority': self.get_total_authority(),
            'layers': [],
        }
        for i, pool in enumerate(self.pools):
            n = pool.get_perceptron_count(brain_perceptrons)
            status['layers'].append({
                'name': pool.name,
                'perceptrons': n,
                'authority': pool.authority,
                'spawn_count': pool.spawn_count,
                'residual_size': len(pool.residual),
                'spawn_threshold': pool.spawn_threshold,
            })
        return status

In [3]:
# ============================================================================
# CELL 3 PART 1: Brain Class — Init + Chain-Aware Data Management
# ============================================================================
# SYNC TO AI AGENT v17.4 (Multi-Pool Pipeline):
# 1. NEW: Pipeline definitions (battle, overworld, bag, party) in __init__
# 2. NEW: self.pipelines dict indexing all pipelines
# 3. NEW: self.revenge_targets dict for revenge module
# 4. NEW: self.RESIDUAL_FILE path for residual perceptrons
# 5. NEW: Pool-aware perceptron access methods
#    - pool_perceptrons(), get_pipeline(), get_pipeline_stats()
#    - get_pipeline_summary()
# 6. NEW: spawn_into_pipeline_pool() — spawns into specific pool layer
# 7. NEW: get_pool_spawn_context() — determines trigger_context
# 8. NEW: cluster_pipeline_pool() — clusters within a single pool
# 9. NEW: save_residual_file() / load_residual_file()
# 10. All existing teaching-specific functionality PRESERVED:
#     - Bag session recording
#     - Start menu session recording
#     - Human action tracking
#     - Map battle statistics
#     - Text flag management
#     - Type clusters state
# ============================================================================

import gc
import random

class Brain:
    def __init__(self):
        # === MASTER PERCEPTRON LIST ===
        self.perceptrons = []

        # =================================================================
        # === PIPELINE DEFINITIONS (NEW — matches AI agent) ===
        # =================================================================

        # --- BATTLE PIPELINE (6 layers) ---
        self.battle_pipeline = Pipeline(
            pipeline_id="battle",
            name="Battle Pipeline",
            pool_definitions=[
                {'name': 'identification',    'output_width': 8, 'max_perceptrons': 15},
                {'name': 'threat_assessment',  'output_width': 8, 'max_perceptrons': 20},
                {'name': 'stay_or_bail',       'output_width': 8, 'max_perceptrons': 15},
                {'name': 'action_selection',   'output_width': 8, 'max_perceptrons': 20},
                {'name': 'execution',          'output_width': 8, 'max_perceptrons': 10},
                {'name': 'outcome_observation','output_width': 8, 'max_perceptrons': 10},
            ],
            credit_decay=0.7,
        )

        # --- OVERWORLD PIPELINE (7 layers) ---
        self.overworld_pipeline = Pipeline(
            pipeline_id="overworld",
            name="Overworld Pipeline",
            pool_definitions=[
                {'name': 'spatial_awareness',    'output_width': 8, 'max_perceptrons': 15},
                {'name': 'area_classification',  'output_width': 8, 'max_perceptrons': 10},
                {'name': 'frontier_detection',   'output_width': 8, 'max_perceptrons': 15},
                {'name': 'objective_management', 'output_width': 8, 'max_perceptrons': 15},
                {'name': 'pathfinding',          'output_width': 8, 'max_perceptrons': 10},
                {'name': 'execution',            'output_width': 8, 'max_perceptrons': 10},
                {'name': 'outcome_observation',  'output_width': 8, 'max_perceptrons': 10},
            ],
            credit_decay=0.7,
        )

        # --- BAG PIPELINE (3 layers) ---
        self.bag_pipeline = Pipeline(
            pipeline_id="bag",
            name="Bag Pipeline",
            pool_definitions=[
                {'name': 'inventory_awareness', 'output_width': 8, 'max_perceptrons': 10},
                {'name': 'item_selection',      'output_width': 8, 'max_perceptrons': 10},
                {'name': 'execution',           'output_width': 8, 'max_perceptrons': 8},
            ],
            credit_decay=0.7,
        )

        # --- PARTY PIPELINE (2 layers) ---
        self.party_pipeline = Pipeline(
            pipeline_id="party",
            name="Party Pipeline",
            pool_definitions=[
                {'name': 'assessment', 'output_width': 8, 'max_perceptrons': 10},
                {'name': 'execution',  'output_width': 8, 'max_perceptrons': 8},
            ],
            credit_decay=0.7,
        )

        # All pipelines indexed by id
        self.pipelines = {
            'battle': self.battle_pipeline,
            'overworld': self.overworld_pipeline,
            'bag': self.bag_pipeline,
            'party': self.party_pipeline,
        }

        # =================================================================
        # === RESIDUAL PERCEPTRONS FILE (NEW) ===
        # =================================================================
        self.RESIDUAL_FILE = BASE_PATH / "residual_perceptrons.json"

        # =================================================================
        # === REVENGE MODULE (NEW — matches AI agent) ===
        # =================================================================
        self.revenge_targets = {}
        self.REVENGE_BASE_MARGIN = 2
        self.REVENGE_MARGIN_PER_LOSS = 2
        self.REVENGE_MAX_MARGIN = 15
        self.REVENGE_MAX_TARGETS = 10

        # === PER-CHAIN LEARNING STATE HISTORY ===
        self.prev_learning_states = deque(maxlen=50)
        self.prev_battle_learning_states = deque(maxlen=50)
        self.prev_party_learning_states = deque(maxlen=20)
        self.prev_bag_learning_states = deque(maxlen=20)

        self.prev_context_states = deque(maxlen=10)
        self.last_positions = deque(maxlen=30)
        self.action_history = deque(maxlen=100)

        self.control_mode = "move"
        self.timestep = 0
        self.last_action = None
        self.last_direction = 0

        # === HUMAN ACTION TRACKING ===
        self.human_action = None

        self.MOVE_UTILITY_FLOOR = 0.05
        self.INTERACT_UTILITY_FLOOR = 0.15

        # === PER-CHAIN ENTITY CAPACITY (matches AI agent) ===
        self.ENTITY_CAPACITY = {
            'overworld': 20,
            'battle': 10,
            'party': 5,
            'bag': 5,
            'shared': 10,
        }
        self.ENTITY_CAPACITY_GROWTH = 1.5

        self.entity_spawn_counts = {'overworld': 0, 'battle': 0, 'party': 0, 'bag': 0, 'shared': 0}
        self.entity_merge_counts = {'overworld': 0, 'battle': 0, 'party': 0, 'bag': 0, 'shared': 0}
        self.ENTITY_CLUSTER_SIMILARITY = 0.85
        self.ENTITY_MIN_ACTIVATIONS = 10

        self.innate_entities_spawned_overworld = False
        self.innate_entities_spawned_battle = False

        # Legacy compat
        self.entity_capacity = self.ENTITY_CAPACITY['overworld']
        self.entity_spawn_count = 0
        self.entity_merge_count = 0
        self.innate_entities_spawned = False

        # === EXPLORATION MEMORY ===
        self.EXPLORATION_MEMORY_FILE = BASE_PATH / "taught_exploration_memory.json"
        self.exploration_memory = {}
        self.current_map_id = None
        self.SAVE_INTERVAL = 100
        self.MAX_MAPS_IN_MEMORY = 20

        self.DIRECTION_NAMES = {0: "DOWN", 1: "UP", 2: "LEFT", 3: "RIGHT"}
        self.DIRECTION_TO_INT = {"DOWN": 0, "UP": 1, "LEFT": 2, "RIGHT": 3}
        self.INT_TO_ACTION = {0: "DOWN", 1: "UP", 2: "LEFT", 3: "RIGHT"}
        self.DIRECTION_DELTAS_INT = {0: (0, 1), 1: (0, -1), 2: (-1, 0), 3: (1, 0)}
        self.ACTION_DELTAS = {"UP": (0, -1), "DOWN": (0, 1), "LEFT": (-1, 0), "RIGHT": (1, 0)}
        self.DELTA_TO_DIRECTION = {(0, 1): 0, (0, -1): 1, (-1, 0): 2, (1, 0): 3}

        self.load_exploration_memory()

        # === MARKOV TRANSITION SYSTEM ===
        self.taught_transitions = []
        self.taught_batches = []
        self.taught_metadata = {}
        self.markov_enabled = True
        self.markov_action_count = 0
        self.curiosity_action_count = 0
        self.last_markov_score = 0.0
        self.last_markov_action = None
        self.recent_actions_buffer = deque(maxlen=15)

        # === TAUGHT MODEL REFERENCE ===
        self.taught_reference = {'utilities': {}, 'weights': {}, 'loaded': False}

        # === BLEND SYSTEM ===
        self.blend_tier = 0
        self.last_blend_timestep = 0
        self.BLEND_COOLDOWN = 50
        self.blend_count = 0
        self.BLEND_RATIOS = {1: (0.80, 0.20), 2: (0.60, 0.40), 3: (0.40, 0.60)}
        self.BLEND_TIER_TRIGGERS = {
            1: {'pattern_repeats': 3, 'pos_stagnation': 8, 'consecutive': 12},
            2: {'pattern_repeats': 6, 'pos_stagnation': 15, 'consecutive': 15},
            3: {'pattern_repeats': 10, 'state_stagnation_mult': 2.0}
        }

        # === BATTLE DATA (EXPANDED) ===
        self.battle_data = DEFAULT_BATTLE_DATA.copy()
        self.prev_battle_data = DEFAULT_BATTLE_DATA.copy()
        self.battle_data_buffer = []
        self.in_battle_last_frame = False
        self.current_battle_id = 0

        # === PARTY DATA ===
        self.party_data = DEFAULT_PARTY_DATA.copy()
        self.prev_party_data = DEFAULT_PARTY_DATA.copy()

        # === MENU DATA ===
        self.menu_data = DEFAULT_MENU_DATA.copy()
        self.prev_menu_data = DEFAULT_MENU_DATA.copy()

        # === BAG DATA ===
        self.bag_data = DEFAULT_BAG_DATA.copy()
        self.prev_bag_data = DEFAULT_BAG_DATA.copy()

        # === RAW GAME STATE ===
        self.game_state_raw = 0
        self.prev_game_state_raw = 0
        self.text_flag = 0
        self.prev_text_flag = 0

        # === MAP BATTLE STATISTICS (matches AI agent Phase 4) ===
        self.map_battle_stats = {}
        self.map_battle_stats_dirty = False
        self.map_step_counters = {}

        # === BAG SESSION RECORDING ===
        self.bag_session_active = False
        self.bag_session_context = "none"
        self.bag_session_start_timestep = 0
        self.bag_session_frame_count = 0
        self.bag_session_last_action = None
        self.bag_session_items_used = []
        self.bag_session_pockets_visited = set()
        self.bag_session_frames = []

        self.bag_sessions_accumulated = []
        self.bag_sessions_metadata = {
            'total_bag_frames': 0,
            'bag_sessions_recorded': 0,
            'items_used': set(),
            'pockets_visited': set(),
        }

        # =================================================================
        # === START MENU SESSION RECORDING ===
        # =================================================================
        self.start_menu_session_active = False
        self.start_menu_session_start_timestep = 0
        self.start_menu_session_frame_count = 0
        self.start_menu_session_last_action = None
        self.start_menu_session_frames = []
        self.start_menu_session_last_mc = -1

        self.start_menu_sessions_accumulated = []
        self.start_menu_sessions_metadata = {
            'total_start_menu_frames': 0,
            'start_menu_sessions_recorded': 0,
            'targets_navigated': {},
        }

        # =================================================================
        # === TYPE CLUSTERS STATE ===
        # =================================================================
        self.move_type_clusters = {}
        self.species_type_clusters = {}
        self.cluster_effectiveness = {}
        self.move_to_cluster = {}
        self.species_to_cluster = {}
        self.clustering_run_count = 0
        self.last_clustering_timestep = 0
        self.type_clusters_dirty = False

        # === START MENU STATS (for checkpoint) ===
        self.start_menu_total_actions = 0
        self.start_menu_markov_actions = 0

        # === ACTION EXECUTION ===
        self.pending_action = None
        self.pending_action_frames = 0
        self.ACTION_CONFIRM_FRAMES = 3
        self.last_confirmed_action = None

        # === TILE INTERACTION ===
        self.INTERACTION_VERIFY_FRAMES = 8
        self.MIN_SUCCESS_RATE_THRESHOLD = 0.1
        self.pending_interaction_verify = None
        self.interaction_verify_countdown = 0

        # === MENU TRAP ===
        self.menu_trap_frames = 0
        self.menu_trap_b_boost = 1.0
        self.menu_trap_position = None
        self.B_BOOST_INCREMENT = 0.15
        self.B_BOOST_MAX = 3.0
        self.MENU_TRAP_THRESHOLD = 5
        self.original_b_utility = None

        # === MODE SWAPPING ===
        self.DEFAULT_MOVE_TO_INTERACT_THRESHOLD = 15
        self.DEFAULT_INTERACT_TO_MOVE_THRESHOLD = 25
        self.move_to_interact_threshold = self.DEFAULT_MOVE_TO_INTERACT_THRESHOLD
        self.interact_to_move_threshold = self.DEFAULT_INTERACT_TO_MOVE_THRESHOLD
        self.THRESHOLD_INCREMENT = 15
        self.MAX_THRESHOLD = 150
        self.frames_in_current_mode = 0
        self.swap_chain_count = 0
        self.position_at_mode_swap = None
        self.last_map_id = None
        self.last_battle_state = None

        # === UNPRODUCTIVE SWAP ===
        self.UNPRODUCTIVE_SWAP_THRESHOLD = 3
        self.unproductive_swap_count = 0

        # === STATE STAGNATION ===
        self.STATE_STAGNATION_THRESHOLD = 20
        self.state_stagnation_count = 0
        self.last_context_state_hash = None
        self.stagnation_initiator_action = None

        # === BOTH MODE ===
        self.BOTH_MODE_STAGNATION_THRESHOLD = 35
        self.BOTH_MODE_SWAP_THRESHOLD = 5
        self.last_direction_for_progress = None
        self.direction_change_counts_as_progress = True

        # === NOVELTY WEIGHTS ===
        self.UNVISITED_TILE_BONUS = 1.5
        self.OBSTRUCTION_PENALTY = 0.25

        # === TRANSITIONS & DEBT ===
        self.TRANSITION_ATTRACTION_WEIGHT = 0.6
        self.TEMP_DEBT_ACCUMULATION = 0.5
        self.TEMP_DEBT_DECAY = 0.02
        self.TEMP_DEBT_MAX = 15.0
        self.MAX_MAP_DEBT = 10.0
        self.MAX_LOCATION_DEBT = 5.0
        self.DEBT_DECAY_RATE = 0.005

        # === TRANSITION BAN ===
        self.transition_bans = {}
        self.BAN_VICINITY_RADIUS = 3
        self.BAN_COVERAGE_LIFT_THRESHOLD = 0.6
        self.BAN_TIMEOUT_STEPS = 300

        # Multi-scale memory
        self.visited_maps = {}
        self.map_novelty_debt = {}
        self.location_memory = {}
        self.location_novelty = {}
        self.action_execution_count = {}
        self.MAX_LOCATIONS = 500

        self.swap_perceptron = ControlSwapPerceptron()

        # === ERROR HISTORY (overworld = legacy) ===
        self.error_history = deque(maxlen=1000)
        self.numeric_error_history = deque(maxlen=1000)
        self.visual_error_history = deque(maxlen=1000)

        # === PER-CHAIN ERROR HISTORY ===
        self.battle_error_history = deque(maxlen=500)
        self.party_error_history = deque(maxlen=200)
        self.bag_error_history = deque(maxlen=200)

        self._entity_norms_cache = {}
        self._cache_valid = False

        # === REPETITION ===
        self.consecutive_action_count = 0
        self.current_repeated_action = None
        self.LEARNING_SLOWDOWN_START = 3
        self.LEARNING_SLOWDOWN_MAX = 10
        self.PENALTY_THRESHOLD = 12
        self.HARD_RESET_THRESHOLD = 18

        # === PATTERN ===
        self.PATTERN_CHECK_WINDOW = 50
        self.PATTERN_MIN_REPEATS = 3
        self.PATTERN_MAX_LENGTH = 10
        self.detected_pattern = None
        self.pattern_repeat_count = 0

        # === PROBE CACHE ===
        self._cached_probe_action = None
        self._cached_probe_dir = None
        self._probe_cache_position = None

    # =========================================================================
    # CHAIN-SPECIFIC PERCEPTRON ACCESS (matches AI agent)
    # =========================================================================

    def actions(self, chain=None):
        if chain is None:
            return [p for p in self.perceptrons if p.kind == "action"]
        return [p for p in self.perceptrons if p.kind == "action" and p.chain == chain]

    def entities(self, chain=None):
        if chain is None:
            return [p for p in self.perceptrons if p.kind == "entity"]
        return [p for p in self.perceptrons if p.kind == "entity" and p.chain == chain]

    def add(self, p):
        self.perceptrons.append(p)
        self._cache_valid = False

    def get_chain_entity_count(self, chain):
        return len(self.entities(chain=chain))

    def get_chain_entity_capacity(self, chain):
        return self.ENTITY_CAPACITY.get(chain, 10)

    def get_chain_stats(self):
        stats = {}
        for chain in ['overworld', 'battle', 'party', 'bag', 'shared']:
            n_act = len(self.actions(chain=chain))
            n_ent = len(self.entities(chain=chain))
            if n_act > 0 or n_ent > 0:
                stats[chain] = {'actions': n_act, 'entities': n_ent}
        return stats

    # =========================================================================
    # POOL-SPECIFIC PERCEPTRON ACCESS (NEW — matches AI agent)
    # =========================================================================

    def pool_perceptrons(self, pool_id):
        """Get all perceptrons belonging to a specific pool."""
        return [p for p in self.perceptrons if p.pool_id == pool_id]

    def get_pipeline(self, pipeline_id):
        """Get a pipeline by ID."""
        return self.pipelines.get(pipeline_id)

    def get_pipeline_stats(self):
        """Get pool population stats for all pipelines."""
        stats = {}
        for pid, pipeline in self.pipelines.items():
            p_stats = pipeline.get_status(self.perceptrons)
            stats[pid] = p_stats
        return stats

    def get_pipeline_summary(self):
        """Compact summary for logging."""
        parts = []
        for pid, pipeline in self.pipelines.items():
            total_p = sum(pool.get_perceptron_count(self.perceptrons) for pool in pipeline.pools)
            total_auth = pipeline.get_total_authority()
            if total_p > 0 or total_auth > 0:
                parts.append(f"{pid}:{total_p}p({total_auth:.0%})")
        return ' | '.join(parts) if parts else 'all empty'

    # =========================================================================
    # CHAIN-SPECIFIC HELPERS
    # =========================================================================

    def get_chain_error_history(self, chain):
        if chain == "battle": return self.battle_error_history
        elif chain == "party": return self.party_error_history
        elif chain == "bag": return self.bag_error_history
        else: return self.error_history

    def get_chain_spawn_threshold(self, chain, percentile=75):
        history = self.get_chain_error_history(chain)
        if len(history) >= 50:
            return max(0.001, np.percentile(history, percentile))
        return 0.0005

    def get_chain_entity_signal(self, chain, learning_state):
        chain_ents = self.entities(chain=chain)
        if not chain_ents:
            return 0.5
        return np.mean([abs(e.predict(learning_state)) * e.utility for e in chain_ents])

    # =========================================================================
    # POOL-AWARE ENTITY SPAWNING (NEW — matches AI agent)
    # =========================================================================

    def get_pool_spawn_context(self, pipeline_id, layer_index, game_state_data=None):
        """
        Determine the trigger_context for a new perceptron being spawned
        into a pipeline pool. This key is used for residual file lookup.
        """
        if game_state_data is None:
            game_state_data = {}

        if pipeline_id == "battle":
            if layer_index == 0:
                es = game_state_data.get('enemy_species', -1)
                ps = game_state_data.get('player_species', -1)
                if es > 0:
                    return f"battle_id_es{es}"
                return f"battle_id_ps{ps}"
            elif layer_index == 1:
                es = game_state_data.get('enemy_species', -1)
                ps = game_state_data.get('player_species', -1)
                return f"battle_threat_{ps}v{es}"
            else:
                return f"battle_L{layer_index}_{self.current_battle_id}"

        elif pipeline_id == "overworld":
            if layer_index == 0:
                map_id = game_state_data.get('map_id', self.current_map_id or 0)
                return f"ow_spatial_map{map_id}"
            elif layer_index == 1:
                map_id = game_state_data.get('map_id', self.current_map_id or 0)
                return f"ow_area_map{map_id}"
            else:
                map_id = game_state_data.get('map_id', self.current_map_id or 0)
                return f"ow_L{layer_index}_map{map_id}_{self.timestep}"

        elif pipeline_id == "bag":
            pocket = game_state_data.get('pocket', self.bag_data.get('pocket', -1))
            return f"bag_L{layer_index}_pk{pocket}"

        elif pipeline_id == "party":
            return f"party_L{layer_index}_{self.timestep}"

        return f"{pipeline_id}_L{layer_index}_{self.timestep}"

    def spawn_into_pipeline_pool(self, pipeline_id, layer_index, input_state,
                                  game_state_data=None, entity_type=None):
        """
        Spawn a perceptron into a specific pipeline pool layer.
        Checks residual first, creates fresh if not found.
        """
        pipeline = self.pipelines.get(pipeline_id)
        if pipeline is None:
            return None

        if layer_index < 0 or layer_index >= pipeline.num_layers:
            return None

        pool = pipeline.pools[layer_index]

        # Check capacity
        n_current = pool.get_perceptron_count(self.perceptrons)
        if n_current >= pool.max_perceptrons:
            self.cluster_pipeline_pool(pipeline_id, layer_index)
            n_current = pool.get_perceptron_count(self.perceptrons)
            if n_current >= pool.max_perceptrons:
                pool.max_perceptrons = int(pool.max_perceptrons * 1.5)
                print(f"  🧩 [{pipeline_id}.{pool.name}] Pool capacity grown to {pool.max_perceptrons}")

        # Determine trigger context
        trigger_context = self.get_pool_spawn_context(pipeline_id, layer_index, game_state_data)

        # Build entity type name
        if entity_type is None:
            entity_type = f"{pipeline_id}_{pool.name}_{pool.spawn_count}"

        # Create perceptron
        p = Perceptron("entity", entity_type=entity_type, chain=pipeline_id)
        p.trigger_context = trigger_context

        # Initialize weights from input state
        p.ensure_weights(len(input_state))
        state_norm = np.linalg.norm(input_state)
        if state_norm > 0:
            p.weights = (input_state / state_norm) * 0.1
        else:
            p.weights = np.random.randn(len(input_state)) * 0.001
        p.utility = 1.0

        # Spawn via pipeline (checks residual first)
        result = pipeline.spawn_into_pool(layer_index, p, self)

        return result

    def cluster_pipeline_pool(self, pipeline_id, layer_index):
        """
        Cluster perceptrons within a specific pipeline pool.
        Pruned perceptrons are paged to the pool's residual store.
        """
        pipeline = self.pipelines.get(pipeline_id)
        if pipeline is None:
            return

        pool = pipeline.pools[layer_index]
        pool_perceptrons = [p for p in self.perceptrons if p.pool_id == pool.pool_id]

        if len(pool_perceptrons) < 2:
            return

        clusterable = [p for p in pool_perceptrons
                       if len(p.cluster_activations) >= self.ENTITY_MIN_ACTIVATIONS]
        too_young = [p for p in pool_perceptrons
                     if len(p.cluster_activations) < self.ENTITY_MIN_ACTIVATIONS]

        if len(clusterable) < 2:
            return

        max_len = max(len(p.cluster_activations) for p in clusterable)
        activation_vecs = []
        for p in clusterable:
            vec = list(p.cluster_activations)
            while len(vec) < max_len:
                vec.append(0.0)
            activation_vecs.append(np.array(vec))

        merged_indices = set()
        merge_groups = []

        for i in range(len(clusterable)):
            if i in merged_indices:
                continue
            group = [i]
            vec_i = activation_vecs[i]
            norm_i = np.linalg.norm(vec_i)
            if norm_i < 1e-10:
                continue

            for j in range(i + 1, len(clusterable)):
                if j in merged_indices:
                    continue
                vec_j = activation_vecs[j]
                norm_j = np.linalg.norm(vec_j)
                if norm_j < 1e-10:
                    continue
                cosine_sim = np.dot(vec_i, vec_j) / (norm_i * norm_j)
                if cosine_sim >= self.ENTITY_CLUSTER_SIMILARITY:
                    group.append(j)
                    merged_indices.add(j)

            if len(group) > 1:
                merged_indices.add(i)
                merge_groups.append(group)

        if not merge_groups:
            return

        new_perceptrons = []
        paged_set = set()

        for group in merge_groups:
            group_ents = [clusterable[idx] for idx in group]
            min_dim = min(len(p.weights) for p in group_ents if p.weights is not None)
            if min_dim == 0:
                continue

            avg_w = np.zeros(min_dim)
            for p in group_ents:
                avg_w += p.weights[:min_dim]
            avg_w /= len(group_ents)

            merged = Perceptron("entity",
                               entity_type=f"{pipeline_id}_{pool.name}_merged_{pool.spawn_count}",
                               chain=pipeline_id)
            merged.weights = avg_w
            merged.utility = max(p.utility for p in group_ents)
            merged.familiarity = np.mean([p.familiarity for p in group_ents])
            merged.learning_rate = np.mean([p.learning_rate for p in group_ents])
            best_ent = max(group_ents, key=lambda p: p.activation_fit_score)
            merged.active_activation = best_ent.active_activation
            merged.activation_fit_score = best_ent.activation_fit_score
            merged.pool_id = pool.pool_id
            merged.layer_index = layer_index
            merged.trigger_context = best_ent.trigger_context
            pool.spawn_count += 1

            new_perceptrons.append(merged)

            for idx in group:
                p = clusterable[idx]
                pool.page_to_residual(p)
                paged_set.add(id(p))

        kept = [p for p in clusterable if id(p) not in paged_set]
        other = [p for p in self.perceptrons if p.pool_id != pool.pool_id]
        self.perceptrons = other + kept + too_young + new_perceptrons
        self._cache_valid = False

        total_merged = sum(len(g) for g in merge_groups)
        print(f"  🧩 [{pipeline_id}.{pool.name}] POOL CLUSTERED: "
              f"{total_merged} → {len(new_perceptrons)} | "
              f"residual: {len(pool.residual)}")

    # =========================================================================
    # RESIDUAL FILE PERSISTENCE (NEW — matches AI agent)
    # =========================================================================

    def save_residual_file(self, filepath=None):
        """Save all pipeline pool residual stores to disk."""
        filepath = filepath or self.RESIDUAL_FILE
        try:
            data = {}
            for pid, pipeline in self.pipelines.items():
                pipeline_residuals = {}
                for i, pool in enumerate(pipeline.pools):
                    if pool.residual:
                        pipeline_residuals[pool.pool_id] = pool.residual
                if pipeline_residuals:
                    data[pid] = pipeline_residuals

            if data:
                with open(filepath, 'w') as f:
                    json.dump(data, f, indent=2)

        except Exception as e:
            print(f"  ⚠️ Error saving residual file: {e}")

    def load_residual_file(self, filepath=None):
        """Load residual stores from disk into pipeline pools."""
        filepath = filepath or self.RESIDUAL_FILE
        try:
            if not Path(filepath).exists():
                return

            with open(filepath, 'r') as f:
                data = json.load(f)

            total_loaded = 0
            for pid, pipeline_residuals in data.items():
                pipeline = self.pipelines.get(pid)
                if pipeline is None:
                    continue
                for pool_id, residual_entries in pipeline_residuals.items():
                    for pool in pipeline.pools:
                        if pool.pool_id == pool_id:
                            pool.residual = residual_entries
                            total_loaded += len(residual_entries)
                            break

            if total_loaded > 0:
                print(f"  🔄 Residual file loaded: {total_loaded} paged perceptrons")

        except Exception as e:
            print(f"  ⚠️ Error loading residual file: {e}")

    # =========================================================================
    # PIPELINE POOL SIGNAL QUERY (NEW — matches AI agent)
    # =========================================================================

    def get_pipeline_pool_signal(self, pipeline_id, layer_index, input_state=None):
        """Get the output of a specific pipeline pool layer."""
        pipeline = self.pipelines.get(pipeline_id)
        if pipeline is None:
            return np.zeros(Pool.DEFAULT_OUTPUT_WIDTH), 0.0

        if layer_index < 0 or layer_index >= pipeline.num_layers:
            return np.zeros(Pool.DEFAULT_OUTPUT_WIDTH), 0.0

        pool = pipeline.pools[layer_index]

        if input_state is not None:
            output = pool.compute_output(input_state, self.perceptrons)
            return output, pool.authority
        else:
            return pool.get_cached_output(), pool.authority

    # =========================================================================
    # HUMAN ACTION TRACKING
    # =========================================================================

    def set_human_action(self, action_name):
        if action_name is None:
            return
        expanded = ACTION_MAP.get(action_name, action_name)
        if expanded:
            expanded = expanded.upper()
        if expanded == "START": expanded = "Start"
        elif expanded == "SELECT": expanded = "Select"

        self.human_action = expanded
        self.last_action = expanded
        self.recent_actions_buffer.append(expanded)
        self.record_action_execution(expanded)
        self.track_consecutive_action(expanded)

    def get_recent_actions_list(self, count=15):
        actions = list(self.recent_actions_buffer)
        if len(actions) > count:
            return actions[-count:]
        return actions

    # =========================================================================
    # PARTY DATA MANAGEMENT
    # =========================================================================

    def update_party_data(self, party_data):
        self.prev_party_data = {
            'count': self.party_data.get('count', 0),
            'slots': [s.copy() for s in self.party_data.get('slots', [])],
        }
        self.party_data = {
            'count': party_data.get('count', 0),
            'slots': [s.copy() for s in party_data.get('slots', [])],
        }

    def get_party_hp_ratios(self):
        return [s.get('hp',0)/s.get('max_hp',1) if s.get('max_hp',0) > 0 else 0.0
                for s in self.party_data.get('slots', [])]

    def get_party_status_flags(self):
        return [s.get('status', 0) for s in self.party_data.get('slots', [])]

    def get_party_levels(self):
        return [s.get('level', 0) for s in self.party_data.get('slots', [])]

    def get_lowest_hp_ratio(self):
        living = [r for r in self.get_party_hp_ratios() if r > 0.0]
        return min(living) if living else 0.0

    def has_status_condition_in_party(self):
        return any(s.get('hp',0) > 0 and s.get('status',0) != 0
                   for s in self.party_data.get('slots', []))

    def get_team_avg_level(self):
        """Get average level of living party members (matches AI agent)."""
        levels = [s.get('level', 0) for s in self.party_data.get('slots', [])
                  if s.get('hp', 0) > 0]
        return np.mean(levels) if levels else 0.0

    def get_party_snapshot(self):
        return {
            'hp_ratios': self.get_party_hp_ratios(),
            'status_flags': self.get_party_status_flags(),
            'count': self.party_data.get('count', 0),
            'levels': self.get_party_levels(),
        }

    def get_party_context_for_bag(self):
        return {
            'hp_ratios': self.get_party_hp_ratios(),
            'status_flags': self.get_party_status_flags(),
            'lowest_hp_ratio': self.get_lowest_hp_ratio(),
            'has_status': self.has_status_condition_in_party(),
            'party_count': self.party_data.get('count', 0),
        }

    # =========================================================================
    # MENU DATA MANAGEMENT
    # =========================================================================

    def update_menu_data(self, menu_data):
        self.prev_menu_data = self.menu_data.copy()
        self.menu_data = menu_data.copy()

    # =========================================================================
    # BAG DATA MANAGEMENT
    # =========================================================================

    def update_bag_data(self, bag_data):
        self.prev_bag_data = {
            'pocket': self.bag_data.get('pocket', -1),
            'cursor': self.bag_data.get('cursor', -1),
            'active': self.bag_data.get('active', 0),
            'items': [item.copy() for item in self.bag_data.get('items', [])],
        }
        self.bag_data = {
            'pocket': bag_data.get('pocket', -1),
            'cursor': bag_data.get('cursor', -1),
            'active': bag_data.get('active', 0),
            'items': [item.copy() for item in bag_data.get('items', [])],
        }

    def get_item_at_cursor(self):
        items = self.bag_data.get('items', [])
        cursor = self.bag_data.get('cursor', -1)
        if 0 <= cursor < len(items):
            return items[cursor].get('id', -1)
        return -1

    # =========================================================================
    # BATTLE DATA MANAGEMENT
    # =========================================================================

    def update_battle_data(self, battle_data, in_battle):
        self.prev_battle_data = self.battle_data.copy()
        for k in ('player_stat_stages', 'enemy_stat_stages'):
            v = self.battle_data.get(k)
            if isinstance(v, list): self.prev_battle_data[k] = list(v)

        self.battle_data = battle_data.copy()
        for k in ('player_stat_stages', 'enemy_stat_stages'):
            v = battle_data.get(k)
            if isinstance(v, list): self.battle_data[k] = list(v)

        currently_in_battle = in_battle > 0.5
        if currently_in_battle and not self.in_battle_last_frame:
            self.current_battle_id += 1
        if currently_in_battle:
            self.battle_data_buffer.append({
                'timestep': self.timestep,
                'battle_id': self.current_battle_id,
                'battle_data': battle_data.copy()
            })
        self.in_battle_last_frame = currently_in_battle

    def get_battle_data_for_timestep(self, timestep):
        for entry in reversed(self.battle_data_buffer):
            if entry['timestep'] == timestep: return entry['battle_data']
            if entry['timestep'] < timestep: break
        return DEFAULT_BATTLE_DATA.copy()

    def get_all_battle_data_buffer(self):
        return self.battle_data_buffer

    # =========================================================================
    # MAP BATTLE STATISTICS (matches AI agent Phase 4)
    # =========================================================================

    def get_map_battle_stats(self, map_id):
        if map_id not in self.map_battle_stats:
            self.map_battle_stats[map_id] = {
                'battles_fought': 0, 'wild_battles': 0, 'trainer_battles': 0, 'losses': 0,
                'total_hp_cost': 0.0, 'avg_hp_cost': 0.0,
                'total_enemy_levels': 0, 'avg_enemy_level': 0.0,
                'species_seen': [], 'total_steps_on_map': 0,
                'encounter_rate': 0.0, 'last_updated': 0,
            }
        return self.map_battle_stats[map_id]

    def increment_map_steps(self, map_id):
        self.map_step_counters[map_id] = self.map_step_counters.get(map_id, 0) + 1

    def get_map_battle_stats_for_save(self):
        return {str(k): v for k, v in self.map_battle_stats.items()}

    # =========================================================================
    # TEXT FLAG MANAGEMENT
    # =========================================================================

    def update_text_flag(self, text_flag):
        self.prev_text_flag = self.text_flag
        self.text_flag = int(text_flag)

    def is_text_active(self):
        return self.text_flag == 1

    # =========================================================================
    # BAG SESSION RECORDING (UNCHANGED)
    # =========================================================================

    def start_bag_session(self, context):
        self.bag_session_active = True
        self.bag_session_context = context
        self.bag_session_start_timestep = self.timestep
        self.bag_session_frame_count = 0
        self.bag_session_last_action = None
        self.bag_session_items_used = []
        self.bag_session_pockets_visited = set()
        self.bag_session_frames = []
        pocket = self.bag_data.get('pocket', -1)
        if pocket >= 0: self.bag_session_pockets_visited.add(pocket)
        print(f"  🎒 BAG SESSION START: {context} at step {self.timestep}")

    def record_bag_frame(self, action):
        if not self.bag_session_active or action is None:
            return

        if action != self.bag_session_last_action:
            batch_type = "action_change"
            frame_offset = 0
        else:
            batch_type = "steady"
            frame_offset = 0
            for f in reversed(self.bag_session_frames):
                if f.get('batch_type') == 'steady': frame_offset += 1
                else: break

        pocket = self.bag_data.get('pocket', -1)
        if pocket >= 0: self.bag_session_pockets_visited.add(pocket)

        prev_mc = self.prev_menu_data.get('mc', -1)
        if self.bag_session_last_action == "A" and prev_mc == 0:
            prev_items = self.prev_bag_data.get('items', [])
            prev_cursor = self.prev_bag_data.get('cursor', -1)
            if 0 <= prev_cursor < len(prev_items):
                item_id = prev_items[prev_cursor].get('id', -1)
                if item_id > 0:
                    self.bag_session_items_used.append(item_id)
                    self.bag_sessions_metadata['items_used'].add(item_id)

        frame = {
            'action': action,
            'recent_actions': self.get_recent_actions_list(15),
            'state': {
                'gs': self.game_state_raw,
                'pocket': pocket,
                'cursor': self.bag_data.get('cursor', -1),
                'item_id': self.get_item_at_cursor(),
                'mc': self.menu_data.get('mc', -1),
                'mm': self.menu_data.get('mm', -1),
                'in_battle': self.bag_session_context == "battle",
            },
            'party_context': self.get_party_context_for_bag(),
            'batch_type': batch_type,
            'frame_offset': frame_offset,
        }

        self.bag_session_frames.append(frame)
        self.bag_session_frame_count += 1
        self.bag_session_last_action = action

    def end_bag_session(self):
        if not self.bag_session_active: return
        n_frames = len(self.bag_session_frames)
        duration = self.timestep - self.bag_session_start_timestep

        if n_frames > 0:
            self.bag_sessions_accumulated.extend(self.bag_session_frames)
            self.bag_sessions_metadata['total_bag_frames'] += n_frames
            self.bag_sessions_metadata['bag_sessions_recorded'] += 1
            self.bag_sessions_metadata['pockets_visited'].update(self.bag_session_pockets_visited)
            items_str = f" items={self.bag_session_items_used}" if self.bag_session_items_used else ""
            print(f"  🎒 BAG SESSION END: {self.bag_session_context} | {n_frames}f {duration}ts{items_str}")
        else:
            print(f"  🎒 BAG SESSION END: {self.bag_session_context} | empty")

        self.bag_session_active = False
        self.bag_session_context = "none"
        self.bag_session_start_timestep = 0
        self.bag_session_frame_count = 0
        self.bag_session_last_action = None
        self.bag_session_items_used = []
        self.bag_session_pockets_visited = set()
        self.bag_session_frames = []

    def detect_bag_session_boundaries(self):
        gs = self.game_state_raw
        prev_gs = self.prev_game_state_raw
        in_battle = self.battle_data.get('battle_cursor', -1) != -1
        bg_active = self.bag_data.get('active', 0)
        prev_bg_active = self.prev_bag_data.get('active', 0)

        if self.bag_session_active:
            if not in_battle and gs != 14: return "closed"
            if in_battle and bg_active == 0 and prev_bg_active == 0: return "closed"
            if in_battle and not (self.battle_data.get('battle_cursor', -1) != -1): return "closed"
            return None
        else:
            if not in_battle and gs == 14 and prev_gs != 14: return "opened_overworld"
            if in_battle and bg_active == 1 and prev_bg_active != 1: return "opened_battle"
            return None

    def get_accumulated_bag_data(self):
        return {
            'bag_frames': self.bag_sessions_accumulated,
            'metadata': {
                'total_bag_frames': self.bag_sessions_metadata['total_bag_frames'],
                'bag_sessions_recorded': self.bag_sessions_metadata['bag_sessions_recorded'],
                'items_used': sorted(list(self.bag_sessions_metadata['items_used'])),
                'pockets_visited': sorted(list(self.bag_sessions_metadata['pockets_visited'])),
            }
        }

    def get_bag_recording_stats(self):
        return {
            'session_active': self.bag_session_active,
            'session_context': self.bag_session_context,
            'session_frames': self.bag_session_frame_count,
            'total_sessions': self.bag_sessions_metadata['bag_sessions_recorded'],
            'total_frames': self.bag_sessions_metadata['total_bag_frames'],
            'items_used': sorted(list(self.bag_sessions_metadata['items_used'])),
            'pockets_visited': sorted(list(self.bag_sessions_metadata['pockets_visited'])),
        }

    # =========================================================================
    # START MENU SESSION RECORDING (UNCHANGED)
    # =========================================================================

    def start_start_menu_session(self):
        self.start_menu_session_active = True
        self.start_menu_session_start_timestep = self.timestep
        self.start_menu_session_frame_count = 0
        self.start_menu_session_last_action = None
        self.start_menu_session_frames = []
        self.start_menu_session_last_mc = -1
        print(f"  📋 START MENU SESSION START at step {self.timestep}")

    def record_start_menu_frame(self, action):
        if not self.start_menu_session_active or action is None:
            return

        if action != self.start_menu_session_last_action:
            batch_type = "action_change"
            frame_offset = 0
        else:
            batch_type = "steady"
            frame_offset = 0
            for f in reversed(self.start_menu_session_frames):
                if f.get('batch_type') == 'steady': frame_offset += 1
                else: break

        mc = self.menu_data.get('mc', -1)
        mm = self.menu_data.get('mm', -1)
        pc = self.menu_data.get('pc', -1)
        sc = self.menu_data.get('sc', -1)

        if action == "A" and 0 <= mc <= 6:
            self.start_menu_session_last_mc = mc

        frame = {
            'action': action,
            'recent_actions': self.get_recent_actions_list(15),
            'state': {
                'gs': self.game_state_raw,
                'mc': mc, 'mm': mm, 'pc': pc, 'sc': sc,
            },
            'context': {
                'target': 'unknown',
                'target_mc': -1,
                'party_hp_lowest': self.get_lowest_hp_ratio(),
                'has_status': self.has_status_condition_in_party(),
                'in_battle': False,
                'map_id': self.current_map_id if self.current_map_id is not None else 0,
            },
            'batch_type': batch_type,
            'frame_offset': frame_offset,
        }

        self.start_menu_session_frames.append(frame)
        self.start_menu_session_frame_count += 1
        self.start_menu_session_last_action = action

    def end_start_menu_session(self, reason=""):
        if not self.start_menu_session_active:
            return

        n_frames = len(self.start_menu_session_frames)
        duration = self.timestep - self.start_menu_session_start_timestep

        target_name = "unknown"
        target_mc = self.start_menu_session_last_mc

        gs = self.game_state_raw
        pc = self.menu_data.get('pc', -1)

        if reason == "bag_opened" or gs == 14:
            target_name = "bag"; target_mc = 2
        elif reason == "party_took_over" or (0 <= pc <= 5):
            target_name = "pokemon"; target_mc = 1
        elif reason == "gs_left_1" or gs == 0:
            if self.start_menu_session_last_mc >= 0:
                target_name = START_MENU_MC_NAMES.get(self.start_menu_session_last_mc, "unknown")
                target_mc = self.start_menu_session_last_mc
            else:
                target_name = "exit"; target_mc = 6
        elif reason == "battle_started":
            target_name = "interrupted"; target_mc = -1
        else:
            if self.start_menu_session_last_mc >= 0:
                target_name = START_MENU_MC_NAMES.get(self.start_menu_session_last_mc, "unknown")
                target_mc = self.start_menu_session_last_mc

        for frame in self.start_menu_session_frames:
            frame['context']['target'] = target_name
            frame['context']['target_mc'] = target_mc

        if n_frames > 0 and target_name != "interrupted":
            self.start_menu_sessions_accumulated.extend(self.start_menu_session_frames)
            self.start_menu_sessions_metadata['total_start_menu_frames'] += n_frames
            self.start_menu_sessions_metadata['start_menu_sessions_recorded'] += 1
            if target_name != "unknown":
                tc = self.start_menu_sessions_metadata['targets_navigated']
                tc[target_name] = tc.get(target_name, 0) + 1
            print(f"  📋 START MENU SESSION END: {reason} | {n_frames}f {duration}ts → {target_name}(mc={target_mc})")
        else:
            reason_str = reason or "empty"
            if target_name == "interrupted": reason_str = "battle interrupted"
            print(f"  📋 START MENU SESSION END: {reason_str} | discarded ({n_frames}f)")

        self.start_menu_session_active = False
        self.start_menu_session_start_timestep = 0
        self.start_menu_session_frame_count = 0
        self.start_menu_session_last_action = None
        self.start_menu_session_frames = []
        self.start_menu_session_last_mc = -1

    def detect_start_menu_session_boundaries(self):
        gs = self.game_state_raw
        prev_gs = self.prev_game_state_raw
        in_battle = self.battle_data.get('battle_cursor', -1) != -1
        mc = self.menu_data.get('mc', -1)
        mm = self.menu_data.get('mm', -1)
        pc = self.menu_data.get('pc', -1)

        if self.start_menu_session_active:
            if in_battle: return "closed_battle"
            if gs != 1:
                if gs == 14: return "closed_bag"
                else: return "closed_exit"
            if 0 <= pc <= 5: return "closed_party"
            return None
        else:
            if in_battle: return None
            if self.bag_session_active: return None
            if gs == 1 and prev_gs != 1:
                if 0 <= mc <= 6 and mm >= 4:
                    if not (0 <= pc <= 5):
                        return "opened"
            return None

    def get_accumulated_start_menu_data(self):
        n_frames = self.start_menu_sessions_metadata['total_start_menu_frames']
        n_sessions = self.start_menu_sessions_metadata['start_menu_sessions_recorded']
        avg_len = n_frames / max(1, n_sessions)
        return {
            'start_menu_frames': self.start_menu_sessions_accumulated,
            'metadata': {
                'total_frames': n_frames,
                'sessions_recorded': n_sessions,
                'targets_navigated': dict(self.start_menu_sessions_metadata['targets_navigated']),
                'avg_session_length': round(avg_len, 1),
            }
        }

    def get_start_menu_recording_stats(self):
        return {
            'session_active': self.start_menu_session_active,
            'session_frames': self.start_menu_session_frame_count,
            'total_sessions': self.start_menu_sessions_metadata['start_menu_sessions_recorded'],
            'total_frames': self.start_menu_sessions_metadata['total_start_menu_frames'],
            'targets_navigated': dict(self.start_menu_sessions_metadata['targets_navigated']),
        }

    # =========================================================================
    # CORE HELPERS
    # =========================================================================

    def get_location_key(self, x, y, mid, bs=5): return (int(mid), int(x//bs)*bs, int(y//bs)*bs)
    def is_near_map_edge(self, x, y): return x < 10 or x > 245 or y < 10 or y > 245
    def record_action_execution(self, a):
        if a: self.action_execution_count[a] = self.action_execution_count.get(a, 0) + 1
    def get_position_stagnation(self):
        if len(self.last_positions) < 2: return 0
        cp = self.last_positions[-1]
        return sum(1 for p in reversed(list(self.last_positions)[:-1]) if p == cp)
    def get_group_weight(self, g): return sum(a.utility for a in self.actions() if a.group == g)
    def log_state(self, ls, cs): self.prev_learning_states.append(ls); self.prev_context_states.append(cs)
    def update_position(self, x, y): self.last_positions.append((int(x), int(y)))

# ============================================================================
# CELL 3 PART 2: Exploration Memory, Markov, Blend, Transitions, Stagnation
# ============================================================================
# SYNC TO AI AGENT v17.2:
# 1. cleanup_memory() trims battle buffer + bag buffer + start menu buffer
# 2. get_memory_stats() includes start menu buffer size
# 3. All methods use chain-aware actions()/entities() signatures
#    (chain=None returns all, which is backward compatible)
# 4. All Markov, blend, exploration, tile interaction, transition, stagnation,
#    pattern, and mode swap methods UNCHANGED from previous
# ============================================================================

    # =========================================================================
    # MEMORY MANAGEMENT
    # =========================================================================
    def cleanup_memory(self):
        if len(self.location_memory) > self.MAX_LOCATIONS:
            sl = sorted(self.location_memory.items(), key=lambda x: x[1], reverse=True)
            self.location_memory = dict(sl[:self.MAX_LOCATIONS // 2])
            self.location_novelty = {k: v for k, v in self.location_novelty.items() if k in self.location_memory}
        if len(self.exploration_memory) > self.MAX_MAPS_IN_MEMORY:
            self.save_exploration_memory()
            sm = sorted(self.exploration_memory.items(), key=lambda x: x[1].get('last_visited_timestep', 0), reverse=True)
            self.exploration_memory = dict(sm[:self.MAX_MAPS_IN_MEMORY // 2])
        if len(self.battle_data_buffer) > 50000:
            self.battle_data_buffer = self.battle_data_buffer[-25000:]
        if len(self.bag_sessions_accumulated) > 100000:
            self.bag_sessions_accumulated = self.bag_sessions_accumulated[-50000:]
        # NEW v17.2: trim start menu buffer
        if len(self.start_menu_sessions_accumulated) > 50000:
            self.start_menu_sessions_accumulated = self.start_menu_sessions_accumulated[-25000:]
        self._entity_norms_cache.clear(); self._cache_valid = False; gc.collect()

    def get_memory_stats(self):
        return {'exploration_maps': len(self.exploration_memory), 'location_memory': len(self.location_memory),
                'error_history': len(self.error_history), 'perceptrons': len(self.perceptrons),
                'total_tiles': sum(len(m.get('visited_tiles', set())) for m in self.exploration_memory.values()),
                'battle_buffer_size': len(self.battle_data_buffer),
                'bag_accumulated_frames': len(self.bag_sessions_accumulated),
                'start_menu_accumulated_frames': len(self.start_menu_sessions_accumulated)}

    # =========================================================================
    # TAUGHT REFERENCE + BLEND
    # =========================================================================
    def load_taught_reference(self, fp):
        try:
            if not Path(fp).exists(): print(f"  No taught reference at {fp}"); return
            with open(fp, 'r') as f: model = json.load(f)
            if "perceptrons" not in model: return
            for sa in model["perceptrons"].get("actions", []):
                an = sa.get("action")
                if an:
                    self.taught_reference['utilities'][an] = sa.get("utility", 1.0)
                    if sa.get("weights_nonzero"):
                        dim = sa.get("weights_shape", 1376); w = np.zeros(dim)
                        for idx, val in sa["weights_nonzero"]:
                            if idx < dim: w[idx] = val
                        self.taught_reference['weights'][an] = w
            self.taught_reference['loaded'] = True
            print(f"  📖 Taught reference loaded: {list(self.taught_reference['utilities'].keys())}")
        except Exception as e: print(f"  ⚠️ Error loading taught reference: {e}")

    def blend_from_taught(self, tier):
        if not self.taught_reference['loaded'] or tier not in self.BLEND_RATIOS: return
        if self.timestep - self.last_blend_timestep < self.BLEND_COOLDOWN: return
        ai_w, tw = self.BLEND_RATIOS[tier]; bw = (tier == 3)
        for a in self.actions():
            if a.action not in self.taught_reference['utilities']: continue
            tu = self.taught_reference['utilities'][a.action]; a.utility = ai_w * a.utility + tw * tu
            if tu > 1.0: a.utility = max(a.utility, tu * 0.5)
            fl = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
            a.utility = max(min(a.utility, 2.0), fl)
            if bw and a.action in self.taught_reference['weights'] and a.weights is not None:
                taw = self.taught_reference['weights'][a.action]; md = min(len(a.weights), len(taw))
                a.weights[:md] = ai_w * a.weights[:md] + tw * taw[:md]
        self.last_blend_timestep = self.timestep; self.blend_tier = tier; self.blend_count += 1

    def get_blend_tier(self):
        t3 = self.BLEND_TIER_TRIGGERS[3]
        if self.detected_pattern and self.pattern_repeat_count >= t3['pattern_repeats']: return 3
        if self.state_stagnation_count >= self.STATE_STAGNATION_THRESHOLD * t3['state_stagnation_mult']: return 3
        t2 = self.BLEND_TIER_TRIGGERS[2]
        if self.detected_pattern and self.pattern_repeat_count >= t2['pattern_repeats']: return 2
        if self.get_position_stagnation() >= t2['pos_stagnation']: return 2
        if self.consecutive_action_count >= t2['consecutive']: return 2
        t1 = self.BLEND_TIER_TRIGGERS[1]
        if self.detected_pattern and self.pattern_repeat_count >= t1['pattern_repeats']: return 1
        if self.get_position_stagnation() >= t1['pos_stagnation']: return 1
        if self.consecutive_action_count >= t1['consecutive']: return 1
        return 0

    def try_blend_if_needed(self):
        if not self.taught_reference['loaded']: return False
        tier = self.get_blend_tier()
        if tier == 0: return False
        if tier <= self.blend_tier and (self.timestep - self.last_blend_timestep) < self.BLEND_COOLDOWN: return False
        self.blend_from_taught(tier); return True

    # =========================================================================
    # MARKOV
    # =========================================================================
    def load_taught_transitions(self, fp=None):
        fp = fp or TAUGHT_TRANSITIONS_FILE
        try:
            if Path(fp).exists():
                with open(fp, 'r') as f: data = json.load(f)
                self.taught_transitions = []; self.taught_batches = data.get('batches', [])
                for batch in self.taught_batches:
                    bt = batch.get('batch_type', 'steady'); ta = batch.get('trigger_action')
                    for frame in batch.get('frames', []):
                        self.taught_transitions.append({'state': frame.get('state', {}), 'action': frame.get('action'),
                            'recent_actions': frame.get('recent_actions', []), 'frame_offset': frame.get('frame_offset', 0),
                            'batch_type': bt, 'trigger_action': ta})
                self.taught_metadata = data.get('metadata', {})
                print(f"  📚 Taught transitions: {len(self.taught_batches)} batches, {len(self.taught_transitions)} frames")
            else: self.taught_transitions = []; self.taught_batches = []; self.taught_metadata = {}
        except Exception as e:
            print(f"  Error loading transitions: {e}")
            self.taught_transitions = []; self.taught_batches = []; self.taught_metadata = {}

    def extract_partial_context(self, cs, rp=None):
        rx = rp[0] if rp else int(cs[0]*255); ry = rp[1] if rp else int(cs[1]*255); cm = int(cs[2])
        mb = self.get_position_stagnation() > 3
        nt = False; mem = self.get_current_map_memory(cm)
        for t in mem.get('transitions', []):
            tp = tuple(t['position']) if isinstance(t['position'], list) else t['position']
            if abs(rx - tp[0]) + abs(ry - tp[1]) <= 2: nt = True; break
        return {'in_battle': cs[3] > 0.5, 'in_menu': cs[4] > 0.5, 'movement_blocked': mb,
                'near_transition': nt, 'tile_probed': not self.should_interact_at_tile(rx, ry, cm)}

    def compute_markov_similarity(self, cs, rp=None, taught_frames=None):
        frames = taught_frames if taught_frames is not None else self.taught_transitions
        smc = taught_frames is not None
        if not frames: return 0.0, None, -1
        rx = rp[0] if rp else int(cs[0]*255); ry = rp[1] if rp else int(cs[1]*255)
        cm = int(cs[2]); cd = int(cs[5]); ib = cs[3] > 0.5; im = cs[4] > 0.5
        ca = list(self.action_history); cp = self.extract_partial_context(cs, rp)
        bs, ba, bi = 0.0, None, -1
        for idx, tr in enumerate(frames):
            ts = tr.get('state', {}); ta = tr.get('action'); trc = tr.get('recent_actions', []); bt = tr.get('batch_type', 'steady')
            if not ta or ta == "NONE": continue
            isc = 0.0
            if not smc:
                if ts.get('map_id') != cm: continue
            isc += 0.25
            tx, ty = ts.get('x', 0), ts.get('y', 0); pd = abs(rx-tx) + abs(ry-ty)
            if pd == 0: isc += MARKOV_POS_EXACT_BONUS
            elif pd <= 2: isc += MARKOV_POS_NEAR_BONUS
            elif pd <= MARKOV_POS_MAX_DIST: isc += MARKOV_POS_FAR_BONUS
            else: continue
            if ts.get('direction') == cd: isc += 0.2
            tib = ts.get('in_battle', 0) == 1; tim = ts.get('in_menu', 0) == 1
            if tib == ib: isc += 0.1
            if tim == im: isc += 0.1
            ssc = 0.0
            if trc and ca:
                if len(ca) >= 8 and len(trc) >= 8 and list(ca)[-8:] == trc[-8:]: ssc = MARKOV_SEQ_FULL_WEIGHT
                if ssc < MARKOV_SEQ_MEDIUM_WEIGHT and len(ca) >= 5 and len(trc) >= 5 and list(ca)[-5:] == trc[-5:]: ssc = MARKOV_SEQ_MEDIUM_WEIGHT
                if ssc < MARKOV_SEQ_SHORT_WEIGHT and len(ca) >= 3 and len(trc) >= 3 and list(ca)[-3:] == trc[-3:]: ssc = MARKOV_SEQ_SHORT_WEIGHT
            pm = sum(1 for a, b in [(tib, cp['in_battle']), (tim, cp['in_menu'])] if a == b)
            total = MARKOV_IMMEDIATE_WEIGHT * isc + MARKOV_SEQUENTIAL_WEIGHT * ssc + MARKOV_PARTIAL_WEIGHT * (pm / 2)
            if bt == "action_change": total *= 1.2
            if tr.get('frame_offset', 0) == 0: total *= 1.1
            if total > bs: bs, ba, bi = total, ta, idx
        return bs, ba, bi

    def get_markov_action(self, cs, rp=None, taught_frames=None):
        if not self.markov_enabled: return False, None, 0.0
        frames = taught_frames if taught_frames is not None else self.taught_transitions
        if not frames: return False, None, 0.0
        sc, ac, ix = self.compute_markov_similarity(cs, rp, taught_frames=frames)
        self.last_markov_score = sc
        if sc >= MARKOV_FAMILIARITY_THRESHOLD: self.last_markov_action = ac; return True, ac, sc
        return False, None, sc

    # =========================================================================
    # ACTION EXECUTION CONFIRMATION
    # =========================================================================
    def set_pending_action(self, a): self.pending_action = a; self.pending_action_frames = 0
    def confirm_action_executed(self, cs, pcs):
        if self.pending_action is None: return True
        self.pending_action_frames += 1; ae = False
        if pcs is not None:
            if self.pending_action in ["UP","DOWN","LEFT","RIGHT"]:
                ae = cs[0] != pcs[0] or cs[1] != pcs[1] or cs[5] != pcs[5]
            elif self.pending_action in ["A","B","Start","Select"]:
                ae = abs(cs[4]-pcs[4]) > 0.1 or cs[3] != pcs[3] or cs[2] != pcs[2]
        if ae or self.pending_action_frames >= self.ACTION_CONFIRM_FRAMES:
            self.last_confirmed_action = self.pending_action; self.pending_action = None; self.pending_action_frames = 0; return True
        return False
    def should_send_new_action(self):
        return self.pending_action is None or self.pending_action_frames >= self.ACTION_CONFIRM_FRAMES

    # =========================================================================
    # EXPLORATION MEMORY
    # =========================================================================
    def load_exploration_memory(self):
        try:
            if self.EXPLORATION_MEMORY_FILE.exists():
                with open(self.EXPLORATION_MEMORY_FILE, 'r') as f: data = json.load(f)
                self.exploration_memory = {}
                for mk, md in data.items():
                    self.exploration_memory[int(mk.replace('map_', ''))] = self._deserialize_map_memory(md)
                print(f"  Loaded exploration: {len(self.exploration_memory)} maps")
            else: self.exploration_memory = {}
        except Exception as e: print(f"  Error loading exploration: {e}"); self.exploration_memory = {}

    def _deserialize_map_memory(self, d):
        ti = {}
        for tk, td in d.get('tile_interactions', {}).items():
            ti[tk] = {'directions_tried': set(td.get('directions_tried', [])),
                      'direction_attempts': {int(k): v for k, v in td.get('direction_attempts', {}).items()},
                      'direction_successes': {int(k): v for k, v in td.get('direction_successes', {}).items()},
                      'exhausted': td.get('exhausted', False)}
        return {'visited_tiles': set(tuple(t) for t in d.get('visited_tiles', [])),
                'obstructions': set(tuple(t) for t in d.get('obstructions', [])),
                'interactable_objects': d.get('interactable_objects', []),
                'last_visited_timestep': d.get('last_visited_timestep', 0),
                'transitions': d.get('transitions', []), 'temp_debt': d.get('temp_debt', 0.0),
                'tile_interactions': ti}

    def save_exploration_memory(self):
        try:
            data = {f'map_{mid}': self._serialize_map_memory(md) for mid, md in self.exploration_memory.items()}
            with open(self.EXPLORATION_MEMORY_FILE, 'w') as f: json.dump(data, f)
        except Exception as e: print(f"  Error saving exploration: {e}")

    def _serialize_map_memory(self, d):
        sti = {}
        for tk, td in d.get('tile_interactions', {}).items():
            sti[tk] = {'directions_tried': list(td.get('directions_tried', set())),
                       'direction_attempts': {str(k): v for k, v in td.get('direction_attempts', {}).items()},
                       'direction_successes': {str(k): v for k, v in td.get('direction_successes', {}).items()},
                       'exhausted': td.get('exhausted', False)}
        return {'visited_tiles': [list(t) for t in d['visited_tiles']], 'obstructions': [list(t) for t in d['obstructions']],
                'interactable_objects': d['interactable_objects'], 'last_visited_timestep': d['last_visited_timestep'],
                'transitions': d.get('transitions', []), 'temp_debt': d.get('temp_debt', 0.0), 'tile_interactions': sti}

    def get_current_map_memory(self, mid):
        if mid not in self.exploration_memory:
            self.exploration_memory[mid] = {'visited_tiles': set(), 'obstructions': set(), 'interactable_objects': [],
                'last_visited_timestep': self.timestep, 'transitions': [], 'temp_debt': 0.0, 'tile_interactions': {}}
        return self.exploration_memory[mid]

    def record_visited_tile(self, x, y, mid):
        m = self.get_current_map_memory(mid); m['visited_tiles'].add((int(x), int(y))); m['last_visited_timestep'] = self.timestep
    def record_obstruction(self, x, y, mid, d):
        dx, dy = self.DIRECTION_DELTAS_INT.get(d, (0,0)); self.get_current_map_memory(mid)['obstructions'].add((int(x+dx), int(y+dy)))

    def merge_taught_exploration(self, fp):
        if not Path(fp).exists(): print(f"  No taught exploration at {fp}"); return
        try:
            with open(fp, 'r') as f: td = json.load(f)
            ta, ia = 0, 0
            for mk, md in td.items():
                mid = int(mk.replace('map_', '')); tm = self._deserialize_map_memory(md); am = self.get_current_map_memory(mid)
                am['visited_tiles'].update(tm['visited_tiles']); am['obstructions'].update(tm['obstructions'])
                for tt in tm.get('transitions', []):
                    tp = tuple(tt['position']) if isinstance(tt['position'], list) else tt['position']
                    if not any((tuple(e['position']) if isinstance(e['position'], list) else e['position']) == tp and e['direction'] == tt['direction'] for e in am['transitions']):
                        am['transitions'].append(tt); ta += 1
                for ti in tm.get('interactable_objects', []):
                    if ti not in am['interactable_objects']: am['interactable_objects'].append(ti); ia += 1
            print(f"  Merged: {ta} transitions, {ia} interactables")
        except Exception as e: print(f"  Error merging: {e}")

    # =========================================================================
    # TILE INTERACTION
    # =========================================================================
    def get_tile_interaction_key(self, x, y): return f"{int(x)}_{int(y)}"
    def get_tile_interaction_state(self, x, y, mid):
        m = self.get_current_map_memory(mid); tk = self.get_tile_interaction_key(x, y)
        if tk not in m['tile_interactions']:
            m['tile_interactions'][tk] = {'directions_tried': set(), 'direction_attempts': {0:0,1:0,2:0,3:0},
                'direction_successes': {0:0,1:0,2:0,3:0}, 'exhausted': False}
        return m['tile_interactions'][tk]
    def should_interact_at_tile(self, x, y, mid):
        ts = self.get_tile_interaction_state(x, y, mid)
        if ts['exhausted']: return False
        if len(ts['directions_tried']) < 4: return True
        return any(ts['direction_attempts'].get(d,0) > 0 and ts['direction_successes'].get(d,0)/ts['direction_attempts'][d] >= self.MIN_SUCCESS_RATE_THRESHOLD for d in range(4))
    def get_untried_directions(self, x, y, mid):
        return [d for d in range(4) if d not in self.get_tile_interaction_state(x, y, mid)['directions_tried']]
    def get_best_interaction_direction(self, x, y, mid):
        ts = self.get_tile_interaction_state(x, y, mid)
        u = self.get_untried_directions(x, y, mid)
        if u: return u[0]
        bd, br = None, 0.0
        for d in range(4):
            a = ts['direction_attempts'].get(d, 0)
            if a > 0:
                r = ts['direction_successes'].get(d, 0) / a
                if r > br: br, bd = r, d
        return bd
    def get_best_probe_action(self, rx, ry, cm, cd):
        ck = (rx, ry, cm, cd)
        if self._probe_cache_position == ck: return self._cached_probe_action, self._cached_probe_dir
        if not self.should_interact_at_tile(rx, ry, cm): r = (None, None)
        else:
            u = self.get_untried_directions(rx, ry, cm)
            if not u:
                bd = self.get_best_interaction_direction(rx, ry, cm)
                r = ('A', cd) if bd is not None and cd == bd else (self.INT_TO_ACTION[bd], bd) if bd is not None else (None, None)
            elif cd in u: r = ('A', cd)
            else: r = (self.INT_TO_ACTION[u[0]], u[0])
        self._probe_cache_position = ck; self._cached_probe_action, self._cached_probe_dir = r; return r
    def record_tile_interaction_attempt(self, x, y, mid, d, success):
        ts = self.get_tile_interaction_state(x, y, mid); ts['directions_tried'].add(d)
        ts['direction_attempts'][d] = ts['direction_attempts'].get(d, 0) + 1
        if success:
            ts['direction_successes'][d] = ts['direction_successes'].get(d, 0) + 1
            m = self.get_current_map_memory(mid); dn = self.DIRECTION_NAMES.get(d, str(d))
            io = [int(x), int(y), dn]
            if io not in m['interactable_objects']: m['interactable_objects'].append(io); print(f"  🎯 INTERACTABLE: ({x},{y}) {dn}")
        self._check_tile_exhaustion(x, y, mid)
    def _check_tile_exhaustion(self, x, y, mid):
        ts = self.get_tile_interaction_state(x, y, mid)
        if len(ts['directions_tried']) >= 4 and not any(ts['direction_successes'].get(d,0) > 0 for d in range(4)):
            ts['exhausted'] = True
    def start_interaction_verification(self, x, y, mid, d):
        self.pending_interaction_verify = {'x': x, 'y': y, 'map_id': mid, 'direction': d}
        self.interaction_verify_countdown = self.INTERACTION_VERIFY_FRAMES
    def check_interaction_verification(self, cs, pcs):
        if self.pending_interaction_verify is None: return
        self.interaction_verify_countdown -= 1; success = False
        if pcs is not None:
            in_overworld = pcs[3] <= 0.5 and pcs[4] <= 0.5
            if in_overworld:
                success = abs(cs[4]-pcs[4]) > 0.1 or (cs[3] > 0.5 and pcs[3] <= 0.5) or int(cs[2]) != int(pcs[2])
        if success or self.interaction_verify_countdown <= 0:
            i = self.pending_interaction_verify
            self.record_tile_interaction_attempt(i['x'], i['y'], i['map_id'], i['direction'], success)
            self.pending_interaction_verify = None
    def get_tile_interaction_stats(self, mid):
        m = self.get_current_map_memory(mid); ti = m.get('tile_interactions', {})
        return {'probed': len(ti), 'exhausted': sum(1 for t in ti.values() if t.get('exhausted', False)),
                'with_success': sum(1 for t in ti.values() if any(t.get('direction_successes',{}).get(d,0) > 0 for d in range(4)))}
    def get_exploration_coverage(self, mid):
        m = self.get_current_map_memory(mid); v = len(m['visited_tiles']); o = len(m['obstructions'])
        return v / (v + o) if v > 0 and v + o >= 10 else 0.0

    # =========================================================================
    # TRANSITIONS, BANS, DEBT
    # =========================================================================
    def record_transition(self, fp, fm, tm, d, at):
        m = self.get_current_map_memory(fm)
        for t in m['transitions']:
            if t['position'] == list(fp) and t['direction'] == d: t['use_count'] += 1; t['last_used'] = self.timestep; return
        m['transitions'].append({'position': list(fp), 'direction': d, 'action': at, 'destination_map': tm, 'use_count': 1, 'last_used': self.timestep})
        print(f"  🚪 TRANSITION: Map {fm} ({fp}) → Map {tm}")
    def get_transition_attraction(self, cm):
        m = self.get_current_map_memory(cm); ts = m.get('transitions', [])
        if not ts: return 0.0, None
        cd = self.map_novelty_debt.get(cm, 0.0); ctd = self.get_temp_debt(cm); cc = self.get_exploration_coverage(cm)
        ba, bt = 0.0, None
        for t in ts:
            if self.is_transition_banned(cm, t['position'], t['direction']): continue
            dm = t['destination_map']; dd = self.map_novelty_debt.get(dm, 0.0); dtd = self.get_temp_debt(dm); dc = self.get_exploration_coverage(dm)
            a = (cd + ctd*2.0 - dd - dtd*2.0)*0.5 + (cc - dc)*0.5
            if t['use_count'] < 3: a *= 1.5
            if a > ba: ba, bt = a, t
        return ba * self.TRANSITION_ATTRACTION_WEIGHT, bt
    def create_transition_ban(self, mid, tp, db):
        self.transition_bans[mid] = {'banned_tile': tp, 'banned_direction': db, 'vicinity_radius': self.BAN_VICINITY_RADIUS,
            'vicinity_active': False, 'created_at': self.timestep}
    def is_transition_banned(self, mid, pos, d):
        if mid not in self.transition_bans: return False
        b = self.transition_bans[mid]; bt = tuple(b['banned_tile']) if isinstance(b['banned_tile'], list) else b['banned_tile']
        pos = tuple(pos) if isinstance(pos, list) else pos
        if pos == bt and d == b['banned_direction']: return True
        if b['vicinity_active'] and abs(pos[0]-bt[0])+abs(pos[1]-bt[1]) <= b['vicinity_radius'] and d == b['banned_direction']: return True
        return False
    def is_position_banned(self, mid, x, y, d): return self.is_transition_banned(mid, (x,y), d)
    def update_transition_ban(self, mid, cp):
        if mid not in self.transition_bans: return
        b = self.transition_bans[mid]; bt = tuple(b['banned_tile']) if isinstance(b['banned_tile'], list) else b['banned_tile']
        if not b['vicinity_active'] and abs(cp[0]-bt[0])+abs(cp[1]-bt[1]) >= 3: b['vicinity_active'] = True
    def check_ban_lift_conditions(self, mid):
        if mid not in self.transition_bans: return
        b = self.transition_bans[mid]; m = self.get_current_map_memory(mid)
        nb = [t for t in m.get('transitions',[]) if not self.is_transition_banned(mid, t['position'], t['direction'])]
        if nb or self.get_exploration_coverage(mid) >= self.BAN_COVERAGE_LIFT_THRESHOLD or self.timestep - b['created_at'] >= self.BAN_TIMEOUT_STEPS:
            del self.transition_bans[mid]
    def get_temp_debt(self, mid):
        m = self.get_current_map_memory(mid); rd = m.get('temp_debt', 0.0)
        if mid != self.current_map_id:
            return max(0.0, rd - (self.timestep - m.get('last_visited_timestep', 0)) * self.TEMP_DEBT_DECAY)
        return rd
    def accumulate_temp_debt(self, mid):
        m = self.get_current_map_memory(mid); m['temp_debt'] = min(self.TEMP_DEBT_MAX, m.get('temp_debt', 0.0) + self.TEMP_DEBT_ACCUMULATION)
    def decay_all_debts(self):
        for mid in list(self.map_novelty_debt.keys()):
            if mid != self.current_map_id:
                self.map_novelty_debt[mid] *= (1.0 - self.DEBT_DECAY_RATE)
                if self.map_novelty_debt[mid] < 0.1: del self.map_novelty_debt[mid]
    def detect_obstruction(self, pc, cs, rp, prp):
        if pc is None or prp is None or self.last_action not in ['UP','DOWN','LEFT','RIGHT']: return False
        if rp == prp: self.record_obstruction(rp[0], rp[1], int(cs[2]), int(cs[5])); return True
        return False

    # =========================================================================
    # MENU TRAP
    # =========================================================================
    def update_menu_trap_tracking(self, cs, at, rp=None):
        cp = rp if rp else (round(cs[0]*255), round(cs[1]*255))
        if self.menu_trap_position is not None and cp != self.menu_trap_position: self.reset_menu_trap_boost(); return
        if self.get_context_state_hash(cs) == self.last_context_state_hash and at in ["A","B","Start","Select"]:
            self.menu_trap_frames += 1; self.menu_trap_position = cp
            if self.menu_trap_frames > self.MENU_TRAP_THRESHOLD:
                if self.original_b_utility is None:
                    for a in self.actions():
                        if a.action == 'B': self.original_b_utility = a.utility; break
                self.menu_trap_b_boost = min(self.B_BOOST_MAX, self.menu_trap_b_boost + self.B_BOOST_INCREMENT)
        elif cp != self.menu_trap_position: self.reset_menu_trap_boost()
    def reset_menu_trap_boost(self):
        if self.menu_trap_b_boost > 1.0 and self.original_b_utility is not None:
            for a in self.actions():
                if a.action == 'B': a.utility = self.original_b_utility; break
        self.menu_trap_frames = 0; self.menu_trap_b_boost = 1.0; self.menu_trap_position = None; self.original_b_utility = None

    # =========================================================================
    # STAGNATION, PATTERN, MODE
    # =========================================================================
    def get_context_state_hash(self, cs):
        return (round(cs[0],2), round(cs[1],2), int(cs[2]), int(cs[3]), round(cs[4],2), int(cs[5]))
    def check_state_stagnation(self, cs):
        ch = self.get_context_state_hash(cs)
        if ch == self.last_context_state_hash:
            self.state_stagnation_count += 1
            if self.state_stagnation_count == 1 and self.last_action: self.stagnation_initiator_action = self.last_action
        else: self.state_stagnation_count = 0; self.stagnation_initiator_action = None
        self.last_context_state_hash = ch
        return self.state_stagnation_count >= self.STATE_STAGNATION_THRESHOLD
    def apply_stagnation_initiator_penalty(self):
        if self.stagnation_initiator_action is None: return
        for a in self.actions():
            if a.action == self.stagnation_initiator_action:
                fl = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
                a.utility = max(fl, a.utility * 0.5); break
        self.stagnation_initiator_action = None
    def should_force_random(self):
        f = self.get_position_stagnation() >= 8 or self.consecutive_action_count >= 15 or \
            (self.detected_pattern and self.pattern_repeat_count >= 4) or \
            self.state_stagnation_count >= self.STATE_STAGNATION_THRESHOLD * 2
        if f: self.try_blend_if_needed()
        return f
    def get_forced_random_action_name(self):
        c = ["UP","DOWN","LEFT","RIGHT","A","B"]
        if self.current_repeated_action in c: c.remove(self.current_repeated_action)
        if self.detected_pattern:
            for a in self.detected_pattern:
                if a in c: c.remove(a)
        return random.choice(c or ["UP","DOWN","LEFT","RIGHT"])
    def check_productive_change(self, cs):
        cm = int(cs[2]); cb = cs[3] > 0.5; cp = (cs[0], cs[1]); p, r = False, ""
        if self.last_map_id is not None and cm != self.last_map_id: p, r = True, "map change"
        if self.last_battle_state is not None and cb != self.last_battle_state: p, r = True, "battle change"
        if self.position_at_mode_swap is not None:
            d = np.sqrt((cp[0]-self.position_at_mode_swap[0])**2 + (cp[1]-self.position_at_mode_swap[1])**2)
            if d > 0.03: p, r = True, f"moved {d*255:.1f}"
        cd = int(cs[5])
        if self.direction_change_counts_as_progress and self.last_direction_for_progress is not None and cd != self.last_direction_for_progress:
            self.state_stagnation_count = max(0, self.state_stagnation_count - 5)
        self.last_direction_for_progress = cd; self.last_map_id = cm; self.last_battle_state = cb
        return p, r
    def on_productive_change(self, r):
        self.move_to_interact_threshold = self.DEFAULT_MOVE_TO_INTERACT_THRESHOLD
        self.interact_to_move_threshold = self.DEFAULT_INTERACT_TO_MOVE_THRESHOLD
        self.swap_chain_count = 0; self.state_stagnation_count = 0; self.stagnation_initiator_action = None; self.unproductive_swap_count = 0
        if self.blend_tier > 0: self.blend_tier = 0
    def on_mode_swap(self, fm, tm):
        self.swap_chain_count += 1; self.frames_in_current_mode = 0; self.unproductive_swap_count += 1
        if self.unproductive_swap_count >= self.UNPRODUCTIVE_SWAP_THRESHOLD:
            self._reset_highest_to_third(tm); self.unproductive_swap_count = 0
        if tm == "interact": self.interact_to_move_threshold = min(self.MAX_THRESHOLD, self.interact_to_move_threshold + self.THRESHOLD_INCREMENT)
        else: self.move_to_interact_threshold = min(self.MAX_THRESHOLD, self.move_to_interact_threshold + self.THRESHOLD_INCREMENT)
    def _reset_highest_to_third(self, mode):
        if mode in ["battle","both"]: return
        g = "move" if mode == "move" else "interact"; ga = sorted([a for a in self.actions() if a.group == g], key=lambda a: a.utility, reverse=True)
        if len(ga) >= 3:
            fl = self.INTERACT_UTILITY_FLOOR if g == "interact" else self.MOVE_UTILITY_FLOOR
            ga[0].utility = max(ga[2].utility * 0.9, fl)
    def should_use_both_mode(self):
        return self.state_stagnation_count > self.BOTH_MODE_STAGNATION_THRESHOLD or self.unproductive_swap_count > self.BOTH_MODE_SWAP_THRESHOLD
    def determine_control_mode(self, cs, raw_position=None):
        if cs[3] > 0.5: return "battle"
        self.frames_in_current_mode += 1; ps = self.get_position_stagnation()
        p, r = self.check_productive_change(cs)
        if p: self.on_productive_change(r)
        if self.should_use_both_mode(): return "both"
        if self.check_state_stagnation(cs):
            self.apply_stagnation_initiator_penalty()
            nm = "interact" if self.control_mode == "move" else "move"
            self.control_mode = nm; self.position_at_mode_swap = (cs[0], cs[1])
            self.on_mode_swap(self.control_mode, nm); self.state_stagnation_count = 0; return self.control_mode
        rx = raw_position[0] if raw_position else int(cs[0]*255); ry = raw_position[1] if raw_position else int(cs[1]*255); cm = int(cs[2])
        tp = self.should_interact_at_tile(rx, ry, cm); ud = self.get_untried_directions(rx, ry, cm)
        if tp and ud and self.control_mode == "move" and self.frames_in_current_mode >= 3:
            self.control_mode = "interact"; self.position_at_mode_swap = (cs[0], cs[1]); self.frames_in_current_mode = 0; return self.control_mode
        if self.control_mode == "move" and ps >= self.move_to_interact_threshold:
            self.control_mode = "interact"; self.position_at_mode_swap = (cs[0], cs[1]); self.on_mode_swap("move", "interact")
        elif self.control_mode == "interact":
            if (not tp or not ud) and self.frames_in_current_mode >= 5:
                self.control_mode = "move"; self.position_at_mode_swap = (cs[0], cs[1]); self.frames_in_current_mode = 0
            elif self.frames_in_current_mode >= self.interact_to_move_threshold:
                self.control_mode = "move"; self.position_at_mode_swap = (cs[0], cs[1]); self.on_mode_swap("interact", "move")
        return self.control_mode

    # =========================================================================
    # REPETITION & PATTERN
    # =========================================================================
    def track_consecutive_action(self, an):
        if an == self.current_repeated_action: self.consecutive_action_count += 1
        else: self.current_repeated_action = an; self.consecutive_action_count = 1
    def get_learning_multiplier(self, an):
        if an != self.current_repeated_action or self.consecutive_action_count < self.LEARNING_SLOWDOWN_START: return 1.0
        return max(0.05, 1.0 - 0.95 * min(1.0, (self.consecutive_action_count - self.LEARNING_SLOWDOWN_START) / (self.LEARNING_SLOWDOWN_MAX - self.LEARNING_SLOWDOWN_START)))
    def detect_pattern(self):
        if len(self.action_history) < 6: return None, 0
        recent = list(self.action_history)[-self.PATTERN_CHECK_WINDOW:]
        for pl in range(1, self.PATTERN_MAX_LENGTH + 1):
            if len(recent) < pl * self.PATTERN_MIN_REPEATS: continue
            cand = tuple(recent[-pl:]); rc, ix = 0, len(recent) - pl
            while ix >= 0 and tuple(recent[ix:ix+pl]) == cand: rc += 1; ix -= pl
            if rc >= self.PATTERN_MIN_REPEATS: return cand, rc
        return None, 0
    def apply_pattern_penalty(self):
        pat, rc = self.detect_pattern()
        if pat is None: self.detected_pattern = None; self.pattern_repeat_count = 0; return
        self.detected_pattern, self.pattern_repeat_count = pat, rc
        pf = max(0.3, 1.0 - rc * 0.15)
        for an in set(pat):
            for a in self.actions():
                if a.action == an:
                    fl = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
                    a.utility = max(fl, a.utility * pf); break
    def apply_repetition_penalty(self):
        if self.current_repeated_action is None: return
        for a in self.actions():
            if a.action == self.current_repeated_action:
                fl = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
                if self.consecutive_action_count >= self.HARD_RESET_THRESHOLD:
                    a.utility = fl; self.consecutive_action_count = 0
                elif self.consecutive_action_count >= self.PENALTY_THRESHOLD:
                    a.utility = max(a.utility * 0.5, fl)
                break

    # =========================================================================
    # EXPLORATION TRACKING
    # =========================================================================
    def update_exploration_tracking(self, cs, pcs, rp=None, prp=None):
        cm = int(cs[2]); rx = rp[0] if rp else int(cs[0]*255); ry = rp[1] if rp else int(cs[1]*255); cp = (rx, ry)
        if self.current_map_id is not None and cm != self.current_map_id:
            pm = self.current_map_id
            if pcs is not None and prp is not None:
                self.record_transition(prp, pm, cm, int(pcs[5]), 'interact' if self.last_action == 'A' else 'walk')
            if prp is not None:
                ed = int(cs[5]) if pcs is not None else 0
                self.create_transition_ban(cm, cp, (ed + 2) % 4)
            self.on_map_change(cm)
        self.current_map_id = cm; self.record_visited_tile(rx, ry, cm); self.accumulate_temp_debt(cm)
        self.update_transition_ban(cm, cp); self.check_ban_lift_conditions(cm)
        if pcs is not None and prp is not None: self.detect_obstruction(pcs, cs, rp, prp)
        self.check_interaction_verification(cs, pcs); self.last_direction = int(cs[5])
        if self.timestep % 300 == 0: self.decay_all_debts()
    def on_map_change(self, nm):
        self.save_exploration_memory(); self.control_mode = "move"; self.frames_in_current_mode = 0
        m = self.get_current_map_memory(nm)
        print(f"  🗺️ MAP CHANGE → {nm}: {len(m['visited_tiles'])} visited, {len(m['obstructions'])} obs")

# ============================================================================
# CELL 3 PART 3: Chain-Specific Entity Spawning/Clustering, Learning, Save/Load
# ============================================================================
# SYNC TO AI AGENT v17.4 (Multi-Pool Pipeline):
# 1. spawn_innate_overworld_entities() — UNCHANGED
# 2. spawn_innate_battle_entities() — UNCHANGED
# 3. spawn_entity_for_chain() — UNCHANGED
# 4. check_chain_entity_capacity() — UNCHANGED
# 5. cluster_chain_entities() — UNCHANGED
# 6. NEW: learn_pipeline() — generic pipeline forward+backward+spawn
# 7. UPDATED: learn_battle_chain() — runs BOTH pipeline AND legacy chain
# 8. UPDATED: learn_party_chain() — runs BOTH pipeline AND legacy chain
# 9. UPDATED: learn_bag_chain() — runs BOTH pipeline AND legacy chain
# 10. UPDATED: learn() — runs overworld pipeline, filters pool-assigned
#     perceptrons from legacy updates
# 11. NEW: prune_pipeline_pools() — prune low-quality pool perceptrons
# 12. UPDATED: save_model_checkpoint() — includes pipeline state, revenge
#     targets, pool_state per perceptron
# 13. UPDATED: load_taught_model() — restores pipeline state, revenge
#     targets, pool_state per perceptron
# 14. Legacy wrappers UNCHANGED
# ============================================================================

    # =========================================================================
    # CHAIN-SPECIFIC ENTITY SPAWNING (matches AI agent)
    # =========================================================================

    def spawn_innate_overworld_entities(self, learning_state):
        if self.innate_entities_spawned_overworld:
            return
        for etype, indices in [("sense_menu", [5, 6]), ("sense_battle", [3, 4]),
                                ("sense_movement", [0, 1]), ("sense_map_transition", [2])]:
            entity = Perceptron("entity", entity_type=etype, chain="overworld")
            entity.ensure_weights(len(learning_state))
            entity.weights = np.zeros(len(learning_state))
            for idx in indices:
                if idx < len(learning_state):
                    entity.weights[idx] = 0.5 if len(indices) > 1 else 1.0
            self.add(entity)
        self.innate_entities_spawned_overworld = True
        self.innate_entities_spawned = True
        print(f"  🧬 Innate overworld entities spawned (4)")

    def spawn_innate_battle_entities(self, learning_state):
        if self.innate_entities_spawned_battle:
            return
        innate_battle = [
            ("battle_hp_crisis", [1]),
            ("battle_enemy_weak", [20]),
            ("battle_species_match", [0, 19]),
            ("battle_status", [3, 22]),
            ("battle_trainer", [38]),
        ]
        for etype, indices in innate_battle:
            entity = Perceptron("entity", entity_type=etype, chain="battle")
            entity.ensure_weights(len(learning_state))
            entity.weights = np.zeros(len(learning_state))
            for idx in indices:
                if idx < len(learning_state):
                    entity.weights[idx] = 0.5 if len(indices) > 1 else 1.0
            self.add(entity)
        self.innate_entities_spawned_battle = True
        print(f"  🧬 Innate battle entities spawned ({len(innate_battle)})")

    def spawn_entity_for_chain(self, chain, learning_state, context_state=None, raw_position=None):
        count = self.entity_spawn_counts.get(chain, 0)
        entity = Perceptron("entity", entity_type=f"{chain}_spawned_{count}", chain=chain)
        entity.ensure_weights(len(learning_state))

        state_norm = np.linalg.norm(learning_state)
        if state_norm > 0:
            entity.weights = (learning_state / state_norm) * 0.1
        else:
            entity.weights = np.random.randn(len(learning_state)) * 0.001

        entity.utility = 1.0
        self.add(entity)
        self.entity_spawn_counts[chain] = count + 1

        if chain == "overworld":
            self.entity_spawn_count = self.entity_spawn_counts['overworld']

        self.check_chain_entity_capacity(chain)

    def check_chain_entity_capacity(self, chain):
        n_entities = self.get_chain_entity_count(chain)
        capacity = self.get_chain_entity_capacity(chain)
        if n_entities < capacity:
            return
        before = n_entities
        self.cluster_chain_entities(chain)
        after = self.get_chain_entity_count(chain)
        if after >= before * 0.9:
            old_cap = self.ENTITY_CAPACITY[chain]
            self.ENTITY_CAPACITY[chain] = int(old_cap * self.ENTITY_CAPACITY_GROWTH)
            print(f"  🧩 [{chain}] Entity capacity: {old_cap} → {self.ENTITY_CAPACITY[chain]} "
                  f"(clustering {before} → {after})")

    def cluster_chain_entities(self, chain):
        chain_entities = self.entities(chain=chain)

        innate_types = {"sense_menu", "sense_battle", "sense_movement", "sense_map_transition",
                        "battle_hp_crisis", "battle_enemy_weak", "battle_species_match",
                        "battle_status", "battle_trainer"}
        spawned = [e for e in chain_entities if e.entity_type not in innate_types]
        innate = [e for e in chain_entities if e.entity_type in innate_types]

        if len(spawned) < 2:
            return

        clusterable = [e for e in spawned if len(e.cluster_activations) >= self.ENTITY_MIN_ACTIVATIONS]
        too_young = [e for e in spawned if len(e.cluster_activations) < self.ENTITY_MIN_ACTIVATIONS]

        if len(clusterable) < 2:
            return

        max_len = max(len(e.cluster_activations) for e in clusterable)
        activation_vecs = []
        for e in clusterable:
            vec = list(e.cluster_activations)
            while len(vec) < max_len: vec.append(0.0)
            activation_vecs.append(np.array(vec))

        merged_indices = set()
        merge_groups = []

        for i in range(len(clusterable)):
            if i in merged_indices: continue
            group = [i]
            vec_i = activation_vecs[i]
            norm_i = np.linalg.norm(vec_i)
            if norm_i < 1e-10: continue
            for j in range(i + 1, len(clusterable)):
                if j in merged_indices: continue
                vec_j = activation_vecs[j]
                norm_j = np.linalg.norm(vec_j)
                if norm_j < 1e-10: continue
                cosine_sim = np.dot(vec_i, vec_j) / (norm_i * norm_j)
                if cosine_sim >= self.ENTITY_CLUSTER_SIMILARITY:
                    group.append(j)
                    merged_indices.add(j)
            if len(group) > 1:
                merged_indices.add(i)
                merge_groups.append(group)

        if not merge_groups:
            return

        new_entities = []
        merged_set = set()

        for group in merge_groups:
            group_ents = [clusterable[idx] for idx in group]
            min_dim = min(len(e.weights) for e in group_ents if e.weights is not None)
            if min_dim == 0: continue

            avg_w = np.zeros(min_dim)
            for e in group_ents: avg_w += e.weights[:min_dim]
            avg_w /= len(group_ents)

            merge_count = self.entity_merge_counts.get(chain, 0)
            merged = Perceptron("entity", entity_type=f"{chain}_merged_{merge_count}", chain=chain)
            merged.weights = avg_w
            merged.utility = max(e.utility for e in group_ents)
            merged.familiarity = np.mean([e.familiarity for e in group_ents])
            merged.learning_rate = np.mean([e.learning_rate for e in group_ents])
            best_ent = max(group_ents, key=lambda e: e.activation_fit_score)
            merged.active_activation = best_ent.active_activation
            merged.activation_fit_score = best_ent.activation_fit_score

            new_entities.append(merged)
            self.entity_merge_counts[chain] = merge_count + 1

            for idx in group: merged_set.add(id(clusterable[idx]))

        kept_spawned = [e for e in clusterable if id(e) not in merged_set]

        other_perceptrons = [p for p in self.perceptrons if p.kind != "entity" or p.chain != chain]
        self.perceptrons = other_perceptrons + innate + kept_spawned + too_young + new_entities
        self._cache_valid = False

        total_merged = sum(len(g) for g in merge_groups)
        print(f"  🧩 [{chain}] CLUSTERED: {total_merged} → {len(new_entities)} | "
              f"Total: {self.get_chain_entity_count(chain)}")

    # Legacy wrappers
    def spawn_entity_from_novelty(self, learning_state, context_state=None, raw_position=None):
        self.spawn_entity_for_chain("overworld", learning_state, context_state, raw_position)
    def check_entity_capacity(self):
        self.check_chain_entity_capacity("overworld")
    def cluster_entities(self):
        self.cluster_chain_entities("overworld")
    def spawn_innate_entities(self, learning_state):
        self.spawn_innate_overworld_entities(learning_state)

    # =========================================================================
    # PIPELINE POOL PRUNING (NEW — matches AI agent)
    # =========================================================================

    def prune_pipeline_pools(self):
        """Prune low-quality perceptrons from pipeline pools, page to residual."""
        total_pruned = 0
        for pid, pipeline in self.pipelines.items():
            for i, pool in enumerate(pipeline.pools):
                pool_perceptrons = [p for p in self.perceptrons if p.pool_id == pool.pool_id]
                if len(pool_perceptrons) < pool.max_perceptrons * 0.7:
                    continue

                prunable = []
                for p in pool_perceptrons:
                    if len(p.cluster_activations) < 20:
                        continue
                    if p.familiarity >= 3.0 and p.activation_fit_score <= 0.05:
                        prunable.append(p)

                if not prunable:
                    continue

                max_prune = max(1, len(pool_perceptrons) // 3)
                prunable.sort(key=lambda p: p.utility)
                to_prune = prunable[:max_prune]

                for p in to_prune:
                    pool.page_to_residual(p)

                prune_ids = {id(p) for p in to_prune}
                self.perceptrons = [p for p in self.perceptrons if id(p) not in prune_ids]
                self._cache_valid = False
                total_pruned += len(to_prune)

                if to_prune:
                    print(f"  🧹 [{pid}.{pool.name}] Pruned {len(to_prune)} → residual "
                          f"(pool: {len(pool_perceptrons) - len(to_prune)} remain, "
                          f"residual: {len(pool.residual)})")

        return total_pruned

    # =========================================================================
    # CORE LEARNING HELPERS
    # =========================================================================

    def enforce_utility_floors(self):
        for a in self.actions():
            fl = self.MOVE_UTILITY_FLOOR if a.group == "move" else self.INTERACT_UTILITY_FLOOR
            a.utility = max(a.utility, fl)

    def get_spawn_threshold_adaptive(self, error_type='combined', percentile=50):
        history = {'numeric': self.numeric_error_history, 'visual': self.visual_error_history}.get(
            error_type, self.error_history)
        return max(0.001, np.percentile(history, percentile)) if len(history) >= 100 else 0.0005

    def stagnation_level(self, w=10):
        if len(self.prev_learning_states) < w: return 0.0
        r = list(self.prev_learning_states)[-w:]
        return 1.0 - np.tanh(np.mean([np.linalg.norm(r[i][:min(len(r[i]),len(r[i-1]))] - r[i-1][:min(len(r[i]),len(r[i-1]))]) for i in range(1, len(r))]) * 2.0)

    def predict_future_error(self, st, ac, cs, rp=None):
        ow_ents = self.entities(chain="overworld")
        entity_novelty = np.mean([e.predict(st) * e.utility for e in ow_ents]) if ow_ents else 0.5
        shared_ents = self.entities(chain="shared")
        if shared_ents:
            shared_signal = np.mean([e.predict(st) * e.utility for e in shared_ents])
            entity_novelty = entity_novelty * 0.7 + shared_signal * 0.3

        # Blend with overworld pipeline signal if available (NEW)
        ow_pipeline = self.overworld_pipeline
        if ow_pipeline.active:
            ow_output = ow_pipeline.pools[-2].get_cached_output()  # execution layer
            if np.any(ow_output != 0):
                pipeline_signal = np.mean(np.abs(ow_output))
                entity_novelty = entity_novelty * 0.5 + pipeline_signal * 0.5

        comb = entity_novelty * 0.7 + ac.utility * 0.3; cm = int(cs[2])
        loc = self.get_location_key(*(rp if rp else (cs[0]*255, cs[1]*255)), cm)
        td = min(self.map_novelty_debt.get(cm, 0.0), self.MAX_MAP_DEBT) + self.get_temp_debt(cm) + min(self.location_novelty.get(loc, 0.0), self.MAX_LOCATION_DEBT) * 0.5
        comb *= 1.0 / (1.0 + td * 5.0)
        if ac.action == self.current_repeated_action and self.consecutive_action_count > self.LEARNING_SLOWDOWN_START:
            comb *= 1.0 / (1.0 + (self.consecutive_action_count - self.LEARNING_SLOWDOWN_START) * 0.15)
        if self.detected_pattern and ac.action in self.detected_pattern:
            comb *= 1.0 / (1.0 + self.pattern_repeat_count * 0.2)
        return comb + np.random.randn() * 0.05

    def compute_multi_modal_error(self, s, ns):
        ml = min(len(s), len(ns)); d = [abs(ns[i]-s[i]) for i in range(min(8, ml))]
        w = [0.5,0.5,10.0,5.0,3.0,2.0,1.5,0.3]
        we = sum(di*wi for di, wi in zip(d, w[:len(d)])) + (np.linalg.norm(ns[8:ml]-s[8:ml])*2.0 if ml > 8 else 0.0)
        return we, sum(d), (np.linalg.norm(ns[8:ml]-s[8:ml]) if ml > 8 else 0.0)

    # =========================================================================
    # PIPELINE LEARNING (NEW — matches AI agent learn_pipeline)
    # =========================================================================

    def learn_pipeline(self, pipeline_id, raw_input, prev_raw_input=None,
                       error_scale=1.0, game_state_data=None):
        """
        Generic pipeline learning step:
        1. Forward pass through all layers
        2. Compute error from input change (if prev available)
        3. Backward credit assignment
        4. Spawn into pools that need it

        Returns: (pipeline_output, error, pipeline_active)
        """
        pipeline = self.pipelines.get(pipeline_id)
        if pipeline is None:
            return np.zeros(Pool.DEFAULT_OUTPUT_WIDTH), 0.0, False

        # Forward pass
        output, active = pipeline.forward(raw_input, self.perceptrons)

        if prev_raw_input is None:
            return output, 0.0, active

        # Compute error from state change
        min_dim = min(len(raw_input), len(prev_raw_input))
        error = np.linalg.norm(raw_input[:min_dim] - prev_raw_input[:min_dim]) * error_scale

        # Backward credit assignment
        pipeline.backward(error, self.perceptrons)

        # Spawn into pools that need it
        for i, pool in enumerate(pipeline.pools):
            if pool.needs_spawn(error):
                n_current = pool.get_perceptron_count(self.perceptrons)
                if n_current < pool.max_perceptrons:
                    layer_input = pipeline._layer_inputs[i] if i < len(pipeline._layer_inputs) else raw_input
                    self.spawn_into_pipeline_pool(
                        pipeline_id, i, layer_input,
                        game_state_data=game_state_data,
                    )

        return output, error, active

    # =========================================================================
    # CHAIN-GENERIC LEARNING (legacy — still used as fallback)
    # =========================================================================

    def learn_chain(self, chain, learning_state, next_learning_state, error_scale=1.0):
        if learning_state.shape != next_learning_state.shape:
            max_dim = max(len(learning_state), len(next_learning_state))
            learning_state = np.pad(learning_state, (0, max(0, max_dim - len(learning_state))))
            next_learning_state = np.pad(next_learning_state, (0, max(0, max_dim - len(next_learning_state))))

        diff = next_learning_state - learning_state
        error = np.linalg.norm(diff) * error_scale

        chain_history = self.get_chain_error_history(chain)
        chain_history.append(error)

        chain_actions = self.actions(chain=chain)
        chain_entities = self.entities(chain=chain)
        shared_actions = self.actions(chain="shared")
        shared_entities = self.entities(chain="shared")
        all_chain_perceptrons = chain_actions + chain_entities + shared_actions + shared_entities

        for p in all_chain_perceptrons:
            p.update(learning_state, error, stagnation=0.0)

        spawn_threshold = self.get_chain_spawn_threshold(chain, percentile=75)
        if error > spawn_threshold and len(chain_history) >= 50:
            n_ents = self.get_chain_entity_count(chain)
            cap = self.get_chain_entity_capacity(chain)
            if n_ents < cap * 1.5:
                self.spawn_entity_for_chain(chain, learning_state)

        if chain == "battle": self.prev_battle_learning_states.append(learning_state)
        elif chain == "party": self.prev_party_learning_states.append(learning_state)
        elif chain == "bag": self.prev_bag_learning_states.append(learning_state)

        return error

    # =========================================================================
    # CHAIN LEARNING — DUAL PIPELINE + LEGACY (UPDATED)
    # =========================================================================

    def learn_battle_chain(self, battle_data, party_data, turn_count):
        """Battle chain learning — runs both pipeline AND legacy chain."""
        current_state = build_learning_state_battle(battle_data, party_data, turn_count)

        if not self.innate_entities_spawned_battle:
            self.spawn_innate_battle_entities(current_state)

        prev_state = self.prev_battle_learning_states[-1] if len(self.prev_battle_learning_states) > 0 else None

        # === Pipeline learning (NEW) ===
        game_data = {
            'enemy_species': battle_data.get('enemy_species', -1),
            'player_species': battle_data.get('player_species', -1),
        }
        pipeline_output, pipeline_error, pipeline_active = self.learn_pipeline(
            "battle", current_state, prev_state,
            error_scale=1.0, game_state_data=game_data
        )

        # === Legacy chain learning (fallback — always runs) ===
        chain_error = 0.0
        if prev_state is not None:
            chain_error = self.learn_chain("battle", prev_state, current_state)
        else:
            self.prev_battle_learning_states.append(current_state)

        # Return blended error
        authority = self.battle_pipeline.get_total_authority()
        if authority > 0 and pipeline_error > 0:
            return pipeline_error * authority + chain_error * (1.0 - authority)
        return chain_error

    def learn_party_chain(self, party_data, active_slot=-1):
        """Party chain learning — pipeline + legacy."""
        current_state = build_learning_state_party(party_data, active_slot)
        prev_state = self.prev_party_learning_states[-1] if len(self.prev_party_learning_states) > 0 else None

        # Pipeline learning
        pipeline_output, pipeline_error, pipeline_active = self.learn_pipeline(
            "party", current_state, prev_state,
            error_scale=0.5
        )

        # Legacy chain learning
        chain_error = 0.0
        if prev_state is not None:
            chain_error = self.learn_chain("party", prev_state, current_state, error_scale=0.5)
        else:
            self.prev_party_learning_states.append(current_state)

        authority = self.party_pipeline.get_total_authority()
        if authority > 0 and pipeline_error > 0:
            return pipeline_error * authority + chain_error * (1.0 - authority)
        return chain_error

    def learn_bag_chain(self, bag_data, party_data, menu_data, in_battle=False):
        """Bag chain learning — pipeline + legacy."""
        current_state = build_learning_state_bag(bag_data, party_data, menu_data, in_battle)
        prev_state = self.prev_bag_learning_states[-1] if len(self.prev_bag_learning_states) > 0 else None

        # Pipeline learning
        game_data = {'pocket': bag_data.get('pocket', -1)}
        pipeline_output, pipeline_error, pipeline_active = self.learn_pipeline(
            "bag", current_state, prev_state,
            error_scale=0.5, game_state_data=game_data
        )

        # Legacy chain learning
        chain_error = 0.0
        if prev_state is not None:
            chain_error = self.learn_chain("bag", prev_state, current_state, error_scale=0.5)
        else:
            self.prev_bag_learning_states.append(current_state)

        authority = self.bag_pipeline.get_total_authority()
        if authority > 0 and pipeline_error > 0:
            return pipeline_error * authority + chain_error * (1.0 - authority)
        return chain_error

    # =========================================================================
    # MAIN LEARN — OVERWORLD + DISPATCH TO ACTIVE CHAINS (UPDATED)
    # =========================================================================

    def learn(self, ls, nls, cs, ncs, dead=False, raw_position=None, next_raw_position=None):
        if ls.shape != nls.shape:
            md = max(len(ls), len(nls)); ls = np.pad(ls, (0, max(0, md-len(ls)))); nls = np.pad(nls, (0, max(0, md-len(nls))))

        if not self.innate_entities_spawned_overworld:
            self.spawn_innate_overworld_entities(ls)

        pc = self.prev_context_states[-1] if self.prev_context_states else None
        pr = getattr(self, '_last_raw_position', None)
        self.update_exploration_tracking(cs, pc, raw_position, pr); self._last_raw_position = raw_position

        # Track previous game state for start menu detection
        self._prev_game_state_raw = self.game_state_raw

        we, ne, ve = self.compute_multi_modal_error(ls, nls)
        self.error_history.append(we); self.numeric_error_history.append(ne); self.visual_error_history.append(ve)

        # === OVERWORLD PIPELINE LEARNING (NEW) ===
        if cs[3] <= 0.5 and cs[4] <= 0.5:
            prev_ow = self.prev_learning_states[-1] if len(self.prev_learning_states) > 0 else None
            game_data = {'map_id': int(cs[2])}
            ow_output, ow_error, ow_active = self.learn_pipeline(
                "overworld", ls, prev_ow,
                error_scale=1.0, game_state_data=game_data
            )

        # Overworld entity spawning (legacy)
        spawn_threshold = self.get_spawn_threshold_adaptive('combined', percentile=75)
        if we > spawn_threshold and len(self.error_history) >= 100:
            menu_active = cs[4] > 0.5; battle_active = cs[3] > 0.5
            if not menu_active and not battle_active:
                self.spawn_entity_for_chain("overworld", ls, cs, raw_position)

        cm = int(cs[2]); loc = self.get_location_key(*(raw_position if raw_position else (cs[0]*255, cs[1]*255)), cm)
        self.visited_maps[cm] = self.visited_maps.get(cm, 0) + 1
        if len(self.location_memory) < self.MAX_LOCATIONS: self.location_memory[loc] = self.location_memory.get(loc, 0) + 1
        if self.visited_maps[cm] > 10: self.map_novelty_debt[cm] = min(self.MAX_MAP_DEBT, self.map_novelty_debt.get(cm, 0.0) + 0.05*(self.visited_maps[cm]-10))
        if self.location_memory.get(loc, 0) > 15: self.location_novelty[loc] = min(self.MAX_LOCATION_DEBT, self.location_novelty.get(loc, 0.0) + 0.1*(self.location_memory.get(loc,0)-15))
        if self.visited_maps[cm] > 30: we *= 0.5
        if self.location_memory.get(loc, 0) > 25: we *= 0.7

        stag = self.stagnation_level()
        lm = self.get_learning_multiplier(self.last_action) if self.last_action else 1.0
        if self.detected_pattern and self.last_action and self.last_action in self.detected_pattern: lm *= 0.5

        # Update overworld + shared perceptrons (legacy flat system)
        # FILTER OUT pool-assigned perceptrons — they get updates via pipeline backward (NEW)
        ow_perceptrons = self.actions(chain="overworld") + self.entities(chain="overworld")
        shared_perceptrons = self.actions(chain="shared") + self.entities(chain="shared")

        legacy_ow = [p for p in ow_perceptrons if p.pool_id is None]
        legacy_shared = [p for p in shared_perceptrons if p.pool_id is None]

        for p in legacy_ow + legacy_shared:
            m = lm if (p.kind == "action" and p.action == self.last_action) else 1.0
            if p.kind == "action" and self.detected_pattern and p.action in self.detected_pattern: m *= 0.5
            p.update(ls, we * m, stagnation=stag)

        for a in self.actions():
            if a.action in ['Start','Select'] and a.weights is not None: a.weights *= 0.999

        self.apply_repetition_penalty(); self.apply_pattern_penalty(); self.enforce_utility_floors()

        # Movement boost
        if pc is not None and np.linalg.norm(cs[:2] - pc[:2]) > 0.001 and self.last_action and self.consecutive_action_count < self.PENALTY_THRESHOLD:
            menu_active = cs[4] > 0.5; battle_active = cs[3] > 0.5
            if not menu_active and not battle_active:
                boost = 1.08
                if raw_position and self.is_near_map_edge(*raw_position): boost = 1.15
                for a in self.actions(chain="overworld") + self.actions(chain="shared"):
                    if a.action == self.last_action:
                        a.utility = min(a.utility * boost, 2.0); break

        # === DISPATCH TO ACTIVE CONTEXT CHAINS ===
        if cs[3] > 0.5 and self.battle_data.get('battle_cursor', -1) != -1:
            self.learn_battle_chain(self.battle_data, self.party_data, getattr(self, 'turn_count', 0))

        # Party chain
        if self.game_state_raw == 1 and self.menu_data.get('pc', -1) >= 0:
            self.learn_party_chain(self.party_data, -1)

        # Bag chain
        if self.bag_session_active or self.game_state_raw == 14:
            in_battle = cs[3] > 0.5
            self.learn_bag_chain(self.bag_data, self.party_data, self.menu_data, in_battle)

        # Periodic cleanup
        if self.timestep % 1000 == 0:
            self.cleanup_memory()
            self.prune_pipeline_pools()
        if self.timestep % self.SAVE_INTERVAL == 0: self.save_exploration_memory()
        self.action_history.append(self.last_action)

    # =========================================================================
    # SAVE MODEL CHECKPOINT (UPDATED — pipeline + revenge + pool_state)
    # =========================================================================

    def save_model_checkpoint(self, filepath=None):
        if filepath is None: filepath = BASE_PATH / "taught_model_checkpoint.json"
        model = {
            "timestep": self.timestep,
            "perceptrons": {"actions": [], "entities": []},
            "debt_tracking": {
                "map_novelty_debt": {str(k): v for k, v in self.map_novelty_debt.items()},
                "location_novelty": {str(k): v for k, v in self.location_novelty.items()},
                "visited_maps": {str(k): v for k, v in self.visited_maps.items()}
            },
            "control_mode": self.control_mode,
            "markov_stats": {
                "markov_action_count": self.markov_action_count,
                "curiosity_action_count": self.curiosity_action_count
            },
            "blend_stats": {"blend_count": self.blend_count, "last_blend_tier": self.blend_tier},
            "battle_stats": {
                "battles_recorded": self.current_battle_id,
                "battle_buffer_size": len(self.battle_data_buffer)
            },
            "chain_stats": {
                "entity_spawn_counts": self.entity_spawn_counts,
                "entity_merge_counts": self.entity_merge_counts,
                "entity_capacities": dict(self.ENTITY_CAPACITY),
            },
            "bag_stats": {
                "sessions_recorded": self.bag_sessions_metadata['bag_sessions_recorded'],
                "total_frames": self.bag_sessions_metadata['total_bag_frames'],
                "items_used": sorted(list(self.bag_sessions_metadata['items_used'])),
                "pockets_visited": sorted(list(self.bag_sessions_metadata['pockets_visited'])),
            },
            "start_menu_stats": {
                "start_menu_total_actions": self.start_menu_total_actions,
                "start_menu_markov_actions": self.start_menu_markov_actions,
                "sessions_recorded": self.start_menu_sessions_metadata['start_menu_sessions_recorded'],
                "total_frames": self.start_menu_sessions_metadata['total_start_menu_frames'],
                "targets_navigated": dict(self.start_menu_sessions_metadata['targets_navigated']),
            },
            "map_battle_stats": self.get_map_battle_stats_for_save(),
            "type_clusters": {
                "move_type_clusters": {str(k): v for k, v in self.move_type_clusters.items()},
                "species_type_clusters": {str(k): v for k, v in self.species_type_clusters.items()},
                "cluster_effectiveness": self.cluster_effectiveness,
                "move_to_cluster": {str(k): v for k, v in self.move_to_cluster.items()},
                "species_to_cluster": {str(k): v for k, v in self.species_to_cluster.items()},
                "clustering_run_count": self.clustering_run_count,
            },
            # === PIPELINE STATE (NEW) ===
            "pipelines": {pid: p.get_save_state() for pid, p in self.pipelines.items()},
            # === REVENGE TARGETS (NEW) ===
            "revenge_targets": self.revenge_targets,
        }

        # Action perceptrons with pool_state (NEW)
        for a in self.actions():
            model["perceptrons"]["actions"].append({
                "action": a.action, "group": a.group, "chain": a.chain,
                "utility": float(a.utility),
                "weights_shape": len(a.weights) if a.weights is not None else 0,
                "weights_nonzero": [[i, float(v)] for i, v in enumerate(a.weights) if abs(v) > 1e-10] if a.weights is not None else [],
                "learning_rate": float(a.learning_rate), "familiarity": float(a.familiarity),
                "activation_state": a.get_activation_state(),
                "pool_state": a.get_pool_state(),   # NEW
            })

        # Entity perceptrons with pool_state (NEW)
        for e in self.entities():
            model["perceptrons"]["entities"].append({
                "entity_type": e.entity_type, "chain": e.chain,
                "utility": float(e.utility),
                "weights_shape": len(e.weights) if e.weights is not None else 0,
                "weights_nonzero": [[i, float(v)] for i, v in enumerate(e.weights) if abs(v) > 1e-10] if e.weights is not None else [],
                "familiarity": float(e.familiarity),
                "learning_rate": float(e.learning_rate),
                "activation_state": e.get_activation_state(),
                "pool_state": e.get_pool_state(),   # NEW
            })

        try:
            with open(filepath, 'w') as f: json.dump(model, f, indent=2)
            n_act = len(model["perceptrons"]["actions"])
            n_ent = len(model["perceptrons"]["entities"])
            act_counts = {}
            for p in self.perceptrons:
                act_counts[p.active_activation] = act_counts.get(p.active_activation, 0) + 1
            act_str = ' '.join(f'{k}:{v}' for k, v in sorted(act_counts.items(), key=lambda x: x[1], reverse=True))
            sm_sessions = self.start_menu_sessions_metadata['start_menu_sessions_recorded']
            sm_frames = self.start_menu_sessions_metadata['total_start_menu_frames']
            sm_str = f" SM:{sm_sessions}sess/{sm_frames}f" if sm_sessions > 0 else ""
            tc_str = f" TC:{self.clustering_run_count}runs" if self.clustering_run_count > 0 else ""
            p_summary = self.get_pipeline_summary()
            p_str = f" Pipes:[{p_summary}]" if p_summary != 'all empty' else ""
            print(f"💾 Model saved: step {self.timestep} | {n_act}a {n_ent}e | act:[{act_str}]{sm_str}{tc_str}{p_str} → {filepath}")
        except Exception as e: print(f"❌ Save error: {e}")

    # =========================================================================
    # LOAD TAUGHT MODEL (UPDATED — pipeline + revenge + pool_state)
    # =========================================================================

    def load_taught_model(self, fp):
        if not Path(fp).exists(): return 0
        try:
            with open(fp, 'r') as f: model = json.load(f)
            if "perceptrons" not in model: return 0

            # Restore action perceptrons
            for sa in model["perceptrons"]["actions"]:
                for a in self.actions():
                    if a.action == sa["action"]:
                        a.utility = sa["utility"]
                        a.learning_rate = sa.get("learning_rate", 0.01)
                        a.familiarity = sa.get("familiarity", 0.0)
                        a.chain = sa.get("chain", "shared")
                        if sa.get("weights_nonzero"):
                            dim = sa.get("weights_shape", 1376); a.weights = np.zeros(dim)
                            for idx, val in sa["weights_nonzero"]:
                                if idx < dim: a.weights[idx] = val
                        a.set_activation_state(sa.get("activation_state"))
                        a.set_pool_state(sa.get("pool_state"))   # NEW
                        break

            # Restore entity perceptrons
            saved_entities = model["perceptrons"].get("entities", [])
            if saved_entities:
                innate_types = {"sense_menu", "sense_battle", "sense_movement", "sense_map_transition",
                                "battle_hp_crisis", "battle_enemy_weak", "battle_species_match",
                                "battle_status", "battle_trainer"}

                for se in saved_entities:
                    et = se.get("entity_type", "unknown")
                    chain = se.get("chain", "shared")

                    if et in innate_types:
                        existing = None
                        for e in self.entities():
                            if e.entity_type == et:
                                existing = e; break
                        if existing:
                            existing.utility = se.get("utility", 1.0)
                            existing.familiarity = se.get("familiarity", 0.0)
                            existing.learning_rate = se.get("learning_rate", 0.01)
                            existing.chain = chain
                            if se.get("weights_nonzero"):
                                dim = se.get("weights_shape", 1376)
                                existing.ensure_weights(dim)
                                existing.weights = np.zeros(dim)
                                for idx, val in se["weights_nonzero"]:
                                    if idx < dim: existing.weights[idx] = val
                            existing.set_activation_state(se.get("activation_state"))
                            existing.set_pool_state(se.get("pool_state"))   # NEW
                            continue

                    if se.get("weights_nonzero"):
                        e = Perceptron("entity", entity_type=et, chain=chain)
                        dim = se.get("weights_shape", 1376)
                        e.weights = np.zeros(dim)
                        for idx, val in se["weights_nonzero"]:
                            if idx < dim: e.weights[idx] = val
                        e.utility = se.get("utility", 1.0)
                        e.familiarity = se.get("familiarity", 0.0)
                        e.learning_rate = se.get("learning_rate", 0.01)
                        e.set_activation_state(se.get("activation_state"))
                        e.set_pool_state(se.get("pool_state"))   # NEW
                        self.add(e)

                self.innate_entities_spawned = True
                ow_innate = {"sense_menu", "sense_battle", "sense_movement", "sense_map_transition"}
                bat_innate = {"battle_hp_crisis", "battle_enemy_weak", "battle_species_match", "battle_status", "battle_trainer"}
                restored_types = {se.get("entity_type") for se in saved_entities}
                if ow_innate.issubset(restored_types):
                    self.innate_entities_spawned_overworld = True
                if bat_innate.issubset(restored_types):
                    self.innate_entities_spawned_battle = True

                print(f"  🧩 Restored {len(saved_entities)} entities from checkpoint")

            # Restore debt tracking
            if "debt_tracking" in model:
                d = model["debt_tracking"]
                self.map_novelty_debt = {int(k): v for k, v in d.get("map_novelty_debt", {}).items()}
                self.visited_maps = {int(k): v for k, v in d.get("visited_maps", {}).items()}
                for k, v in d.get("location_novelty", {}).items():
                    try: self.location_novelty[eval(k)] = v
                    except: pass

            # Restore stats
            ms = model.get("markov_stats", {})
            self.markov_action_count = ms.get("markov_action_count", 0)
            self.curiosity_action_count = ms.get("curiosity_action_count", 0)

            bs = model.get("blend_stats", {})
            self.blend_count = bs.get("blend_count", 0)
            self.blend_tier = bs.get("last_blend_tier", 0)

            bts = model.get("battle_stats", {})
            self.current_battle_id = bts.get("battles_recorded", 0)

            cs = model.get("chain_stats", {})
            self.entity_spawn_counts = cs.get("entity_spawn_counts", self.entity_spawn_counts)
            self.entity_merge_counts = cs.get("entity_merge_counts", self.entity_merge_counts)
            for chain, cap in cs.get("entity_capacities", {}).items():
                self.ENTITY_CAPACITY[chain] = cap

            bag_s = model.get("bag_stats", {})
            self.bag_sessions_metadata['bag_sessions_recorded'] = bag_s.get("sessions_recorded", 0)
            self.bag_sessions_metadata['total_bag_frames'] = bag_s.get("total_frames", 0)
            self.bag_sessions_metadata['items_used'] = set(bag_s.get("items_used", []))
            self.bag_sessions_metadata['pockets_visited'] = set(bag_s.get("pockets_visited", []))

            sm_s = model.get("start_menu_stats", {})
            self.start_menu_total_actions = sm_s.get("start_menu_total_actions", 0)
            self.start_menu_markov_actions = sm_s.get("start_menu_markov_actions", 0)
            self.start_menu_sessions_metadata['start_menu_sessions_recorded'] = sm_s.get("sessions_recorded", 0)
            self.start_menu_sessions_metadata['total_start_menu_frames'] = sm_s.get("total_frames", 0)
            self.start_menu_sessions_metadata['targets_navigated'] = sm_s.get("targets_navigated", {})

            if "map_battle_stats" in model:
                self.map_battle_stats = {int(k): v for k, v in model["map_battle_stats"].items()}
                self.map_step_counters = {mid: s.get('total_steps_on_map', 0) for mid, s in self.map_battle_stats.items()}

            if "type_clusters" in model:
                tc = model["type_clusters"]
                self.move_type_clusters = {int(k): v for k, v in tc.get('move_type_clusters', {}).items()}
                self.species_type_clusters = {int(k): v for k, v in tc.get('species_type_clusters', {}).items()}
                self.cluster_effectiveness = tc.get('cluster_effectiveness', {})
                self.move_to_cluster = {int(k): int(v) for k, v in tc.get('move_to_cluster', {}).items()}
                self.species_to_cluster = {int(k): int(v) for k, v in tc.get('species_to_cluster', {}).items()}
                self.clustering_run_count = tc.get('clustering_run_count', 0)
                if self.move_type_clusters:
                    print(f"  🧬 Type clusters restored: {len(self.move_type_clusters)}mc "
                          f"{len(self.species_type_clusters)}sc {len(self.cluster_effectiveness)}eff")

            # === LOAD PIPELINE STATE (NEW) ===
            if "pipelines" in model:
                for pid, p_state in model["pipelines"].items():
                    pipeline = self.pipelines.get(pid)
                    if pipeline:
                        pipeline.load_save_state(p_state)
                print(f"  🔗 Pipeline state restored")

            # === LOAD REVENGE TARGETS (NEW) ===
            if "revenge_targets" in model:
                self.revenge_targets = model["revenge_targets"]
                active_revenge = sum(1 for t in self.revenge_targets.values()
                                    if t.get('status') in ('grinding', 'ready'))
                if self.revenge_targets:
                    print(f"  ⚔️🔥 Revenge targets restored: {len(self.revenge_targets)} "
                          f"({active_revenge} active)")

            self.timestep = model.get("timestep", 0)

            # Log chain distribution
            chain_stats = self.get_chain_stats()
            if chain_stats:
                parts = [f"{c}:{s['actions']}a+{s['entities']}e" for c, s in chain_stats.items()]
                print(f"  🧬 Chains: {' | '.join(parts)}")

            # Log pipeline summary
            p_summary = self.get_pipeline_summary()
            if p_summary != 'all empty':
                print(f"  🔗 Pipelines: {p_summary}")

            # Log activation discoveries
            act_counts = {}
            for p in self.perceptrons:
                act_counts[p.active_activation] = act_counts.get(p.active_activation, 0) + 1
            if len(act_counts) > 1:
                parts = [f"{k}:{v}" for k, v in sorted(act_counts.items(), key=lambda x: x[1], reverse=True)]
                print(f"  🧬 Activations: {' '.join(parts)}")

            if self.start_menu_sessions_metadata['start_menu_sessions_recorded'] > 0:
                sm_sess = self.start_menu_sessions_metadata['start_menu_sessions_recorded']
                sm_fr = self.start_menu_sessions_metadata['total_start_menu_frames']
                sm_tgt = self.start_menu_sessions_metadata['targets_navigated']
                print(f"  📋 Start menu: {sm_sess}sess {sm_fr}f targets={sm_tgt}")

            print(f"  📖 Model loaded: step {self.timestep}")
            print(f"     Utilities: {[f'{a.action}:{a.utility:.3f}' for a in self.actions()]}")

            return self.timestep
        except Exception as e:
            print(f"  ⚠️ Load error: {e}")
            return 0

    def save_model(self, fp=None): self.save_model_checkpoint(fp)
    def load_model(self, fp=None): return self.load_taught_model(fp or (BASE_PATH / "taught_model_checkpoint.json"))

In [4]:
# ============================================================================
# CELL 6A: Post-Processors (6 total)
# ============================================================================
# SYNC TO AI AGENT v17.2:
# 1. run_per_map_analysis() — UNCHANGED
# 2. run_nav_target_extraction() — UNCHANGED
# 3. run_battle_extraction() — UNCHANGED (expanded battle data fields)
# 4. run_bag_extraction() — UNCHANGED
# 5. run_event_timeline_generation() — UNCHANGED
# 6. NEW: run_start_menu_extraction() — writes taught_start_menu_transitions.json
# 7. run_all_post_processing() — calls all 6
# ============================================================================

from collections import defaultdict
from datetime import datetime

# =========================================================================
# POST-PROCESSOR 1: per_map_analysis (UNCHANGED)
# =========================================================================
def run_per_map_analysis():
    if not TAUGHT_TRANSITIONS_FILE.exists():
        print("  ⚠️ No taught_transitions.json — skipping per_map_analysis")
        return
    with open(TAUGHT_TRANSITIONS_FILE, 'r') as f:
        data = json.load(f)
    batches = data.get('batches', [])
    if not batches:
        print("  ⚠️ No batches — skipping per_map_analysis")
        return
    all_frames = []
    for batch in batches:
        for frame in batch.get('frames', []):
            all_frames.append(frame)
    print(f"  Analyzing {len(all_frames)} frames...")
    frames_by_map = defaultdict(list)
    for i, frame in enumerate(all_frames):
        mid = frame.get('state', {}).get('map_id')
        if mid is not None:
            frames_by_map[mid].append((i, frame))
    per_map = {}
    for map_id, indexed_frames in frames_by_map.items():
        mk = str(map_id)
        ta = defaultdict(lambda: defaultdict(int))
        td = defaultdict(lambda: defaultdict(int))
        tt = defaultdict(int)
        for i, fr in indexed_frames:
            s = fr.get('state', {}); x, y = s.get('x', 0), s.get('y', 0)
            tk = f"{x}_{y}"; act = fr.get('action', 'NONE'); dr = s.get('direction', 0)
            ta[tk][act] += 1; td[tk][str(dr)] += 1; tt[tk] += 1
        ap = {}
        for tk in ta:
            total = tt[tk]
            if total == 0: continue
            probs = {a: round(c/total, 3) for a, c in ta[tk].items()}
            probs['total_frames'] = total; probs['direction_facing'] = dict(td[tk]); ap[tk] = probs
        mg = defaultdict(set)
        for idx in range(len(indexed_frames) - 1):
            _, f1 = indexed_frames[idx]; _, f2 = indexed_frames[idx+1]
            s1, s2 = f1.get('state', {}), f2.get('state', {})
            if s1.get('map_id') != s2.get('map_id'): continue
            x1, y1 = s1.get('x',0), s1.get('y',0); x2, y2 = s2.get('x',0), s2.get('y',0)
            if (x1,y1) != (x2,y2):
                t1, t2 = f"{x1}_{y1}", f"{x2}_{y2}"; mg[t1].add(t2); mg[t2].add(t1)
        mg_s = {k: sorted(list(v)) for k, v in mg.items()}
        dp = []
        for idx in range(1, len(indexed_frames)):
            _, fp = indexed_frames[idx-1]; gi, fc = indexed_frames[idx]
            ap2, ac = fp.get('action','NONE'), fc.get('action','NONE')
            if ap2 != ac and ac != 'NONE' and ap2 != 'NONE':
                s = fc.get('state', {})
                dp.append({'position': [s.get('x',0), s.get('y',0)], 'from_action': ap2, 'to_action': ac,
                    'frame': gi, 'facing': s.get('direction',0),
                    'context': {'in_battle': s.get('in_battle',0), 'in_menu': s.get('in_menu',0)}})
        dd = defaultdict(lambda: {'visits': 0, 'frames': [], 'current_run': 0}); lt = None
        for i, fr in indexed_frames:
            s = fr.get('state', {}); tk = f"{s.get('x',0)}_{s.get('y',0)}"
            if tk == lt: dd[tk]['current_run'] += 1
            else:
                if lt is not None and dd[lt]['current_run'] > 0: dd[lt]['frames'].append(dd[lt]['current_run'])
                dd[tk]['visits'] += 1; dd[tk]['current_run'] = 1; lt = tk
        if lt is not None and dd[lt]['current_run'] > 0: dd[lt]['frames'].append(dd[lt]['current_run'])
        dtimes = {}
        for tk, d in dd.items():
            runs = d['frames']
            if not runs: continue
            total = sum(runs)
            dtimes[tk] = {'total_frames': total, 'visits': d['visits'], 'avg_dwell': round(total/len(runs),1), 'max_dwell': max(runs)}
        ps = []; cs = None
        for idx in range(len(indexed_frames)):
            gi, fr = indexed_frames[idx]; s = fr.get('state', {}); act = fr.get('action', 'NONE')
            x, y = s.get('x',0), s.get('y',0)
            if act not in ('UP','DOWN','LEFT','RIGHT'):
                if cs and len(cs['tiles']) >= 3:
                    cs['end'] = cs['tiles'][-1]; cs['length'] = len(cs['tiles']); cs['frame_end'] = gi; ps.append(cs)
                cs = None; continue
            if cs is None or act != cs['primary_action']:
                if cs and len(cs['tiles']) >= 3:
                    cs['end'] = cs['tiles'][-1]; cs['length'] = len(cs['tiles']); cs['frame_end'] = gi-1; ps.append(cs)
                cs = {'start': [x,y], 'end': [x,y], 'tiles': [[x,y]], 'primary_action': act, 'actions': [act], 'length': 1, 'frame_start': gi, 'frame_end': gi}
            else:
                pos = [x, y]
                if pos != cs['tiles'][-1]: cs['tiles'].append(pos)
                cs['actions'].append(act)
        if cs and len(cs['tiles']) >= 3:
            cs['end'] = cs['tiles'][-1]; cs['length'] = len(cs['tiles']); cs['frame_end'] = indexed_frames[-1][0]; ps.append(cs)
        per_map[mk] = {'action_probabilities': ap, 'movement_graph': mg_s, 'decision_points': dp, 'dwell_times': dtimes, 'path_segments': ps}
        print(f"    Map {map_id}: {len(ap)} tiles, {len(dp)} decisions, {len(ps)} paths")
    data['per_map_analysis'] = per_map
    with open(TAUGHT_TRANSITIONS_FILE, 'w') as f:
        json.dump(data, f)
    print(f"  ✅ per_map_analysis → {TAUGHT_TRANSITIONS_FILE.name}")


# =========================================================================
# POST-PROCESSOR 2: taught_nav_targets.json (UNCHANGED)
# =========================================================================
NAV_TARGETS_PATH = BASE_PATH / "taught_nav_targets.json"
ANALYSIS_WINDOW_AFTER = 40
RECENT_WINDOW = 100
MIN_FORWARD_PROGRESS = 0.5
DEDUP_RADIUS_NAV = 2
BACKTRACK_WINDOW = 50
BACKTRACK_PROXIMITY = 3

def run_nav_target_extraction():
    if not TAUGHT_TRANSITIONS_FILE.exists():
        print("  ⚠️ No taught_transitions.json — writing empty nav targets")
        _write_empty_nav_targets(); return
    with open(TAUGHT_TRANSITIONS_FILE, 'r') as f:
        data = json.load(f)
    all_frames = []
    for batch in data.get('batches', []):
        for frame in batch.get('frames', []): all_frames.append(frame)
    if not all_frames:
        print("  ⚠️ No frames — writing empty nav targets"); _write_empty_nav_targets(); return
    print(f"  Scanning {len(all_frames)} frames for novelty...")
    novelty_points = []
    for i, frame in enumerate(all_frames):
        s = frame.get('state', {}); x, y = s.get('x',0), s.get('y',0)
        mid = s.get('map_id',0); d = s.get('direction',0)
        ib, im = s.get('in_battle',0), s.get('in_menu',0)
        act = frame.get('action', 'NONE')
        ps = all_frames[i-1].get('state', {}) if i > 0 else {}
        pmid, pib = ps.get('map_id', mid), ps.get('in_battle', 0)
        if i > 0 and mid != pmid:
            px, py = ps.get('x', x), ps.get('y', y)
            novelty_points.append({'position': [px,py], 'map_id': pmid, 'direction': ps.get('direction',d),
                'frame_index': i, 'novelty_type': 'map_transition', 'destination_map': mid}); continue
        if ib == 1 and pib == 0:
            novelty_points.append({'position': [x,y], 'map_id': mid, 'direction': d,
                'frame_index': i, 'novelty_type': 'battle', 'destination_map': None}); continue
        if act == 'A' and ib == 0 and im == 0:
            triggered = False
            for j in range(i+1, min(i+9, len(all_frames))):
                fs = all_frames[j].get('state', {})
                if fs.get('in_menu', 0) != im or fs.get('map_id', mid) != mid: triggered = True; break
            if triggered:
                novelty_points.append({'position': [x,y], 'map_id': mid, 'direction': d,
                    'frame_index': i, 'novelty_type': 'interaction', 'destination_map': None}); continue
        if ib == 0 and im == 0 and i > RECENT_WINDOW:
            was_recent = any(all_frames[j].get('state',{}).get('map_id')==mid and all_frames[j].get('state',{}).get('x')==x and all_frames[j].get('state',{}).get('y')==y for j in range(max(0,i-RECENT_WINDOW), max(0,i-5)))
            if not was_recent:
                too_close = novelty_points and novelty_points[-1]['map_id']==mid and abs(novelty_points[-1]['position'][0]-x)+abs(novelty_points[-1]['position'][1]-y)<=DEDUP_RADIUS_NAV
                if not too_close:
                    novelty_points.append({'position': [x,y], 'map_id': mid, 'direction': d,
                        'frame_index': i, 'novelty_type': 'new_area', 'destination_map': None})
    tc = defaultdict(int)
    for np_item in novelty_points: tc[np_item['novelty_type']] += 1
    print(f"    Novelty points: {len(novelty_points)} ({', '.join(f'{t}:{c}' for t,c in tc.items())})")
    scored = []
    for np_item in novelty_points:
        fi = np_item['frame_index']; mid = np_item['map_id']; px, py = np_item['position']
        before = set()
        for j in range(max(0,fi-RECENT_WINDOW), fi):
            js = all_frames[j].get('state', {})
            if js.get('in_battle',0)==0 and js.get('in_menu',0)==0:
                before.add((js.get('map_id',0), js.get('x',0), js.get('y',0)))
        after_new, after_total = 0, 0
        for j in range(fi+1, min(fi+1+ANALYSIS_WINDOW_AFTER, len(all_frames))):
            js = all_frames[j].get('state', {})
            if js.get('in_battle',0)==1 or js.get('in_menu',0)==1: continue
            after_total += 1
            if (js.get('map_id',0), js.get('x',0), js.get('y',0)) not in before: after_new += 1
        fwd = after_new / after_total if after_total > 0 else 0.0
        bt = False
        if np_item['novelty_type'] == 'map_transition':
            for j in range(fi+1, min(fi+1+BACKTRACK_WINDOW, len(all_frames))):
                if all_frames[j].get('state',{}).get('map_id') == mid: bt = True; break
        else:
            for j in range(fi+5, min(fi+1+BACKTRACK_WINDOW, len(all_frames))):
                js = all_frames[j].get('state', {})
                if js.get('map_id')==mid and abs(js.get('x',0)-px)+abs(js.get('y',0)-py)<=BACKTRACK_PROXIMITY: bt = True; break
        if bt or fwd < MIN_FORWARD_PROGRESS: continue
        np_item['forward_progress_score'] = round(fwd, 3); scored.append(np_item)
    print(f"    After filtering: {len(scored)} targets")
    deduped = []
    by_map = defaultdict(list)
    for t in scored: by_map[t['map_id']].append(t)
    for mid, targets in by_map.items():
        targets.sort(key=lambda t: t['forward_progress_score'], reverse=True)
        kept = []
        for t in targets:
            tx, ty = t['position']
            if not any(abs(tx-k['position'][0])+abs(ty-k['position'][1])<=DEDUP_RADIUS_NAV for k in kept): kept.append(t)
        deduped.extend(kept)
    deduped.sort(key=lambda t: t['frame_index'])
    tbm = defaultdict(list); go = []
    for order, t in enumerate(deduped, 1):
        mk = str(t['map_id'])
        tbm[mk].append({'position': t['position'], 'direction': t['direction'], 'order': order,
            'progress_type': t['novelty_type'], 'forward_progress_score': t['forward_progress_score'],
            'destination_map': t.get('destination_map'), 'frame_index': t['frame_index']})
        go.append({'map_id': t['map_id'], 'position': t['position'], 'order': order})
    output = {'targets_by_map': dict(tbm), 'global_order': go,
        'metadata': {'total_targets': len(deduped), 'maps_with_targets': sorted(set(t['map_id'] for t in deduped)),
            'analysis_window_after': ANALYSIS_WINDOW_AFTER, 'min_forward_progress': MIN_FORWARD_PROGRESS,
            'dedup_radius': DEDUP_RADIUS_NAV, 'generated_from_frames': len(all_frames)}}
    with open(NAV_TARGETS_PATH, 'w') as f:
        json.dump(output, f, indent=2)
    print(f"  ✅ taught_nav_targets.json → {len(deduped)} targets across {len(tbm)} maps")

def _write_empty_nav_targets():
    with open(NAV_TARGETS_PATH, 'w') as f:
        json.dump({'targets_by_map': {}, 'global_order': [], 'metadata': {'total_targets': 0, 'maps_with_targets': [],
            'analysis_window_after': ANALYSIS_WINDOW_AFTER, 'min_forward_progress': MIN_FORWARD_PROGRESS,
            'dedup_radius': DEDUP_RADIUS_NAV, 'generated_from_frames': 0}}, f, indent=2)


# =========================================================================
# POST-PROCESSOR 3: taught_battle_transitions.json (UNCHANGED)
# =========================================================================
BATTLE_TRANSITIONS_PATH = BASE_PATH / "taught_battle_transitions.json"
POKEMON_CENTER_MAPS = {1, 2, 3, 4}

def run_battle_extraction():
    if not TAUGHT_TRANSITIONS_FILE.exists():
        print("  ⚠️ No taught_transitions.json — writing empty battle transitions")
        _write_empty_battle_transitions(); return
    with open(TAUGHT_TRANSITIONS_FILE, 'r') as f:
        data = json.load(f)
    all_frames = []
    for batch in data.get('batches', []):
        bt = batch.get('batch_type', 'steady')
        for frame in batch.get('frames', []):
            frame['_batch_type'] = bt
            all_frames.append(frame)
    if not all_frames:
        print("  ⚠️ No frames — writing empty battle transitions")
        _write_empty_battle_transitions(); return
    print(f"  Scanning {len(all_frames)} frames for battles...")

    battle_buffer = brain.get_all_battle_data_buffer()
    buffer_by_battle_id = defaultdict(list)
    for entry in battle_buffer:
        buffer_by_battle_id[entry['battle_id']].append(entry)
    print(f"    Battle data buffer: {len(battle_buffer)} entries across {len(buffer_by_battle_id)} battles")

    battle_sequences = []
    current_battle = None
    battle_id = 0
    for i, frame in enumerate(all_frames):
        s = frame.get('state', {})
        ib = s.get('in_battle', 0)
        if ib == 1 and current_battle is None:
            battle_id += 1
            current_battle = {'battle_id': battle_id, 'start_frame': i, 'end_frame': i,
                'map_id': s.get('map_id', 0), 'frames': [], 'recent_actions_buffer': [],
                'frame_index_in_battle': 0}
        if ib == 1 and current_battle is not None:
            action = frame.get('action', 'NONE')
            recent = frame.get('recent_actions', [])
            if len(recent) < 8: recent = (['NONE'] * (8 - len(recent))) + recent
            elif len(recent) > 8: recent = recent[-8:]
            bd_short = _lookup_battle_data(buffer_by_battle_id, battle_id, current_battle['frame_index_in_battle'])
            battle_frame = {
                'state': {'map_id': s.get('map_id', 0), 'x': s.get('x', 0), 'y': s.get('y', 0),
                    'direction': s.get('direction', 0), 'in_battle': 1, 'in_menu': s.get('in_menu', 0)},
                'action': action if action else 'NONE',
                'recent_actions': recent,
                'frame_offset': frame.get('frame_offset', 0),
                'batch_type': frame.get('_batch_type', 'steady'),
                'battle_data': bd_short}
            current_battle['frames'].append(battle_frame)
            current_battle['end_frame'] = i
            current_battle['frame_index_in_battle'] += 1
            if action and action != 'NONE': current_battle['recent_actions_buffer'].append(action)
        elif ib == 0 and current_battle is not None:
            if len(current_battle['frames']) >= 2:
                outcome = _detect_battle_outcome(current_battle, all_frames, i)
                battle_sequences.append({'battle_id': current_battle['battle_id'],
                    'start_frame': current_battle['start_frame'], 'end_frame': current_battle['end_frame'],
                    'map_id': current_battle['map_id'], 'duration_frames': len(current_battle['frames']),
                    'outcome': outcome, 'frames': current_battle['frames']})
            current_battle = None
    if current_battle is not None and len(current_battle['frames']) >= 2:
        battle_sequences.append({'battle_id': current_battle['battle_id'],
            'start_frame': current_battle['start_frame'], 'end_frame': current_battle['end_frame'],
            'map_id': current_battle['map_id'], 'duration_frames': len(current_battle['frames']),
            'outcome': 'unknown', 'frames': current_battle['frames']})

    flat_frames = []
    for seq in battle_sequences:
        for frame in seq['frames']:
            flat_frame = dict(frame); flat_frame['battle_id'] = seq['battle_id']; flat_frames.append(flat_frame)

    outcomes = defaultdict(int); maps_with_battles = set()
    for seq in battle_sequences: outcomes[seq['outcome']] += 1; maps_with_battles.add(seq['map_id'])
    avg_length = (sum(s['duration_frames'] for s in battle_sequences) / len(battle_sequences)) if battle_sequences else 0
    frames_with_bd = sum(1 for f in flat_frames if f.get('battle_data') and f['battle_data'].get('bc', -1) != -1)

    seq_counts = defaultdict(int)
    for seq in battle_sequences:
        actions = [f['action'] for f in seq['frames'] if f['action'] != 'NONE']
        for win_size in [2, 3, 4]:
            for j in range(len(actions) - win_size + 1):
                seq_counts[tuple(actions[j:j+win_size])] += 1
    common_seqs = sorted(seq_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    most_common = [{'sequence': list(s), 'count': c, 'context': _guess_sequence_context(list(s))} for s, c in common_seqs]

    metadata = {'total_battle_frames': len(flat_frames), 'battles_recorded': len(battle_sequences),
        'avg_battle_length': round(avg_length, 1), 'outcomes': dict(outcomes),
        'maps_with_battles': sorted(maps_with_battles), 'most_common_sequences': most_common,
        'frames_with_battle_data': frames_with_bd,
        'battle_data_coverage': round(frames_with_bd / max(1, len(flat_frames)), 3)}

    with open(BATTLE_TRANSITIONS_PATH, 'w') as f:
        json.dump({'battle_sequences': battle_sequences, 'flat_frames': flat_frames, 'metadata': metadata}, f)
    print(f"  ✅ taught_battle_transitions.json:")
    print(f"     Battles: {len(battle_sequences)} | Frames: {len(flat_frames)}")
    print(f"     Outcomes: {dict(outcomes)} | Avg length: {avg_length:.1f}")
    print(f"     Battle data coverage: {frames_with_bd}/{len(flat_frames)} ({metadata['battle_data_coverage']:.0%})")


def _lookup_battle_data(buffer_by_battle_id, battle_id, frame_index):
    entries = buffer_by_battle_id.get(battle_id, [])
    if not entries:
        return {'bc': -1, 'mc': -1, 'ps': -1, 'es': -1, 'ph': -1, 'pm': -1, 'eh': -1, 'em': -1,
                'pl': -1, 'el': -1, 'pst': 0, 'est': 0, 'bt': 0, 'pc': -1}
    idx = min(frame_index, len(entries) - 1)
    bd = entries[idx].get('battle_data', {})
    return {
        'bc': bd.get('battle_cursor', -1), 'mc': bd.get('move_cursor', -1),
        'ps': bd.get('player_species', -1), 'es': bd.get('enemy_species', -1),
        'ph': bd.get('player_hp', -1), 'pm': bd.get('player_max_hp', -1),
        'eh': bd.get('enemy_hp', -1), 'em': bd.get('enemy_max_hp', -1),
        'pl': bd.get('player_level', -1), 'el': bd.get('enemy_level', -1),
        'pst': bd.get('player_status', 0), 'est': bd.get('enemy_status', 0),
        'bt': bd.get('battle_type', 0), 'pc': bd.get('party_cursor', -1),
    }


def _detect_battle_outcome(battle, all_frames, end_index):
    frames = battle.get('frames', [])
    if frames:
        last_bd = frames[-1].get('battle_data', {})
        eh = last_bd.get('eh', -1); ph = last_bd.get('ph', -1)
        if eh == 0 and eh != -1: return 'win'
        if ph == 0 and ph != -1: return 'loss'
        if eh > 0 and ph > 0: pass
    actions = battle.get('recent_actions_buffer', [])
    if len(actions) >= 3:
        last_actions = actions[-6:] if len(actions) >= 6 else actions
        for pattern in [['DOWN', 'RIGHT', 'A'], ['DOWN', 'A'], ['RIGHT', 'DOWN', 'A']]:
            plen = len(pattern)
            for k in range(len(last_actions) - plen + 1):
                if last_actions[k:k+plen] == pattern and k >= len(last_actions) - plen - 2:
                    return 'run'
    if end_index + 5 < len(all_frames):
        for j in range(end_index, min(end_index + 10, len(all_frames))):
            post_map = all_frames[j].get('state', {}).get('map_id', -1)
            if post_map in POKEMON_CENTER_MAPS and post_map != battle.get('map_id', -1):
                return 'loss'
    return 'win'


def _guess_sequence_context(seq):
    if seq == ['A', 'A', 'A', 'A']: return 'mashing_A_through_text_or_selecting_fight'
    if seq == ['A', 'A']: return 'selecting_fight_and_move'
    if 'DOWN' in seq and 'A' in seq: return 'navigating_menu_then_selecting'
    if seq == ['B', 'B']: return 'cancelling_or_backing_out'
    if all(a == 'A' for a in seq): return 'mashing_A'
    return 'battle_input_sequence'


def _write_empty_battle_transitions():
    with open(BATTLE_TRANSITIONS_PATH, 'w') as f:
        json.dump({'battle_sequences': [], 'flat_frames': [], 'metadata': {
            'total_battle_frames': 0, 'battles_recorded': 0, 'avg_battle_length': 0,
            'outcomes': {}, 'maps_with_battles': [], 'most_common_sequences': [],
            'frames_with_battle_data': 0, 'battle_data_coverage': 0.0}}, f, indent=2)


# =========================================================================
# POST-PROCESSOR 4: taught_bag_transitions.json (UNCHANGED)
# =========================================================================

def run_bag_extraction():
    bag_data = brain.get_accumulated_bag_data()
    n_frames = len(bag_data['bag_frames'])
    n_sessions = bag_data['metadata']['bag_sessions_recorded']

    if n_frames == 0:
        print("  ⚠️ No bag frames recorded — writing empty bag transitions")
        _write_empty_bag_transitions()
        return

    with open(TAUGHT_BAG_TRANSITIONS_FILE, 'w') as f:
        json.dump(bag_data, f)

    items_used = bag_data['metadata']['items_used']
    pockets = bag_data['metadata']['pockets_visited']
    pocket_names = {0: "Items", 1: "KeyItems", 2: "Pokeballs", 3: "TMs", 4: "Berries"}
    pk_strs = [pocket_names.get(p, f"?{p}") for p in pockets]

    print(f"  ✅ taught_bag_transitions.json:")
    print(f"     Frames: {n_frames} | Sessions: {n_sessions}")
    print(f"     Items used: {items_used}")
    print(f"     Pockets visited: {', '.join(pk_strs)}")


def _write_empty_bag_transitions():
    with open(TAUGHT_BAG_TRANSITIONS_FILE, 'w') as f:
        json.dump({'bag_frames': [], 'metadata': {
            'total_bag_frames': 0, 'bag_sessions_recorded': 0,
            'items_used': [], 'pockets_visited': []}}, f, indent=2)


# =========================================================================
# POST-PROCESSOR 5: taught_start_menu_transitions.json (NEW v17.2)
# Writes accumulated start menu session data from brain's buffer.
# Format matches AI agent's load_taught_start_menu_transitions().
# =========================================================================

def run_start_menu_extraction():
    """
    Write taught_start_menu_transitions.json from brain's accumulated
    start menu session data.

    Like bag extraction, the frames are recorded LIVE during the teaching
    session (by brain.record_start_menu_frame()), not extracted from
    taught_transitions.json after the fact.

    Output format (consumed by AI agent Cell 3 Part 2):
    {
        "start_menu_frames": [
            {
                "action": "DOWN",
                "recent_actions": ["START", "DOWN", "DOWN"],
                "state": {"gs": 1, "mc": 2, "mm": 6, "pc": -1, "sc": -1},
                "context": {
                    "target": "bag",
                    "target_mc": 2,
                    "party_hp_lowest": 0.4,
                    "has_status": false,
                    "in_battle": false,
                    "map_id": 3
                },
                "batch_type": "action_change",
                "frame_offset": 0
            },
            ...
        ],
        "metadata": {
            "total_frames": 80,
            "sessions_recorded": 10,
            "targets_navigated": {"bag": 7, "pokemon": 2, "save": 1},
            "avg_session_length": 8
        }
    }
    """
    sm_data = brain.get_accumulated_start_menu_data()
    n_frames = len(sm_data['start_menu_frames'])
    n_sessions = sm_data['metadata']['sessions_recorded']

    if n_frames == 0:
        print("  ⚠️ No start menu frames recorded — writing empty start menu transitions")
        _write_empty_start_menu_transitions()
        return

    with open(TAUGHT_START_MENU_TRANSITIONS_FILE, 'w') as f:
        json.dump(sm_data, f)

    targets = sm_data['metadata']['targets_navigated']
    avg_len = sm_data['metadata']['avg_session_length']

    print(f"  ✅ taught_start_menu_transitions.json:")
    print(f"     Frames: {n_frames} | Sessions: {n_sessions}")
    print(f"     Targets: {', '.join(f'{k}:{v}' for k, v in targets.items())}")
    print(f"     Avg session length: {avg_len}")


def _write_empty_start_menu_transitions():
    with open(TAUGHT_START_MENU_TRANSITIONS_FILE, 'w') as f:
        json.dump({'start_menu_frames': [], 'metadata': {
            'total_frames': 0, 'sessions_recorded': 0,
            'targets_navigated': {}, 'avg_session_length': 0}}, f, indent=2)


# =========================================================================
# POST-PROCESSOR 6: event_timeline.json (UNCHANGED)
# =========================================================================

def run_event_timeline_generation():
    if not TAUGHT_TRANSITIONS_FILE.exists():
        print("  ⚠️ No taught_transitions.json — writing empty event timeline")
        _write_empty_event_timeline()
        return

    with open(TAUGHT_TRANSITIONS_FILE, 'r') as f:
        data = json.load(f)

    all_frames = []
    for batch in data.get('batches', []):
        for frame in batch.get('frames', []):
            all_frames.append(frame)

    if not all_frames:
        print("  ⚠️ No frames — writing empty event timeline")
        _write_empty_event_timeline()
        return

    # Load nav targets for linking events to journey progress
    nav_targets = []
    if NAV_TARGETS_PATH.exists():
        try:
            with open(NAV_TARGETS_PATH, 'r') as f:
                nav_data = json.load(f)
            nav_targets = nav_data.get('global_order', [])
        except:
            pass

    print(f"  Generating event timeline from {len(all_frames)} frames...")
    print(f"    Nav targets available: {len(nav_targets)}")

    # === STEP 1: Walk through frames, detect event boundaries ===
    events = []
    event_order = 0

    prev_battle = 0
    prev_map = None
    prev_gs = 0
    battle_start_frame = -1
    battle_start_party = None
    battle_enemy_species = -1
    battle_enemy_level = -1
    battle_type = 0
    battle_moves_used = set()
    battle_turn_count = 0

    bag_start_frame = -1
    bag_start_party = None
    bag_items_used = []
    bag_pockets_visited = set()

    prev_party_fingerprint = None

    for i, frame in enumerate(all_frames):
        s = frame.get('state', {})
        x, y = s.get('x', 0), s.get('y', 0)
        map_id = s.get('map_id', 0)
        battle_flag = s.get('in_battle', 0)
        gs = s.get('game_state', 0)
        direction = s.get('direction', 0)

        party_snapshot = _extract_party_from_frame(frame)

        # --- BATTLE DETECTION ---
        if battle_flag == 1 and prev_battle == 0:
            battle_start_frame = i
            battle_start_party = party_snapshot
            battle_moves_used = set()
            battle_turn_count = 0
            bd_info = _get_battle_info_near_frame(i, all_frames)
            battle_enemy_species = bd_info.get('enemy_species', -1)
            battle_enemy_level = bd_info.get('enemy_level', -1)
            battle_type = bd_info.get('battle_type', 0)

        elif battle_flag == 0 and prev_battle == 1:
            party_after = party_snapshot
            outcome = _detect_timeline_battle_outcome(all_frames, battle_start_frame, i)
            hp_cost = 0.0
            if battle_start_party and party_after:
                start_ratios = battle_start_party.get('hp_ratios', [])
                end_ratios = party_after.get('hp_ratios', [])
                if start_ratios and end_ratios:
                    costs = []
                    for sr, er in zip(start_ratios, end_ratios):
                        if sr > 0:
                            costs.append(max(0.0, sr - er))
                    hp_cost = sum(costs) / max(1, len(costs))

            bt_str = "trainer" if (battle_type & 8) != 0 else "wild"
            nav_order = _find_nearest_nav_order(x, y, map_id, nav_targets)

            event = {
                'order': event_order, 'type': 'battle', 'nav_target_order': nav_order,
                'map_id': map_id, 'position': [x, y],
                'timestamp_range': [battle_start_frame, i],
                'battle_info': {
                    'battle_type': bt_str, 'enemy_species': battle_enemy_species,
                    'enemy_level': battle_enemy_level, 'outcome': outcome,
                    'turns': battle_turn_count, 'hp_cost': round(hp_cost, 3),
                    'moves_used': sorted(list(battle_moves_used)),
                },
                'party_before': battle_start_party or _empty_party_snapshot(),
                'party_after': party_after or _empty_party_snapshot(),
            }
            events.append(event)
            event_order += 1
            battle_start_frame = -1

        # --- BAG SESSION DETECTION ---
        if gs == 14 and prev_gs != 14 and battle_flag == 0:
            bag_start_frame = i
            bag_start_party = party_snapshot
            bag_items_used = []
            bag_pockets_visited = set()

        elif gs != 14 and prev_gs == 14 and battle_flag == 0:
            if bag_start_frame >= 0:
                party_after = party_snapshot
                reason = _infer_bag_reason(bag_start_party, party_after, bag_items_used,
                                           events, nav_targets, x, y, map_id)
                nav_order = _find_nearest_nav_order(x, y, map_id, nav_targets)

                event = {
                    'order': event_order, 'type': 'bag_session', 'nav_target_order': nav_order,
                    'map_id': map_id, 'position': [x, y],
                    'timestamp_range': [bag_start_frame, i],
                    'bag_info': {
                        'items_used': [{'item_id': iid, 'target_slot': -1, 'effect': 'unknown', 'hp_restored': 0}
                                       for iid in bag_items_used],
                        'pockets_visited': sorted(list(bag_pockets_visited)),
                        'reason_inferred': reason,
                    },
                    'party_before': bag_start_party or _empty_party_snapshot(),
                    'party_after': party_after or _empty_party_snapshot(),
                }
                events.append(event)
                event_order += 1
                bag_start_frame = -1

        if gs == 14 and bag_start_frame >= 0:
            pocket = s.get('bag_pocket', -1)
            if pocket >= 0:
                bag_pockets_visited.add(pocket)

        # --- MAP TRANSITION DETECTION ---
        if prev_map is not None and map_id != prev_map:
            nav_order = _find_nearest_nav_order(x, y, map_id, nav_targets)
            event = {
                'order': event_order, 'type': 'map_transition', 'nav_target_order': nav_order,
                'map_id': prev_map, 'position': [x, y], 'destination_map': map_id,
                'timestamp_range': [max(0, i-1), i],
                'party_snapshot': party_snapshot or _empty_party_snapshot(),
            }
            events.append(event)
            event_order += 1

        # --- PARTY SWITCH DETECTION ---
        current_fingerprint = _get_party_fingerprint(party_snapshot)
        if (prev_party_fingerprint is not None and current_fingerprint is not None and
            current_fingerprint != prev_party_fingerprint and battle_flag == 0):
            from_slot, to_slot = _detect_switch_slots(prev_party_fingerprint, current_fingerprint)
            reason = _infer_switch_reason(party_snapshot)
            nav_order = _find_nearest_nav_order(x, y, map_id, nav_targets)

            event = {
                'order': event_order, 'type': 'party_switch', 'nav_target_order': nav_order,
                'map_id': map_id, 'position': [x, y],
                'timestamp_range': [max(0, i-1), i],
                'switch_info': {'from_slot': from_slot, 'to_slot': to_slot, 'reason_inferred': reason},
                'party_snapshot': party_snapshot or _empty_party_snapshot(),
            }
            events.append(event)
            event_order += 1

        if current_fingerprint is not None:
            prev_party_fingerprint = current_fingerprint

        prev_battle = battle_flag
        prev_map = map_id
        prev_gs = gs

    # === STEP 2: Compute segments ===
    segments = _compute_segments(events, nav_targets)

    # === STEP 3: Identify preparation points ===
    preparation_points = _identify_preparation_points(events)

    # === STEP 4: Compute metadata ===
    type_counts = defaultdict(int)
    for e in events:
        type_counts[e['type']] += 1

    nav_orders_covered = sorted(set(e.get('nav_target_order', -1) for e in events if e.get('nav_target_order', -1) >= 0))

    metadata = {
        'total_events': len(events),
        'total_battles': type_counts.get('battle', 0),
        'total_bag_sessions': type_counts.get('bag_session', 0),
        'total_switches': type_counts.get('party_switch', 0),
        'total_map_transitions': type_counts.get('map_transition', 0),
        'playthrough_timesteps': len(all_frames),
        'nav_targets_covered': nav_orders_covered,
        'generation_timestamp': datetime.now().isoformat(),
    }

    # === STEP 5: Write output ===
    output = {
        'events': events, 'segments': segments,
        'preparation_points': preparation_points, 'metadata': metadata,
    }

    with open(EVENT_TIMELINE_FILE, 'w') as f:
        json.dump(output, f, indent=2)

    print(f"  ✅ event_timeline.json:")
    print(f"     Events: {len(events)} | Segments: {len(segments)} | Prep points: {len(preparation_points)}")
    print(f"     Types: {dict(type_counts)}")
    print(f"     Nav targets covered: {nav_orders_covered}")
    if preparation_points:
        for pp in preparation_points[:3]:
            print(f"     Prep: before #{pp['before_nav_order']} — {pp['reason']} (HP<{pp['party_hp_threshold']})")


# === Event timeline helper functions ===

def _extract_party_from_frame(frame):
    return None

def _get_battle_info_near_frame(frame_idx, all_frames):
    return {'enemy_species': -1, 'enemy_level': -1, 'battle_type': 0}

def _detect_timeline_battle_outcome(all_frames, start_idx, end_idx):
    if end_idx + 10 < len(all_frames):
        for j in range(end_idx, min(end_idx + 10, len(all_frames))):
            post_map = all_frames[j].get('state', {}).get('map_id', -1)
            if post_map in POKEMON_CENTER_MAPS:
                battle_map = all_frames[start_idx].get('state', {}).get('map_id', -1)
                if post_map != battle_map:
                    return 'loss'
    run_frames = all_frames[max(0, end_idx-10):end_idx]
    actions = [f.get('action', 'NONE') for f in run_frames if f.get('state', {}).get('in_battle', 0) == 1]
    for pattern in [['DOWN', 'RIGHT', 'A'], ['DOWN', 'A']]:
        plen = len(pattern)
        for k in range(len(actions) - plen + 1):
            if actions[k:k+plen] == pattern:
                return 'run'
    return 'win'

def _find_nearest_nav_order(x, y, map_id, nav_targets):
    if not nav_targets: return -1
    best_order, best_dist = -1, float('inf')
    for t in nav_targets:
        t_map = t.get('map_id', -1)
        t_pos = t.get('position', [0, 0])
        t_order = t.get('order', -1)
        if t_map == map_id:
            dist = abs(x - t_pos[0]) + abs(y - t_pos[1])
            if dist < best_dist:
                best_dist = dist
                best_order = t_order
    if best_order == -1 and nav_targets:
        for t in reversed(nav_targets):
            best_order = t.get('order', -1)
            break
    return best_order

def _empty_party_snapshot():
    return {'hp_ratios': [], 'status_flags': [], 'count': 0, 'levels': []}

def _get_party_fingerprint(party_snapshot):
    if party_snapshot is None: return None
    levels = party_snapshot.get('levels', [])
    if not levels: return None
    return tuple(levels)

def _detect_switch_slots(prev_fp, curr_fp):
    if prev_fp is None or curr_fp is None: return -1, -1
    for i in range(min(len(prev_fp), len(curr_fp))):
        if prev_fp[i] != curr_fp[i]:
            for j in range(len(curr_fp)):
                if j != i and curr_fp[j] == prev_fp[i]:
                    return i, j
            return i, -1
    return -1, -1

def _infer_switch_reason(party_snapshot):
    if party_snapshot is None: return "unknown"
    hp_ratios = party_snapshot.get('hp_ratios', [])
    if hp_ratios and len(hp_ratios) > 0:
        if hp_ratios[0] < 0.3: return "lead_low_hp"
        if hp_ratios[0] < 0.5: return "lead_moderate_hp"
    status_flags = party_snapshot.get('status_flags', [])
    if status_flags and len(status_flags) > 0:
        if status_flags[0] != 0: return "lead_has_status"
    return "strategic"

def _infer_bag_reason(party_before, party_after, items_used, events, nav_targets, x, y, map_id):
    if party_before is None: return "unknown"
    hp_ratios = party_before.get('hp_ratios', [])
    lowest_hp = min((r for r in hp_ratios if r > 0), default=1.0)
    has_status = any(s != 0 for s in party_before.get('status_flags', []))
    if lowest_hp < 0.5: return "low_hp"
    if has_status: return "status_condition"
    upcoming_battles = [e for e in events if e.get('type') == 'battle' and e.get('nav_target_order', -1) >= 0]
    if upcoming_battles:
        nav_order = _find_nearest_nav_order(x, y, map_id, nav_targets)
        close_battles = [b for b in upcoming_battles if 0 < b.get('nav_target_order', -1) - nav_order <= 3]
        if close_battles:
            trainer_battles = [b for b in close_battles if b.get('battle_info', {}).get('battle_type') == 'trainer']
            if trainer_battles: return "pre_trainer_prep"
            return "pre_battle_prep"
    if lowest_hp >= 0.9: return "item_management"
    return "other"

def _compute_segments(events, nav_targets):
    if not events or not nav_targets: return []
    orders_seen = sorted(set(e.get('nav_target_order', -1) for e in events if e.get('nav_target_order', -1) >= 0))
    if len(orders_seen) < 2: return []
    segments = []
    for i in range(len(orders_seen) - 1):
        from_order = orders_seen[i]
        to_order = orders_seen[i + 1]
        range_events = [e for e in events if from_order <= e.get('nav_target_order', -1) < to_order]
        battles = [e for e in range_events if e.get('type') == 'battle']
        trainer_battles = [b for b in battles if b.get('battle_info', {}).get('battle_type') == 'trainer']
        wild_battles = [b for b in battles if b.get('battle_info', {}).get('battle_type') == 'wild']
        bag_sessions = [e for e in range_events if e.get('type') == 'bag_session']
        map_transitions = [e for e in range_events if e.get('type') == 'map_transition']
        hp_costs = [b.get('battle_info', {}).get('hp_cost', 0.0) for b in battles]
        avg_hp_cost = sum(hp_costs) / max(1, len(hp_costs))
        total_hp_cost = sum(hp_costs)
        segment = {
            'from_nav_order': from_order, 'to_nav_order': to_order,
            'battles_expected': len(battles), 'avg_hp_cost_per_battle': round(avg_hp_cost, 3),
            'total_hp_cost': round(total_hp_cost, 3), 'bag_sessions': len(bag_sessions),
            'trainer_battles': len(trainer_battles), 'wild_battles': len(wild_battles),
            'map_transitions': len(map_transitions),
        }
        segments.append(segment)
    return segments

def _identify_preparation_points(events):
    preparation_points = []
    for i, event in enumerate(events):
        if event.get('type') not in ('bag_session', 'party_switch'): continue
        party = event.get('party_before') or event.get('party_snapshot')
        if party is None: continue
        hp_ratios = party.get('hp_ratios', [])
        if not hp_ratios: continue
        living_ratios = [r for r in hp_ratios if r > 0]
        if not living_ratios: continue
        lowest_hp = min(living_ratios)
        avg_hp = sum(living_ratios) / len(living_ratios)
        if avg_hp >= 0.95: continue
        battle_follows = False
        battle_is_trainer = False
        for j in range(i + 1, min(i + 5, len(events))):
            if events[j].get('type') == 'battle':
                battle_follows = True
                if events[j].get('battle_info', {}).get('battle_type') == 'trainer':
                    battle_is_trainer = True
                break
        if not battle_follows: continue
        nav_order = event.get('nav_target_order', -1)
        reason = event.get('bag_info', {}).get('reason_inferred', 'unknown')
        if event.get('type') == 'party_switch':
            reason = event.get('switch_info', {}).get('reason_inferred', 'unknown')
        if battle_is_trainer: reason = "trainer_battle_ahead"
        elif reason == 'unknown' and lowest_hp < 0.5: reason = "low_hp_before_battle"
        action_taken = event.get('type')
        items_used = []
        if event.get('type') == 'bag_session':
            items_used = [item.get('item_id', -1) for item in event.get('bag_info', {}).get('items_used', [])]
        prep_point = {
            'before_nav_order': nav_order, 'reason': reason,
            'party_hp_threshold': round(lowest_hp, 3), 'action_taken': action_taken,
            'items_used': items_used,
        }
        preparation_points.append(prep_point)
    return preparation_points

def _write_empty_event_timeline():
    with open(EVENT_TIMELINE_FILE, 'w') as f:
        json.dump({
            'events': [], 'segments': [], 'preparation_points': [],
            'metadata': {
                'total_events': 0, 'total_battles': 0, 'total_bag_sessions': 0,
                'total_switches': 0, 'total_map_transitions': 0,
                'playthrough_timesteps': 0, 'nav_targets_covered': [],
                'generation_timestamp': datetime.now().isoformat(),
            }
        }, f, indent=2)


# =========================================================================
# MASTER POST-PROCESSING RUNNER (UPDATED v17.2 — now 6 post-processors)
# =========================================================================

def run_all_post_processing():
    """Run all 6 post-processors."""
    print("\n  📊 Running per_map_analysis...")
    try: run_per_map_analysis()
    except Exception as e: print(f"    ⚠️ per_map_analysis failed: {e}")

    print("  📊 Running nav target extraction...")
    try: run_nav_target_extraction()
    except Exception as e: print(f"    ⚠️ Nav targets failed: {e}")

    print("  📊 Running battle extraction...")
    try: run_battle_extraction()
    except Exception as e: print(f"    ⚠️ Battle extraction failed: {e}")

    print("  📊 Running bag extraction...")
    try: run_bag_extraction()
    except Exception as e: print(f"    ⚠️ Bag extraction failed: {e}")

    print("  📊 Running start menu extraction...")
    try: run_start_menu_extraction()
    except Exception as e: print(f"    ⚠️ Start menu extraction failed: {e}")

    print("  📊 Running event timeline generation...")
    try: run_event_timeline_generation()
    except Exception as e: print(f"    ⚠️ Event timeline failed: {e}")

In [5]:
# ============================================================================
# CELL 6B: Teaching Mode Main Loop
# ============================================================================
# SYNC TO AI AGENT v17.4 (Multi-Pool Pipeline):
# 1. NEW: brain.load_residual_file() at startup
# 2. NEW: brain.save_residual_file() at shutdown and periodic saves
# 3. NEW: Pipeline summary in startup banner
# 4. NEW: Pipeline summary in periodic logging (every 100 steps)
# 5. NEW: Pipeline summary in shutdown stats
# 6. All other functionality UNCHANGED from v17.2
# ============================================================================

# =========================================================================
# HUMAN ACTION READER
# =========================================================================

def read_human_action_from_state():
    """
    Read the human's current button press from input_cache.txt.
    Format per line: action,x,y,map,battle,menu,direction
    Returns expanded action name or None.
    """
    try:
        if not INPUT_FILE.exists():
            return None
        with open(INPUT_FILE, 'r') as f:
            content = f.read().strip()
        if not content:
            return None
        lines = content.split('\n')
        last_line = lines[-1].strip()
        if not last_line:
            return None
        parts = last_line.split(',')
        if parts:
            short_action = parts[0].strip()
            return ACTION_MAP.get(short_action, short_action)
    except:
        return None
    return None


# =========================================================================
# BRAIN INITIALIZATION
# =========================================================================

brain = Brain()

# Action perceptrons — shared chain (matches AI agent)
for b in ["UP", "DOWN", "LEFT", "RIGHT"]:
    brain.add(Perceptron("action", action=b, group="move", chain="shared"))
for b in ["A", "B", "Start", "Select"]:
    brain.add(Perceptron("action", action=b, group="interact", chain="shared"))

TAUGHT_MODEL_SAVE = BASE_PATH / "taught_model_checkpoint.json"
TAUGHT_EXPLORATION_SAVE = BASE_PATH / "taught_exploration_memory.json"
brain.EXPLORATION_MEMORY_FILE = TAUGHT_EXPLORATION_SAVE

# Resume if existing
if TAUGHT_MODEL_SAVE.exists():
    loaded_ts = brain.load_taught_model(TAUGHT_MODEL_SAVE)
    print(f"🎓 RESUME: Loaded taught model from timestep {loaded_ts}")
    print(f"   Utilities: {[f'{a.action}:{a.utility:.3f}' for a in brain.actions()]}")
    n_ent = len(brain.entities())
    print(f"   Entities: {n_ent}")
    chain_stats = brain.get_chain_stats()
    if chain_stats:
        chain_parts = [f"{c}:{s['actions']}a+{s['entities']}e" for c, s in chain_stats.items()]
        print(f"   Chains: {' | '.join(chain_parts)}")
    act_counts = {}
    for p in brain.perceptrons:
        act_counts[p.active_activation] = act_counts.get(p.active_activation, 0) + 1
    if len(act_counts) > 1:
        print(f"   Activations: {' '.join(f'{k}:{v}' for k,v in sorted(act_counts.items(), key=lambda x:x[1], reverse=True))}")
    # Pipeline summary on resume
    p_summary = brain.get_pipeline_summary()
    if p_summary != 'all empty':
        print(f"   Pipelines: {p_summary}")
else:
    print("🎓 FRESH START: No existing taught model")

if TAUGHT_EXPLORATION_SAVE.exists():
    brain.load_exploration_memory()
    print(f"   Exploration: {len(brain.exploration_memory)} maps loaded")

# Load residual perceptrons (NEW)
brain.load_residual_file()

# Resume existing bag data
if TAUGHT_BAG_TRANSITIONS_FILE.exists():
    try:
        with open(TAUGHT_BAG_TRANSITIONS_FILE, 'r') as f:
            existing_bag = json.load(f)
        existing_frames = existing_bag.get('bag_frames', [])
        existing_meta = existing_bag.get('metadata', {})
        if existing_frames:
            brain.bag_sessions_accumulated = existing_frames
            brain.bag_sessions_metadata['total_bag_frames'] = existing_meta.get('total_bag_frames', len(existing_frames))
            brain.bag_sessions_metadata['bag_sessions_recorded'] = existing_meta.get('bag_sessions_recorded', 0)
            brain.bag_sessions_metadata['items_used'] = set(existing_meta.get('items_used', []))
            brain.bag_sessions_metadata['pockets_visited'] = set(existing_meta.get('pockets_visited', []))
            print(f"   Bag data: {len(existing_frames)} frames from {existing_meta.get('bag_sessions_recorded', 0)} sessions resumed")
    except Exception as e:
        print(f"   ⚠️ Error loading existing bag data: {e}")

# Resume existing start menu data
if TAUGHT_START_MENU_TRANSITIONS_FILE.exists():
    try:
        with open(TAUGHT_START_MENU_TRANSITIONS_FILE, 'r') as f:
            existing_sm = json.load(f)
        existing_sm_frames = existing_sm.get('start_menu_frames', [])
        existing_sm_meta = existing_sm.get('metadata', {})
        if existing_sm_frames:
            brain.start_menu_sessions_accumulated = existing_sm_frames
            brain.start_menu_sessions_metadata['total_start_menu_frames'] = existing_sm_meta.get('total_frames', len(existing_sm_frames))
            brain.start_menu_sessions_metadata['start_menu_sessions_recorded'] = existing_sm_meta.get('sessions_recorded', 0)
            brain.start_menu_sessions_metadata['targets_navigated'] = existing_sm_meta.get('targets_navigated', {})
            print(f"   Start menu data: {len(existing_sm_frames)} frames from {existing_sm_meta.get('sessions_recorded', 0)} sessions resumed")
            targets = existing_sm_meta.get('targets_navigated', {})
            if targets:
                print(f"   Start menu targets: {', '.join(f'{k}:{v}' for k,v in targets.items())}")
    except Exception as e:
        print(f"   ⚠️ Error loading existing start menu data: {e}")

prev_context_state = None
prev_raw_position = None
last_human_action = None

print("="*70)
print("TEACHING MODE v3.3 — Multi-Pool Pipeline + Chain-Aware + Activations")
print("="*70)
print(f"  Model → {TAUGHT_MODEL_SAVE.name}")
print(f"  Exploration → {TAUGHT_EXPLORATION_SAVE.name}")
print(f"  Transitions → {TAUGHT_TRANSITIONS_FILE.name} (Lua)")
print(f"  Nav targets → {NAV_TARGETS_PATH.name} (auto)")
print(f"  Battle data → {BATTLE_TRANSITIONS_PATH.name} (auto)")
print(f"  Bag data → {TAUGHT_BAG_TRANSITIONS_FILE.name} (auto + live)")
print(f"  Start menu → {TAUGHT_START_MENU_TRANSITIONS_FILE.name} (live)")
print(f"  Timeline → {EVENT_TIMELINE_FILE.name} (auto)")
print("="*70)
print("ARCHITECTURE:")
print(f"  Chains: overworld | battle | party | bag | shared")
print(f"  Action perceptrons: shared chain (used by all contexts)")
print(f"  Entity perceptrons: per-chain (spawned from novel states)")
print(f"  Activation discovery: {', '.join(ACTIVATION_LIBRARY.get_names())}")
print(f"  Each chain: independent entities, error history, spawn capacity")
print("="*70)
print("PIPELINES:")
for pid, pipeline in brain.pipelines.items():
    layers_str = ' → '.join(pool.name for pool in pipeline.pools)
    total_p = sum(pool.get_perceptron_count(brain.perceptrons) for pool in pipeline.pools)
    total_r = sum(len(pool.residual) for pool in pipeline.pools)
    print(f"  {pid} ({pipeline.num_layers}L, decay={pipeline.credit_decay}): {layers_str}")
    if total_p > 0 or total_r > 0:
        print(f"    perceptrons: {total_p} | residual: {total_r} | authority: {pipeline.get_total_authority():.0%}")
print(f"  Pipeline learning runs alongside legacy chain learning")
print(f"  Pool perceptrons excluded from legacy flat updates (no double-update)")
print("="*70)
print("FEATURES:")
print(f"  - Human action tracking → directed perceptron learning")
print(f"  - Per-chain entity spawning + clustering → learned activation patterns")
print(f"  - Empirical activation fitting → best function per perceptron")
print(f"  - Pipeline forward/backward + spawn during human play (NEW)")
print(f"  - Bag session recording → taught_bag_transitions.json")
print(f"  - Start menu session recording → taught_start_menu_transitions.json")
print(f"  - Full state parsing: gs, mu, bg, pa, b every frame")
print(f"  - Map step counting → map_battle_stats in checkpoint")
print(f"  - Event timeline → post-processed playthrough history")
print(f"  - Type clusters → saved to checkpoint for AI bootstrap")
print(f"  - Pipeline pools + residual → saved to checkpoint for AI bootstrap (NEW)")
print("="*70)
print("START MENU RECORDING:")
print(f"  Cursor positions: 0=Pokedex 1=Pokemon 2=Bag 3=Card 4=Save 5=Option 6=Exit")
print(f"  AI Markov weights: menu={START_MENU_MARKOV_MENU_STATE_WEIGHT} "
      f"seq={START_MENU_MARKOV_ACTION_SEQ_WEIGHT} ctx={START_MENU_MARKOV_CONTEXT_WEIGHT}")
print(f"  Target inference: bag(gs→14) pokemon(pc→0-5) exit(gs→0)")
sm_stats = brain.get_start_menu_recording_stats()
if sm_stats['total_sessions'] > 0:
    print(f"  Resumed: {sm_stats['total_sessions']}sess {sm_stats['total_frames']}f "
          f"targets={sm_stats['targets_navigated']}")
print("="*70)
print("Play the game. Ctrl+C to stop, save, and post-process.")
print("="*70)

try:
    while True:
        # === READ GAME STATE (11 values) ===
        (context_state, palette_state, tile_state, dead, raw_position,
         battle_data, party_data, game_state_raw, menu_data, bag_data,
         text_flag) = read_game_state()

        if np.sum(np.abs(context_state)) < 0.001:
            time.sleep(0.02); continue

        raw_x, raw_y = raw_position
        in_battle = context_state[3]
        current_map = int(context_state[2])
        current_dir = int(context_state[5])

        # === UPDATE ALL DATA EVERY FRAME ===
        brain.update_battle_data(battle_data, in_battle)
        brain.update_party_data(party_data)
        brain.update_menu_data(menu_data)
        brain.update_bag_data(bag_data)
        brain.prev_game_state_raw = brain.game_state_raw
        brain.game_state_raw = game_state_raw
        brain.update_text_flag(text_flag)

        # === MAP STEP COUNTING (matches AI agent Phase 4) ===
        if in_battle <= 0.5 and game_state_raw == 0:
            brain.increment_map_steps(current_map)

        # === READ HUMAN ACTION ===
        human_action = read_human_action_from_state()
        if human_action and human_action != last_human_action:
            brain.set_human_action(human_action)
            last_human_action = human_action
        elif human_action:
            brain.last_action = human_action
            brain.human_action = human_action

        # === BAG SESSION RECORDING ===
        bag_boundary = brain.detect_bag_session_boundaries()
        if bag_boundary == "opened_overworld":
            brain.start_bag_session("overworld")
        elif bag_boundary == "opened_battle":
            brain.start_bag_session("battle")
        elif bag_boundary == "closed":
            brain.end_bag_session()

        if brain.bag_session_active and human_action:
            brain.record_bag_frame(human_action)

        # === START MENU SESSION RECORDING ===
        sm_boundary = brain.detect_start_menu_session_boundaries()
        if sm_boundary == "opened":
            brain.start_start_menu_session()
        elif sm_boundary == "closed_bag":
            brain.end_start_menu_session("bag_opened")
        elif sm_boundary == "closed_party":
            brain.end_start_menu_session("party_took_over")
        elif sm_boundary == "closed_exit":
            brain.end_start_menu_session("gs_left_1")
        elif sm_boundary == "closed_battle":
            brain.end_start_menu_session("battle_started")

        if brain.start_menu_session_active and human_action:
            brain.record_start_menu_frame(human_action)

        # === CORE LEARNING ===
        brain.update_position(raw_x, raw_y)
        derived = compute_derived_features(context_state, prev_context_state)
        learning_state = build_learning_state_overworld(derived, palette_state, tile_state, in_battle)
        brain.log_state(learning_state, context_state)

        time.sleep(0.02)

        # Read next state for learning
        (next_ctx, next_pal, next_til, next_dead, next_raw_pos,
         next_battle_data, next_party_data, next_gs_raw,
         next_menu_data, next_bag_data, next_text_flag) = read_game_state()
        next_derived = compute_derived_features(next_ctx, context_state)
        next_learning_state = build_learning_state_overworld(next_derived, next_pal, next_til, next_ctx[3])

        # Skip learning when text dialogue is on screen
        if not brain.is_text_active():
            brain.learn(learning_state, next_learning_state, context_state, next_ctx,
                        dead=dead, raw_position=raw_position, next_raw_position=next_raw_pos)

        prev_context_state = context_state.copy()
        prev_raw_position = raw_position
        brain.timestep += 1

        # === LOGGING (every 100 steps) ===
        if brain.timestep % 100 == 0:
            mem = brain.get_current_map_memory(current_map)
            vc = len(mem['visited_tiles']); oc = len(mem['obstructions'])
            ic = len(mem['interactable_objects']); cov = brain.get_exploration_coverage(current_map)
            ts = brain.get_tile_interaction_stats(current_map)
            dn = brain.DIRECTION_NAMES.get(current_dir, '?')
            tr = mem.get('transitions', [])

            gs_names = {0: "OW", 1: "MENU", 14: "BAG"}
            gs_str = gs_names.get(game_state_raw, f"GS{game_state_raw}")

            tf_str = " TXT" if brain.is_text_active() else ""
            print(f"\n{'='*60}")
            print(f"Step {brain.timestep} | Map {current_map} | ({raw_x},{raw_y}) {dn} | gs={gs_str} | Battle:{int(in_battle)}{tf_str}")
            print(f"  Human: {brain.human_action or 'None'} | Last: {brain.last_action or 'None'}")

            # Party
            if party_data.get('count', 0) > 0:
                hp_ratios = brain.get_party_hp_ratios()
                levels = brain.get_party_levels()
                lowest = brain.get_lowest_hp_ratio()
                hp_strs = [f"{r:.0%}" for r in hp_ratios]
                print(f"  Party ({party_data['count']}): HP={','.join(hp_strs)} Lv={levels} low={lowest:.0%}")

            # Battle
            if in_battle > 0.5:
                bd = brain.battle_data
                bt = bd.get('battle_type', 0)
                bt_str = "TRAINER" if (bt & 8) != 0 else "WILD"
                cursor_names = {0: 'FIGHT', 1: 'BAG', 2: 'PKMN', 3: 'RUN'}
                bc_str = cursor_names.get(bd.get('battle_cursor', -1), '?')
                print(f"  ⚔️ BATTLE #{brain.current_battle_id}: {bt_str} | Cursor:{bc_str}")
                if bd.get('player_species', -1) > 0:
                    print(f"     Player: sp{bd['player_species']} Lv{bd.get('player_level','?')} HP {bd['player_hp']}/{bd['player_max_hp']}")
                if bd.get('enemy_species', -1) > 0:
                    print(f"     Enemy:  sp{bd['enemy_species']} Lv{bd.get('enemy_level','?')} HP {bd['enemy_hp']}/{bd.get('enemy_max_hp','?')}")

            # Menu (gs==1, not battle)
            if game_state_raw == 1 and in_battle <= 0.5:
                md = brain.menu_data
                mc_name = START_MENU_MC_NAMES.get(md['mc'], f"?{md['mc']}")
                sm_sess = "SESSION" if brain.start_menu_session_active else ""
                print(f"  📋 Start Menu: mc={md['mc']}({mc_name}) mm={md['mm']} pc={md['pc']} sc={md['sc']} {sm_sess}")
            elif game_state_raw > 0 and game_state_raw != 1 and in_battle <= 0.5:
                md = brain.menu_data
                print(f"  Menu: mc={md['mc']}/{md['mm']} pc={md['pc']} sc={md['sc']}")

            # Bag
            if game_state_raw == 14 or brain.bag_session_active:
                bgd = brain.bag_data
                pocket_names = {0: "Items", 1: "KeyItems", 2: "Pokeballs", 3: "TMs", 4: "Berries"}
                pk_str = pocket_names.get(bgd['pocket'], f"?{bgd['pocket']}")
                n_items = len(bgd['items'])
                item_at_cur = brain.get_item_at_cursor()
                sess_str = f" SESSION({brain.bag_session_context},{brain.bag_session_frame_count}f)" if brain.bag_session_active else ""
                print(f"  🎒 Bag: {pk_str} cur={bgd['cursor']} items={n_items}{f' item={item_at_cur}' if item_at_cur>0 else ''}{sess_str}")

            # Bag recording stats
            bag_stats = brain.get_bag_recording_stats()
            if bag_stats['total_sessions'] > 0:
                print(f"  🎒 Recorded: {bag_stats['total_sessions']}sess {bag_stats['total_frames']}f")

            # Start menu recording stats
            sm_stats = brain.get_start_menu_recording_stats()
            if sm_stats['total_sessions'] > 0 or sm_stats['session_active']:
                sess_str = f" ACTIVE({sm_stats['session_frames']}f)" if sm_stats['session_active'] else ""
                tgt_str = ""
                if sm_stats['targets_navigated']:
                    tgt_str = f" tgt={sm_stats['targets_navigated']}"
                print(f"  📋 SM Recorded: {sm_stats['total_sessions']}sess {sm_stats['total_frames']}f{tgt_str}{sess_str}")

            # Exploration
            print(f"  V:{vc} Obs:{oc} Cov:{cov:.0%} Int:{ic}")
            print(f"  Probe:{ts['probed']} Exh:{ts['exhausted']} Suc:{ts['with_success']}")
            if tr: print(f"  Trans: {len(tr)} known")

            # Chain stats
            chain_stats = brain.get_chain_stats()
            if chain_stats:
                chain_parts = [f"{c}:{s['actions']}a+{s['entities']}e" for c, s in chain_stats.items()]
                print(f"  🧬 Chains: {' | '.join(chain_parts)}")

            # Pipeline stats (NEW)
            p_summary = brain.get_pipeline_summary()
            if p_summary != 'all empty':
                print(f"  🔗 Pipelines: {p_summary}")

            # Activation stats
            act_counts = {}
            act_changes = 0
            for p in brain.perceptrons:
                act_counts[p.active_activation] = act_counts.get(p.active_activation, 0) + 1
                act_changes += p.activation_change_count
            if len(act_counts) > 1 or act_changes > 0:
                act_str = ' '.join(f'{k}:{v}' for k, v in sorted(act_counts.items(), key=lambda x: x[1], reverse=True))
                print(f"  🧬 Act: [{act_str}] changes:{act_changes}")

            # Stagnation / pattern
            if brain.state_stagnation_count > 5:
                print(f"  ⚠️ Stag: {brain.state_stagnation_count}/{brain.STATE_STAGNATION_THRESHOLD}")
            if brain.detected_pattern:
                print(f"  🔄 {'-'.join(str(a) for a in brain.detected_pattern)} x{brain.pattern_repeat_count}")

            # Battle buffer + map steps
            print(f"  BatBuf: {len(brain.battle_data_buffer)} ({brain.current_battle_id} battles)")
            ms = brain.map_step_counters.get(current_map, 0)
            mbs = brain.map_battle_stats.get(current_map)
            if mbs and mbs['battles_fought'] > 0:
                print(f"  📊 Map{current_map}: {ms}steps {mbs['battles_fought']}bat avg{mbs['avg_hp_cost']:.0%}")

            # Utilities
            au = sorted([(a.action, a.utility) for a in brain.actions()], key=lambda x: x[1], reverse=True)
            print(f"  Utils: {' '.join(f'{k}:{v:.2f}' for k,v in au)}")
            print(f"  🧩 {len(brain.perceptrons)}p ({len(brain.actions())}a {len(brain.entities())}e)")

        # === PERIODIC SAVE (every 500 steps) ===
        if brain.timestep % 500 == 0 and brain.timestep > 0:
            brain.save_model_checkpoint(TAUGHT_MODEL_SAVE)
            brain.save_exploration_memory()
            brain.save_residual_file()  # NEW
            try: run_bag_extraction()
            except Exception as e: print(f"  ⚠️ Periodic bag save failed: {e}")
            try: run_start_menu_extraction()
            except Exception as e: print(f"  ⚠️ Periodic start menu save failed: {e}")
            print(f"  💾 Auto-saved at step {brain.timestep}")

        # === PERIODIC POST-PROCESSING (every 2000 steps) ===
        if brain.timestep % 2000 == 0 and brain.timestep > 0:
            print(f"\n  📊 Periodic post-processing at step {brain.timestep}...")
            run_all_post_processing()
            print(f"  📊 Post-processing complete")

except KeyboardInterrupt:
    print("\n\n🛑 Stopping teaching...")

    # Close any active sessions
    if brain.bag_session_active:
        brain.end_bag_session()
    if brain.start_menu_session_active:
        brain.end_start_menu_session("shutdown")

    print("\n📁 Step 1/2: Saving model + exploration + residual...")
    brain.save_model_checkpoint(TAUGHT_MODEL_SAVE)
    brain.save_exploration_memory()
    brain.save_residual_file()  # NEW
    print(f"   ✅ {TAUGHT_MODEL_SAVE.name}")
    print(f"   ✅ {TAUGHT_EXPLORATION_SAVE.name}")
    print(f"   ✅ residual_perceptrons.json")

    print("\n📁 Step 2/2: Running all post-processors...")
    run_all_post_processing()

    # Summary
    total_tiles = sum(len(m['visited_tiles']) for m in brain.exploration_memory.values())
    total_entities = len(brain.entities())
    bag_stats = brain.get_bag_recording_stats()
    sm_stats = brain.get_start_menu_recording_stats()
    chain_stats = brain.get_chain_stats()
    mbs_summary = {'maps': len([s for s in brain.map_battle_stats.values() if s['battles_fought'] > 0]),
                   'battles': sum(s['battles_fought'] for s in brain.map_battle_stats.values())}

    # Activation summary
    act_counts = {}
    act_changes = 0
    for p in brain.perceptrons:
        act_counts[p.active_activation] = act_counts.get(p.active_activation, 0) + 1
        act_changes += p.activation_change_count

    print(f"\n{'='*60}")
    print(f"✅ TEACHING COMPLETE")
    print(f"   Timestep: {brain.timestep}")
    print(f"   Maps: {len(brain.exploration_memory)} | Tiles: {total_tiles}")
    print(f"   Battles buffered: {brain.current_battle_id}")
    print(f"   Battle data entries: {len(brain.battle_data_buffer)}")
    print(f"   Bag sessions: {bag_stats['total_sessions']} | Frames: {bag_stats['total_frames']}")
    if bag_stats['items_used']:
        print(f"   Items used in bag: {bag_stats['items_used']}")
    print(f"   Start menu sessions: {sm_stats['total_sessions']} | Frames: {sm_stats['total_frames']}")
    if sm_stats['targets_navigated']:
        print(f"   Start menu targets: {sm_stats['targets_navigated']}")
    print(f"   Map battle stats: {mbs_summary['maps']} maps, {mbs_summary['battles']} battles")

    print(f"\n   PERCEPTRONS: {len(brain.perceptrons)}")
    if chain_stats:
        for cname, cdata in chain_stats.items():
            sc = brain.entity_spawn_counts.get(cname, 0)
            mc = brain.entity_merge_counts.get(cname, 0)
            cap = brain.ENTITY_CAPACITY.get(cname, '?')
            n_act = cdata['actions']
            n_ent = cdata['entities']
            print(f"     {cname}: {n_act}a {n_ent}e (spawn:{sc} merge:{mc} cap:{cap})")

    # Pipeline summary (NEW)
    print(f"\n   PIPELINES:")
    for pid, pipeline in brain.pipelines.items():
        total_p = sum(pool.get_perceptron_count(brain.perceptrons) for pool in pipeline.pools)
        total_r = sum(len(pool.residual) for pool in pipeline.pools)
        if total_p > 0 or total_r > 0:
            print(f"     {pid}: {total_p}p {total_r}r auth={pipeline.get_total_authority():.0%}")
            for pool in pipeline.pools:
                n = pool.get_perceptron_count(brain.perceptrons)
                if n > 0 or len(pool.residual) > 0:
                    print(f"       {pool.name}: {n}p auth={pool.authority:.0%} "
                          f"spawn={pool.spawn_count} residual={len(pool.residual)}")
    p_summary = brain.get_pipeline_summary()
    if p_summary == 'all empty':
        print(f"     (all empty — pipelines will populate as AI plays)")

    print(f"\n   ACTIVATIONS: {act_changes} total changes")
    for k, v in sorted(act_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"     {k}: {v} perceptrons")

    print(f"\n   Utilities: {[f'{a.action}:{a.utility:.3f}' for a in brain.actions()]}")

    print(f"\n📦 Files ready to copy to AI agent:")
    print(f"   1. {TAUGHT_MODEL_SAVE.name} (model + chains + activations + pipelines + entities)")
    print(f"   2. {TAUGHT_TRANSITIONS_FILE.name} (overworld demos)")
    print(f"   3. {TAUGHT_EXPLORATION_SAVE.name} (exploration memory)")
    print(f"   4. {NAV_TARGETS_PATH.name} (navigation waypoints)")
    print(f"   5. {BATTLE_TRANSITIONS_PATH.name} (battle demos)")
    print(f"   6. {TAUGHT_BAG_TRANSITIONS_FILE.name} (bag demos)")
    print(f"   7. {TAUGHT_START_MENU_TRANSITIONS_FILE.name} (start menu demos)")
    print(f"   8. {EVENT_TIMELINE_FILE.name} (playthrough timeline)")
    print(f"   9. residual_perceptrons.json (pipeline residual memory)")
    print(f"{'='*60}")

  Loaded exploration: 0 maps
  🔗 Pipeline state restored
  🧬 Chains: shared:8a+0e
  📖 Model loaded: step 0
     Utilities: ['UP:1.000', 'DOWN:1.000', 'LEFT:1.000', 'RIGHT:1.000', 'A:1.000', 'B:1.000', 'Start:1.000', 'Select:1.000']
🎓 RESUME: Loaded taught model from timestep 0
   Utilities: ['UP:1.000', 'DOWN:1.000', 'LEFT:1.000', 'RIGHT:1.000', 'A:1.000', 'B:1.000', 'Start:1.000', 'Select:1.000']
   Entities: 0
   Chains: shared:8a+0e
  Loaded exploration: 0 maps
   Exploration: 0 maps loaded
TEACHING MODE v3.3 — Multi-Pool Pipeline + Chain-Aware + Activations
  Model → taught_model_checkpoint.json
  Exploration → taught_exploration_memory.json
  Transitions → taught_transitions.json (Lua)
  Nav targets → taught_nav_targets.json (auto)
  Battle data → taught_battle_transitions.json (auto)
  Bag data → taught_bag_transitions.json (auto + live)
  Start menu → taught_start_menu_transitions.json (live)
  Timeline → event_timeline.json (auto)
ARCHITECTURE:
  Chains: overworld | battle | pa